# Análise exploratória da base Bronze

### Este notebook tem como finalidade a avaliação de execuções necessárias na transição da camada bronze para a camada silver do Data Lake.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[1]") \
    .config("spark.driver.memory", "3g") \
    .getOrCreate()

In [2]:
from pathlib import Path
bronze_dir = Path(r"C:\Users\fred\Documents\Estudo de dados\Projeto\credit-risk\data\bronze")

In [3]:
from __future__ import annotations

print(f"{'Dataset':<45} {'Arquivos Parquet':>16}")
print("-" * 63)

for ds_path in sorted(bronze_dir.iterdir()):
    if ds_path.is_dir():
        parquet_count = len(list(ds_path.rglob("*.parquet")))
        print(f"{ds_path.name:<45} {parquet_count:>16}")


Dataset                                       Arquivos Parquet
---------------------------------------------------------------
train_applprev_1_0                                           1
train_applprev_1_1                                           1
train_applprev_2                                             1
train_base                                                   1
train_credit_bureau_a_1_0                                    1
train_credit_bureau_a_1_1                                    1
train_credit_bureau_a_1_2                                    1
train_credit_bureau_a_1_3                                    1
train_credit_bureau_a_2_0                                    1
train_credit_bureau_a_2_1                                    1
train_credit_bureau_a_2_10                                   1
train_credit_bureau_a_2_2                                    1
train_credit_bureau_a_2_3                                    1
train_credit_bureau_a_2_4                             

In [3]:
from __future__ import annotations
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql import DataFrame, SparkSession


def load_logical_dataset(spark: SparkSession, bronze_dir: Path, prefix: str) -> DataFrame:
    """
    Lê e une todas as partições de um dataset lógico pelo prefixo.

    Parâmetros
    ----------
    spark      : SparkSession ativa
    bronze_dir : Path para data/bronze
    prefix     : prefixo do dataset lógico (ex: 'train_applprev_1')

    Retorna
    -------
    DataFrame unificado de todas as partições encontradas
    """
    partitions = sorted(bronze_dir.glob(f"{prefix}*"))

    if not partitions:
        raise ValueError(f"Nenhuma partição encontrada para o prefixo: '{prefix}'")

    print(f"Partições encontradas para '{prefix}':")
    for p in partitions:
        print(f"  {p.name}")

    dfs = [spark.read.parquet(str(p)) for p in partitions]

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

## Dataset train_base (Verificações)

In [9]:
df_base = spark.read.parquet(str(bronze_dir / "train_base"))

In [12]:
df_base.show(5)

+-------+-------------+------+--------+------+--------------------+------------+--------------------+
|case_id|date_decision| MONTH|WEEK_NUM|target|             _run_id|_ingest_date|        _source_file|
+-------+-------------+------+--------+------+--------------------+------------+--------------------+
|      0|   2019-01-03|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      1|   2019-01-03|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      2|   2019-01-04|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      3|   2019-01-03|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      4|   2019-01-04|201901|       0|     1|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
+-------+-------------+------+--------+------+--------------------+------------+--------------------+
only showing top 5 rows



In [13]:
df_base.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- date_decision: string (nullable = true)
 |-- MONTH: long (nullable = true)
 |-- WEEK_NUM: long (nullable = true)
 |-- target: long (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [19]:
from __future__ import annotations

from pyspark.sql import functions as F

total = df_base.count()

null_counts = df_base.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_base.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_base.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


coluna                                          null_count   null_pct
----------------------------------------------------------------------
case_id                                                  0       0.0%
date_decision                                            0       0.0%
MONTH                                                    0       0.0%
WEEK_NUM                                                 0       0.0%
target                                                   0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [20]:
# ── SENTINELAS ──────────────────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_base.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "MONTH", "WEEK_NUM", "target",
                              "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_base.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

print("\n=== SENTINELAS — Strings ===")

string_cols = [c for c, t in df_base.dtypes if t == "string"
               and c not in ("_run_id", "_source_file")]

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]

for c in string_cols:
    counts = df_base.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")


=== SENTINELAS — Numéricos ===

=== SENTINELAS — Strings ===


In [21]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_base.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_base.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1540952
date_decision                                             657
WEEK_NUM                                                   94
MONTH                                                      22
target                                                      2


In [22]:
# ── CHAVE PRIMÁRIA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE — case_id ===")

total      = df_base.count()
unique_ids = df_base.select("case_id").distinct().count()
dups       = total - unique_ids

print(f"  Total de linhas : {total}")
print(f"  case_id únicos  : {unique_ids}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ case_id é chave primária única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar antes da Silver")
    df_base.groupBy("case_id").count() \
           .filter(F.col("count") > 1) \
           .orderBy(F.col("count").desc()) \
           .show(10, truncate=False)


=== UNICIDADE DA CHAVE — case_id ===
  Total de linhas : 1526659
  case_id únicos  : 1526659
  Duplicatas      : 0
  ✅ case_id é chave primária única


In [23]:
# ── DATAS — RANGE E CONSISTÊNCIA ─────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

# date_decision ainda é string no Bronze — inspecionar min/max antes do cast
df_base.select(
    F.min("date_decision").alias("date_decision_min"),
    F.max("date_decision").alias("date_decision_max"),
    F.min("MONTH").alias("MONTH_min"),
    F.max("MONTH").alias("MONTH_max"),
    F.min("WEEK_NUM").alias("WEEK_NUM_min"),
    F.max("WEEK_NUM").alias("WEEK_NUM_max"),
).show(truncate=False)


=== RANGE DE DATAS ===
+-----------------+-----------------+---------+---------+------------+------------+
|date_decision_min|date_decision_max|MONTH_min|MONTH_max|WEEK_NUM_min|WEEK_NUM_max|
+-----------------+-----------------+---------+---------+------------+------------+
|2019-01-01       |2020-10-05       |201901   |202010   |0           |91          |
+-----------------+-----------------+---------+---------+------------+------------+



In [25]:
df_base.filter(F.col("WEEK_NUM") < 0).count()


0

In [28]:
from pyspark.sql import functions as F

week_counts = df_base.groupBy("WEEK_NUM").count()

# Todas as semanas esperadas no range
all_weeks = spark.range(0, 92).toDF("WEEK_NUM")

# Semanas com gap (ausentes ou com count muito abaixo da média)
gaps = all_weeks.join(week_counts, on="WEEK_NUM", how="left") \
                .filter(F.col("count").isNull())

print("=== SEMANAS SEM REGISTROS ===")
gaps.orderBy("WEEK_NUM").show(truncate=False)




=== SEMANAS SEM REGISTROS ===
+--------+-----+
|WEEK_NUM|count|
+--------+-----+
+--------+-----+



In [24]:
# ── DISTRIBUIÇÃO DO TARGET ────────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO DO TARGET ===")

df_base.groupBy("target") \
       .count() \
       .withColumn("pct", F.round(F.col("count") / total * 100, 2)) \
       .orderBy("target") \
       .show(truncate=False)


=== DISTRIBUIÇÃO DO TARGET ===
+------+-------+-----+
|target|count  |pct  |
+------+-------+-----+
|0     |1478665|96.86|
|1     |47994  |3.14 |
+------+-------+-----+



## 📋 Análise — `train_base`

### Schema e Tipos

| Coluna        | dtype Bronze | Ação Silver          | Justificativa                                        |
|---------------|--------------|----------------------|------------------------------------------------------|
| case_id       | long         | manter               | Chave primária, tipo correto                         |
| date_decision | string       | cast → DateType      | Data armazenada como string no Parquet Kaggle        |
| MONTH         | long         | cast → IntegerType   | Range 201901–202010 cabe em int                      |
| WEEK_NUM      | long         | cast → IntegerType   | Range 0–91 cabe em int                               |
| target        | long         | cast → IntegerType   | Variável binária (0/1), long é desnecessário         |
| _run_id       | string       | propagar             | Metadado Bronze, não alterar                         |
| _ingest_date  | date         | propagar             | Metadado Bronze, não alterar                         |
| _source_file  | string       | propagar             | Metadado Bronze, não alterar                         |

### Nulos
Nenhuma coluna apresenta nulo. Nenhuma ação necessária na Silver.

### Sentinelas
Nenhum sentinela numérico ou string detectado. Tabela limpa.

### Cardinalidade
- `case_id`: ~1.54M distintos vs 1.526M linhas — consistente com unicidade confirmada
- `date_decision`: 657 datas distintas em ~20 meses — distribuição esperada
- `MONTH` e `WEEK_NUM`: cardinalidade baixa e coerente com o range temporal
- `target`: 2 valores — binário confirmado ✅

### Chave Primária
`case_id` é chave primária única (0 duplicatas em 1.526.659 linhas). Nenhuma deduplicação necessária na Silver.

### Range de Datas
`date_decision` cobre 2019-01-01 a 2020-10-05 (~22 meses), sem datas impossíveis. Cast seguro.

### Cobertura Temporal
Todas as 92 semanas (WEEK_NUM 0–91) apresentam registros, sem gaps no calendário.  
Volume por semana estável, variando em torno de ~16.000 registros. Nenhuma ação necessária.

### Valores Negativos em WEEK_NUM
WEEK_NUM mínimo = 0. Nenhum valor negativo detectado. Domínio válido.

### Desbalanceamento do Target
96,86% classe `0` (adimplente) vs 3,14% classe `1` (inadimplente).  
⚠️ Nenhuma ação na Silver — tratamento adiado para etapa de modelagem (SMOTE, class_weight ou threshold tuning).

---

### Ações Silver — Casts

```python
df = df.withColumn("date_decision", F.to_date(F.col("date_decision"), "yyyy-MM-dd"))
df = df.withColumn("MONTH",         F.col("MONTH").cast("int"))
df = df.withColumn("WEEK_NUM",      F.col("WEEK_NUM").cast("int"))
df = df.withColumn("target",        F.col("target").cast("int"))



## Dataset train_applprev_1_0

In [32]:
from pathlib import Path

# Sobe dois níveis a partir do notebook (notebooks/ → raiz do projeto)
repo_root  = Path().resolve().parent
bronze_dir = repo_root / "data" / "bronze"

print(f"bronze_dir: {bronze_dir}")  # confirme o path antes de rodar

df_applprev_1 = load_logical_dataset(spark, bronze_dir, "train_applprev_1")


bronze_dir: C:\Users\fred\Documents\Estudo de dados\Projeto\credit-risk\data\bronze
Partições encontradas para 'train_applprev_1':
  train_applprev_1_0
  train_applprev_1_1


In [34]:
df_applprev_1.show(5)

+-------+--------------+------------+-----------------+------------------------+---------------------+------------+-----------------+--------------------------+--------------------+----------------------+---------------------+-------------------+-------------------------+---------------+-------------+------------+------------------+-------------+------------+--------------+-------------------------+---------------+-----------------+----------------+--------------------------+------------------------+-----------------+----------------+----------------------+--------------------+----------+--------------------+---------+----------------+---------------+-----------------+---------------------------+---------------------+-----------+----------+--------------------+------------+--------------------+
|case_id|actualdpd_943P|annuity_853A|approvaldate_319D|byoccupationinc_3656910L|cancelreason_3545846M|childnum_21L|creationdate_885D|credacc_actualbalance_314A|credacc_credlmt_575A|credacc_maxhi

In [39]:
for field in df_applprev_1.schema.fields:
    print(f"{field.name:<45} {str(field.dataType):<20} nullable={field.nullable}")


case_id                                       LongType()           nullable=True
actualdpd_943P                                DoubleType()         nullable=True
annuity_853A                                  DoubleType()         nullable=True
approvaldate_319D                             StringType()         nullable=True
byoccupationinc_3656910L                      DoubleType()         nullable=True
cancelreason_3545846M                         StringType()         nullable=True
childnum_21L                                  DoubleType()         nullable=True
creationdate_885D                             StringType()         nullable=True
credacc_actualbalance_314A                    DoubleType()         nullable=True
credacc_credlmt_575A                          DoubleType()         nullable=True
credacc_maxhisbal_375A                        DoubleType()         nullable=True
credacc_minhisbal_90A                         DoubleType()         nullable=True
credacc_status_367L         

In [40]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_applprev_1.count()

null_counts = df_applprev_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_applprev_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_applprev_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 6525979

coluna                                          null_count   null_pct
----------------------------------------------------------------------
revolvingaccount_394A                              6229229     95.45%
credacc_actualbalance_314A                         6205945      95.1%
credacc_maxhisbal_375A                             6205945      95.1%
credacc_minhisbal_90A                              6205945      95.1%
credacc_status_367L                                6205945      95.1%
credacc_transactions_402L                          6205945      95.1%
isdebitcard_527L                                   6062966     92.91%
byoccupationinc_3656910L                           4991531     76.49%
dtlastpmt_581D                                     4750384     72.79%
dtlastpmtallstes_3545839D                          4043621     61.96%
employedfrom_700D                                  3886478     59.55%
childnum_21L                                       3559424     

In [41]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_applprev_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1")]

for c in numeric_cols:
    counts = df_applprev_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]

string_cols = [c for c, t in df_applprev_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_applprev_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")


=== SENTINELAS — Numéricos ===
  actualdpd_943P: {99: 3}
  annuity_853A: {999: 229, 9999: 20}
  credacc_actualbalance_314A: {99: 10}
  credacc_credlmt_575A: {99: 1, 9999: 2}
  credacc_maxhisbal_375A: {99: 8}
  credacc_minhisbal_90A: {99: 8}
  credamount_590A: {99: 1, 9999: 71, 99999: 11}
  currdebt_94A: {999: 2, 9999: 1}
  downpmt_134A: {99: 7, 999: 10, 9999: 4}
  mainoccupationinc_437A: {9999: 4}
  maxdpdtolerance_577P: {99: 288, 999: 48}
  outstandingdebt_522A: {999: 2, 9999: 1}

=== SENTINELAS — Strings ===


In [42]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_applprev_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_applprev_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1285631
currdebt_94A                                           541771
outstandingdebt_522A                                   377028
credamount_590A                                        245947
credacc_actualbalance_314A                              99957
annuity_853A                                            86299
revolvingaccount_394A                                   85004
credacc_maxhisbal_375A                                  73455
credacc_minhisbal_90A                                   72418
credacc_credlmt_575A                                    50158
byoccupationinc_3656910L                                31000
mainoccupationinc_437A                                  28172
downpmt_134A                                            21983
profession_152M                     

In [43]:
# ── CHAVE COMPOSTA (case_id + num_group1) ────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_applprev_1.select("case_id", "num_group1").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_applprev_1.groupBy("case_id", "num_group1").count() \
                 .filter(F.col("count") > 1) \
                 .orderBy(F.col("count").desc()) \
                 .show(10, truncate=False)

# Distribuição de num_group1 (quantas aplicações anteriores por case_id)
print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_applprev_1.groupBy("num_group1").count() \
             .orderBy("num_group1") \
             .show(25, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas          : 6525979
  Chaves únicas            : 6525979
  Duplicatas               : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |1221522|
|1         |991955 |
|2         |804797 |
|3         |653172 |
|4         |531208 |
|5         |432187 |
|6         |352659 |
|7         |288526 |
|8         |236723 |
|9         |194925 |
|10        |161093 |
|11        |133591 |
|12        |111283 |
|13        |92921  |
|14        |77866  |
|15        |65626  |
|16        |55222  |
|17        |46798  |
|18        |39844  |
|19        |34061  |
+----------+-------+



In [44]:
# ── DATAS — RANGE ────────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [
    "approvaldate_319D", "creationdate_885D", "dateactivated_425D",
    "dtlastpmt_581D", "dtlastpmtallstes_3545839D",
    "employedfrom_700D", "firstnonzeroinstldate_307D"
]

df_applprev_1.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+---------------------+---------------------+---------------------+---------------------+----------------------+----------------------+------------------+------------------+-----------------------------+-----------------------------+---------------------+---------------------+------------------------------+------------------------------+
|approvaldate_319D_min|approvaldate_319D_max|creationdate_885D_min|creationdate_885D_max|dateactivated_425D_min|dateactivated_425D_max|dtlastpmt_581D_min|dtlastpmt_581D_max|dtlastpmtallstes_3545839D_min|dtlastpmtallstes_3545839D_max|employedfrom_700D_min|employedfrom_700D_max|firstnonzeroinstldate_307D_min|firstnonzeroinstldate_307D_max|
+---------------------+---------------------+---------------------+---------------------+----------------------+----------------------+------------------+------------------+-----------------------------+-----------------------------+---------------------+---------------------+--------------------

In [45]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [
    "cancelreason_3545846M", "district_544M", "education_1138M",
    "postype_4733339M", "profession_152M", "rejectreason_755M",
    "rejectreasonclient_4145042M"
]

for c in code_cols:
    outliers = df_applprev_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️  {c}: {count} valor(es) fora do padrão")
        outliers.show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  cancelreason_3545846M: 1 valor(es) fora do padrão
+---------------------+
|cancelreason_3545846M|
+---------------------+
|a55475b1             |
+---------------------+

  ⚠️  district_544M: 3 valor(es) fora do padrão
+-------------+
|district_544M|
+-------------+
|a55475b1     |
|Q3_9_meet    |
|Q3_26_the    |
+-------------+

  ⚠️  education_1138M: 1 valor(es) fora do padrão
+---------------+
|education_1138M|
+---------------+
|a55475b1       |
+---------------+

  ⚠️  postype_4733339M: 1 valor(es) fora do padrão
+----------------+
|postype_4733339M|
+----------------+
|a55475b1        |
+----------------+

  ⚠️  profession_152M: 56 valor(es) fora do padrão
+---------------+
|profession_152M|
+---------------+
|Q1_11_the      |
|Q5_14_look     |
|Q5_12_is       |
|Q6_4_your      |
|Q7_11_similar  |
+---------------+
only showing top 5 rows

  ⚠️  rejectreason_755M: 1 valor(es) fora do padrão
+-----------------+
|rejectreason_755M|


## 📋 Análise — `train_applprev_1` (union de `_0` e `_1`)

### Schema e Tipos

| Coluna                       | dtype Bronze | Ação Silver        | Justificativa                                      |
|------------------------------|--------------|--------------------|----------------------------------------------------|
| case_id                      | long         | manter             | Parte da chave composta, tipo correto              |
| num_group1                   | long         | cast → IntegerType | Range 0–19, long desnecessário                     |
| actualdpd_943P               | double       | manter             | Métrica de atraso, double correto                  |
| annuity_853A                 | double       | manter             | Valor monetário, double correto                    |
| approvaldate_319D            | string       | cast → DateType    | Data armazenada como string                        |
| byoccupationinc_3656910L     | double       | manter             | Renda, double correto                              |
| cancelreason_3545846M        | string       | manter             | Código ofuscado, string correto                    |
| childnum_21L                 | double       | cast → IntegerType | Número de filhos não deve ser double               |
| creationdate_885D            | string       | cast → DateType    | Data armazenada como string                        |
| credacc_actualbalance_314A   | double       | manter             | Saldo, double correto                              |
| credacc_credlmt_575A         | double       | manter             | Limite de crédito, double correto                  |
| credacc_maxhisbal_375A       | double       | manter             | Saldo histórico, aceita negativos legítimos        |
| credacc_minhisbal_90A        | double       | manter             | Saldo histórico, aceita negativos legítimos        |
| credacc_status_367L          | string       | manter             | Categórico, string correto                         |
| credacc_transactions_402L    | double       | cast → IntegerType | Contagem de transações não deve ser double         |
| credamount_590A              | double       | manter             | Valor monetário, double correto                    |
| credtype_587L                | string       | manter             | Categórico, string correto                         |
| currdebt_94A                 | double       | manter             | Valor monetário, double correto                    |
| dateactivated_425D           | string       | cast → DateType    | Data armazenada como string                        |
| district_544M                | string       | manter             | Código ofuscado, string correto                    |
| downpmt_134A                 | double       | manter             | Valor monetário, double correto                    |
| dtlastpmt_581D               | string       | cast → DateType    | Data armazenada como string                        |
| dtlastpmtallstes_3545839D    | string       | cast → DateType    | Data armazenada como string                        |
| education_1138M              | string       | manter             | Código ofuscado, string correto                    |
| employedfrom_700D            | string       | cast → DateType    | Data armazenada como string                        |
| familystate_726L             | string       | manter             | Categórico, string correto                         |
| firstnonzeroinstldate_307D   | string       | cast → DateType    | Data armazenada como string                        |
| inittransactioncode_279L     | string       | manter             | Categórico, string correto                         |
| isbidproduct_390L            | boolean      | manter             | Booleano correto                                   |
| isdebitcard_527L             | boolean      | manter             | Booleano correto                                   |
| mainoccupationinc_437A       | double       | manter             | Renda, double correto                              |
| maxdpdtolerance_577P         | double       | manter             | Métrica de tolerância, double correto              |
| outstandingdebt_522A         | double       | manter             | Valor monetário, double correto                    |
| pmtnum_8L                    | double       | cast → IntegerType | Número de parcelas não deve ser double             |
| postype_4733339M             | string       | manter             | Código ofuscado, string correto                    |
| profession_152M              | string       | manter             | Código ofuscado, string correto                    |
| rejectreason_755M            | string       | manter             | Código ofuscado, string correto                    |
| rejectreasonclient_4145042M  | string       | manter             | Código ofuscado, string correto                    |
| revolvingaccount_394A        | double       | manter             | ID de conta codificado, não valor monetário        |
| status_219L                  | string       | manter             | Categórico, string correto                         |
| tenor_203L                   | double       | cast → IntegerType | Prazo em meses não deve ser double                 |
| _run_id                      | string       | propagar           | Metadado Bronze, não alterar                       |
| _ingest_date                 | date         | propagar           | Metadado Bronze, não alterar                       |
| _source_file                 | string       | propagar           | Metadado Bronze, não alterar                       |

### Nulos

| Classificação | Colunas | null_pct | Ação Silver |
|---|---|---|---|
| Estrutural | `credacc_*` (5 colunas), `revolvingaccount_394A`, `isdebitcard_527L` | 92–95% | Manter null — só existe quando há conta associada |
| Informativo | `byoccupationinc_3656910L`, `dtlastpmt_581D`, `dtlastpmtallstes_3545839D`, `employedfrom_700D`, `childnum_21L`, `dateactivated_425D`, `approvaldate_319D`, `familystate_726L` | 36–76% | Manter null — ausência tem significado de negócio |
| Baixo | `firstnonzeroinstldate_307D`, `pmtnum_8L`, `tenor_203L`, `annuity_853A`, `credamount_590A`, `credtype_587L`, `downpmt_134A`, `inittransactioncode_279L`, `credacc_credlmt_575A`, `mainoccupationinc_437A` | 1–10% | Manter null — imputação é decisão de feature engineering (Gold) |
| Residual | `actualdpd_943P`, `creationdate_885D`, `isbidproduct_390L`, `status_219L` | < 0.05% | Manter null |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

Sentinelas numéricos detectados e a substituir por `null` na Silver:

| Coluna | Valores sentinela encontrados |
|---|---|
| `actualdpd_943P` | 99 |
| `annuity_853A` | 999, 9999 |
| `credacc_actualbalance_314A` | 99 |
| `credacc_credlmt_575A` | 99, 9999 |
| `credacc_maxhisbal_375A` | 99 |
| `credacc_minhisbal_90A` | 99 |
| `credamount_590A` | 99, 9999, 99999 |
| `currdebt_94A` | 999, 9999 |
| `downpmt_134A` | 99, 999, 9999 |
| `mainoccupationinc_437A` | 9999 |
| `maxdpdtolerance_577P` | 99, 999 |
| `outstandingdebt_522A` | 999, 9999 |

Nenhum sentinela string detectado.

### Cardinalidade
- `case_id`: ~1.28M distintos — cobertura parcial do `train_base` (1.526M), esperado para tabela de aplicações anteriores
- `currdebt_94A`, `outstandingdebt_522A`, `credamount_590A`: alta cardinalidade — valores contínuos, esperado
- `profession_152M`: 11.592 distintos — alta cardinalidade para código ofuscado, atenção para encoding na Gold
- `num_group1`: 20 valores distintos (0–19) — máximo de 20 aplicações anteriores por cliente
- Nenhuma coluna constante detectada

### Chave Composta
`(case_id, num_group1)` é chave composta única (0 duplicatas em 6.525.979 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição de `num_group1`
Tabela com até 20 linhas por `case_id`. O volume decresce progressivamente do grupo 0 (1.221.522) ao grupo 19 (34.061), indicando que a maioria dos clientes tem poucas aplicações anteriores — distribuição esperada e sem anomalias.

### Range de Datas
Todas as datas dentro de ranges plausíveis:
- `approvaldate_319D`: 2005-12-30 a 2020-10-19 ✅
- `creationdate_885D`: 2005-12-26 a 2020-10-19 ✅
- `dateactivated_425D`: 2006-01-01 a 2020-10-19 ✅
- `dtlastpmt_581D`: 2008-07-04 a 2020-10-19 ✅
- `dtlastpmtallstes_3545839D`: 2008-07-04 a 2020-10-19 ✅
- `employedfrom_700D`: 1961-09-15 a 2020-07-15 ✅ (datas antigas são legítimas — data de início de emprego)
- `firstnonzeroinstldate_307D`: 2006-01-26 a 2020-11-19 ✅

Casts seguros para todas as colunas de data.

### Colunas Codificadas (`*M`) — Valores Fora do Padrão
⚠️ Valores fora do padrão `P\d+_\d+_\d+` detectados em múltiplas colunas:

| Coluna | Ocorrências | Valores |
|---|---|---|
| `cancelreason_3545846M` | 1 | `a55475b1` |
| `district_544M` | 3 | `a55475b1`, `Q3_9_meet`, `Q3_26_the` |
| `education_1138M` | 1 | `a55475b1` |
| `postype_4733339M` | 1 | `a55475b1` |
| `profession_152M` | 56 | padrão `Q\d+_\d+_word` |
| `rejectreason_755M` | 1 | `a55475b1` |
| `rejectreasonclient_4145042M` | 1 | `a55475b1` |

`a55475b1` aparece em 6 colunas — provável valor sentinela do Kaggle para dado ausente/mascarado. Substituir por `null` na Silver.  
Valores `Q\d+_\d+_word` em `district_544M` e `profession_152M` seguem padrão alternativo — manter como string, documentar como anomalia no `checks_silver.py`.

---

### Ações Silver — Casts

```python
date_cols = [
    "approvaldate_319D", "creationdate_885D", "dateactivated_425D",
    "dtlastpmt_581D", "dtlastpmtallstes_3545839D",
    "employedfrom_700D", "firstnonzeroinstldate_307D"
]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

df = df.withColumn("num_group1",            F.col("num_group1").cast("int"))
df = df.withColumn("childnum_21L",          F.col("childnum_21L").cast("int"))
df = df.withColumn("credacc_transactions_402L", F.col("credacc_transactions_402L").cast("int"))
df = df.withColumn("pmtnum_8L",             F.col("pmtnum_8L").cast("int"))
df = df.withColumn("tenor_203L",            F.col("tenor_203L").cast("int"))
```

### Ações Silver — Sentinelas → null

```python
sentinel_map = {
    "actualdpd_943P":             ,
    "annuity_853A":               ,
    "credacc_actualbalance_314A": ,
    "credacc_credlmt_575A":       ,
    "credacc_maxhisbal_375A":     ,
    "credacc_minhisbal_90A":      ,
    "credamount_590A":            ,
    "currdebt_94A":               ,
    "downpmt_134A":               ,
    "mainoccupationinc_437A":     ,
    "maxdpdtolerance_577P":       ,
    "outstandingdebt_522A":       ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col)))
```

### Ações Silver — Colunas `*M` com valor sentinela

```python
code_cols_with_sentinel = [
    "cancelreason_3545846M", "district_544M", "education_1138M",
    "postype_4733339M", "profession_152M", "rejectreason_755M",
    "rejectreasonclient_4145042M"
]

for c in code_cols_with_sentinel:
    df = df.withColumn(c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c)))
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1").count().filter(F.col("count") > 1).count()
assert dup_count == 0, "train_applprev_1: duplicatas na chave (case_id, num_group1)"

# Alertar valores fora do padrão remanescentes em *M (exceto a55475b1 já tratado)
for c in ["district_544M", "profession_152M"]:
    outliers = df.filter(
        ~F.col(c).rlike(r"^[PQ]\d+_\d+_\w+$") & F.col(c).isNotNull()
    ).count()
    if outliers > 0:
        print(f"⚠️  {c}: {outliers} valor(es) fora do padrão após tratamento")
```

**Granularidade:** N linhas por `case_id` (máx. 20) — chave composta `(case_id, num_group1)`.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.28M de 1.526M).  
**Nulos:** todos preservados — imputação adiada para Gold.


## Dataset train_applprev_2

In [12]:
df_train_applprev_2 = spark.read.parquet(str(bronze_dir / "train_applprev_2"))

In [13]:
df_train_applprev_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- cacccardblochreas_147M: string (nullable = true)
 |-- conts_type_509L: string (nullable = true)
 |-- credacc_cards_status_52L: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [14]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_train_applprev_2.count()

null_counts = df_train_applprev_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_train_applprev_2.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_train_applprev_2.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 14075487

coluna                                          null_count   null_pct
----------------------------------------------------------------------
credacc_cards_status_52L                          13733404     97.57%
conts_type_509L                                    2394056     17.01%
cacccardblochreas_147M                              109249      0.78%
case_id                                                  0       0.0%
num_group1                                               0       0.0%
num_group2                                               0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [15]:
# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_train_applprev_2.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_train_applprev_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Strings ===
  cacccardblochreas_147M: {'a55475b1': 13958443}
  conts_type_509L: nenhum sentinela detectado
  credacc_cards_status_52L: nenhum sentinela detectado


In [16]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_train_applprev_2.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_train_applprev_2.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1285631
num_group1                                                 20
num_group2                                                 12
cacccardblochreas_147M                                      9
conts_type_509L                                             9
credacc_cards_status_52L                                    6


In [17]:
# ── CHAVE COMPOSTA (case_id + num_group1 + num_group2) ───────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===")

unique_keys = df_train_applprev_2.select("case_id", "num_group1", "num_group2").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1, num_group2) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_train_applprev_2.groupBy("case_id", "num_group1", "num_group2").count() \
                       .filter(F.col("count") > 1) \
                       .orderBy(F.col("count").desc()) \
                       .show(10, truncate=False)

# Distribuição de num_group1 e num_group2
print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_train_applprev_2.groupBy("num_group1").count() \
                   .orderBy("num_group1") \
                   .show(25, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group2 ===")
df_train_applprev_2.groupBy("num_group2").count() \
                   .orderBy("num_group2") \
                   .show(25, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===
  Total de linhas          : 14075487
  Chaves únicas            : 14075487
  Duplicatas               : 0
  ✅ (case_id, num_group1, num_group2) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |2276307|
|1         |1976391|
|2         |1685107|
|3         |1421703|
|4         |1190551|
|5         |990136 |
|6         |820840 |
|7         |680934 |
|8         |563599 |
|9         |467994 |
|10        |389190 |
|11        |324193 |
|12        |271808 |
|13        |227714 |
|14        |191417 |
|15        |161700 |
|16        |136458 |
|17        |115979 |
|18        |98875  |
|19        |84591  |
+----------+-------+


=== DISTRIBUIÇÃO DE num_group2 ===
+----------+-------+
|num_group2|count  |
+----------+-------+
|0         |6525978|
|1         |4976173|
|2         |2286939|
|3         |276223 |
|4         |9394   |
|5         |717

In [18]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = ["cacccardblochreas_147M"]

for c in code_cols:
    outliers = df_train_applprev_2.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️  {c}: {count} valor(es) fora do padrão")
        outliers.show(10, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  cacccardblochreas_147M: 1 valor(es) fora do padrão
+----------------------+
|cacccardblochreas_147M|
+----------------------+
|a55475b1              |
+----------------------+



## 📋 Análise — `train_applprev_2`

### Schema e Tipos

| Coluna                   | dtype Bronze | Ação Silver        | Justificativa                              |
|--------------------------|--------------|--------------------|--------------------------------------------|
| case_id                  | long         | manter             | Parte da chave composta, tipo correto      |
| num_group1               | long         | cast → IntegerType | Range 0–19, long desnecessário             |
| num_group2               | long         | cast → IntegerType | Range 0–11, long desnecessário             |
| cacccardblochreas_147M   | string       | manter             | Código ofuscado, string correto            |
| conts_type_509L          | string       | manter             | Categórico, string correto                 |
| credacc_cards_status_52L | string       | manter             | Categórico, string correto                 |
| _run_id                  | string       | propagar           | Metadado Bronze, não alterar               |
| _ingest_date             | date         | propagar           | Metadado Bronze, não alterar               |
| _source_file             | string       | propagar           | Metadado Bronze, não alterar               |

### Nulos

| Classificação | Coluna | null_pct | Ação Silver |
|---|---|---|---|
| Estrutural | `credacc_cards_status_52L` | 97.57% | Manter null — só existe quando há cartão associado |
| Informativo | `conts_type_509L` | 17.01% | Manter null — ausência tem significado de negócio |
| Residual | `cacccardblochreas_147M` | 0.78% | Manter null — após tratamento do sentinela `a55475b1` o null_pct real será maior (ver sentinelas) |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

`cacccardblochreas_147M` apresenta 13.958.443 ocorrências do valor `a55475b1` — padrão recorrente no dataset como sentinela de dado ausente/mascarado (mesmo valor detectado em `train_applprev_1`). Substituir por `null` na Silver.

⚠️ Após a substituição, o null_pct real de `cacccardblochreas_147M` será de aproximadamente **99.77%** (109.249 nulos originais + 13.958.443 sentinelas = ~14.067.692 de 14.075.487 linhas).

### Cardinalidade
- `case_id`: ~1.28M distintos — mesma cobertura de `train_applprev_1`, esperado
- `cacccardblochreas_147M`: 9 valores distintos (incluindo `a55475b1`) — após tratamento, 8 categorias válidas
- `conts_type_509L`: 9 valores distintos — baixa cardinalidade, categórico simples
- `credacc_cards_status_52L`: 6 valores distintos — baixa cardinalidade, categórico simples
- `num_group1`: 20 distintos (0–19) — alinhado com `train_applprev_1`
- `num_group2`: 12 distintos (0–11) — sub-nível dentro de cada aplicação anterior

### Chave Composta
`(case_id, num_group1, num_group2)` é chave composta única (0 duplicatas em 14.075.487 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição
Tabela com **três níveis de granularidade**: `case_id` → `num_group1` (aplicação anterior) → `num_group2` (cartão dentro da aplicação).

- `num_group1`: distribuição decrescente de 0 a 19 — mesmo padrão de `train_applprev_1`, esperado
- `num_group2`: fortemente concentrado em 0 e 1 (96% das linhas); valores acima de 4 são raros (< 800 ocorrências combinadas) — a maioria das aplicações anteriores tem no máximo 2 cartões associados

### Colunas Codificadas (`*M`) — Valores Fora do Padrão
`cacccardblochreas_147M` contém o valor `a55475b1` — mesmo sentinela recorrente identificado em `train_applprev_1`. Tratado via substituição por `null` (ver ação abaixo).

---

### Ações Silver — Casts

```python
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
df = df.withColumn("num_group2", F.col("num_group2").cast("int"))
```

### Ações Silver — Sentinelas → null

```python
df = df.withColumn(
    "cacccardblochreas_147M",
    F.when(F.col("cacccardblochreas_147M") == "a55475b1", None)
     .otherwise(F.col("cacccardblochreas_147M"))
)
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_applprev_2: duplicatas na chave (case_id, num_group1, num_group2)"
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)`.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.28M de 1.526M).  
**Nulos:** todos preservados — imputação adiada para Gold.  
**Observação:** `a55475b1` é sentinela recorrente no dataset Home Credit — aplicar o mesmo tratamento em todas as tabelas onde for detectado.


## Dataset train_credit_bureau_a_1

In [5]:
from pathlib import Path

# Sobe dois níveis a partir do notebook (notebooks/ → raiz do projeto)
repo_root  = Path().resolve().parent
bronze_dir = repo_root / "data" / "bronze"

print(f"bronze_dir: {bronze_dir}")  # confirme o path antes de rodar

df_credit_bureau_a_1 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_a_1")

bronze_dir: C:\Users\fred\Documents\Estudo de dados\Projeto\credit-risk\data\bronze
Partições encontradas para 'train_credit_bureau_a_1':
  train_credit_bureau_a_1_0
  train_credit_bureau_a_1_1
  train_credit_bureau_a_1_2
  train_credit_bureau_a_1_3


In [27]:
df_credit_bureau_a_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- annualeffectiverate_199L: double (nullable = true)
 |-- annualeffectiverate_63L: double (nullable = true)
 |-- classificationofcontr_13M: string (nullable = true)
 |-- classificationofcontr_400M: string (nullable = true)
 |-- contractst_545M: string (nullable = true)
 |-- contractst_964M: string (nullable = true)
 |-- contractsum_5085717L: double (nullable = true)
 |-- credlmt_230A: double (nullable = true)
 |-- credlmt_935A: double (nullable = true)
 |-- dateofcredend_289D: string (nullable = true)
 |-- dateofcredend_353D: string (nullable = true)
 |-- dateofcredstart_181D: string (nullable = true)
 |-- dateofcredstart_739D: string (nullable = true)
 |-- dateofrealrepmt_138D: string (nullable = true)
 |-- debtoutstand_525A: double (nullable = true)
 |-- debtoverdue_47A: double (nullable = true)
 |-- description_351M: string (nullable = true)
 |-- dpdmax_139P: double (nullable = true)
 |-- dpdmax_757P: double (nullable = true)
 |-- dpdmaxd

In [28]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_a_1.count()

null_counts = df_credit_bureau_a_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_a_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_a_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 15940537

coluna                                          null_count   null_pct
----------------------------------------------------------------------
prolongationcount_599L                            15901589     99.76%
interestrate_508L                                 15874717     99.59%
annualeffectiverate_63L                           15629294     98.05%
contractsum_5085717L                              15621206      98.0%
prolongationcount_1120L                           15459068     96.98%
instlamount_852A                                  15346371     96.27%
numberofoverdueinstlmaxdat_641D                   15260655     95.73%
overdueamountmax2date_1142D                       15254776      95.7%
annualeffectiverate_199L                          15213904     95.44%
residualamount_488A                               15052745     94.43%
credlmt_230A                                      15044115     94.38%
nominalrate_281L                                  14960980    

In [29]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_credit_bureau_a_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1")]

for c in numeric_cols:
    counts = df_credit_bureau_a_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_credit_bureau_a_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_credit_bureau_a_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  annualeffectiverate_199L: {99: 36}
  contractsum_5085717L: {9999: 2}
  credlmt_935A: {99: 1, 999: 1, 9999: 1}
  debtoutstand_525A: {99: 1, 999: 6, 9999: 6}
  debtoverdue_47A: {99: 1, 9999: 1}
  dpdmax_139P: {99: 157, 999: 16}
  dpdmax_757P: {99: 1215, 999: 211, -1: 655}
  instlamount_768A: {99: 2, 999: 33, 9999: 6}
  instlamount_852A: {99: 2, 999: 9, 9999: 1}
  monthlyinstlamount_332A: {99: 2, 999: 93, 9999: 23, 99999: 1}
  monthlyinstlamount_674A: {99: 9, 999: 93, 9999: 18, 99999: 1}
  nominalrate_498L: {99: 36}
  numberofcontrsvalue_258L: {99: 1}
  numberofcontrsvalue_358L: {99: 2}
  numberofinstls_229L: {99: 124}
  numberofinstls_320L: {99: 186}
  numberofoutstandinstls_59L: {99: 489}
  numberofoverdueinstlmax_1039L: {99: 196, 999: 10}
  numberofoverdueinstlmax_1151L: {99: 1366, 999: 332}
  numberofoverdueinstls_725L: {99: 42, 999: 7}
  outstandingamount_354A: {99: 1}
  outstandingamount_362A: {999: 1, 9999: 6}
  overdueamount_31A: {99: 1}
  overduea

In [30]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_a_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_a_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1340900
outstandingamount_362A                                1315023
totaloutstanddebtvalue_39A                            1203061
debtoutstand_525A                                     1180860
overdueamountmax2_398A                                 967318
overdueamountmax_35A                                   887585
monthlyinstlamount_674A                                880310
residualamount_856A                                    530214
totalamount_6A                                         483170
monthlyinstlamount_332A                                470524
overdueamountmax2_14A                                  457750
overdueamountmax_155A                                  400753
totalamount_996A                                       280511
contractsum_5085717L                

In [31]:
# ── CHAVE COMPOSTA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_credit_bureau_a_1.select("case_id", "num_group1").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_a_1.groupBy("case_id", "num_group1").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_a_1.groupBy("num_group1").count() \
                    .orderBy("num_group1") \
                    .show(50, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas          : 15940537
  Chaves únicas            : 15940537
  Duplicatas               : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |1386273|
|1         |1386081|
|2         |1385817|
|3         |1385691|
|4         |1385691|
|5         |1385691|
|6         |1385691|
|7         |1385691|
|8         |1385691|
|9         |712528 |
|10        |660550 |
|11        |319007 |
|12        |266917 |
|13        |223530 |
|14        |186904 |
|15        |156624 |
|16        |131211 |
|17        |110186 |
|18        |92891  |
|19        |78139  |
|20        |66081  |
|21        |56170  |
|22        |47816  |
|23        |40939  |
|24        |35092  |
|25        |30302  |
|26        |26103  |
|27        |22630  |
|28        |19693  |
|29        |17255  |
|30        |15199  |
|31        |13325  |
|32        |1

In [32]:
# ── RANGE DE DATAS ───────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [c for c, t in df_credit_bureau_a_1.dtypes
             if t == "string" and c.endswith("D")
             and c not in ("_run_id", "_source_file")]

df_credit_bureau_a_1.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+----------------------+----------------------+----------------------+----------------------+------------------------+------------------------+------------------------+------------------------+------------------------+------------------------+--------------------+--------------------+-------------------+-------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+------------------------+------------------------+
|dateofcredend_289D_min|dateofcredend_289D_max|dateofcredend_353D_min|dateofcredend_353D_max|dateofcredstart_181D_min|dateofcredstart_181D_max|dateofcredstart_739D_min|dateofcredstart_739D_max|dateofrealrepmt_138D_min|dateofrealrepmt_138D_max|lastupdate_1112D_min|lastupdate_1112D_max|lastupdate_388D_min|lastupdate_388D_max|numb

In [11]:
from pyspark.sql import functions as F

date_range_cols = [
    "dateofcredstart_181D", "dateofcredstart_739D",
    "dateofcredend_289D",   "dateofcredend_353D",
    "dateofrealrepmt_138D"
]

# Filtra apenas datas que NÃO são os valores extremos conhecidos (1900, 2098)
for c in date_range_cols:
    df_credit_bureau_a_1.filter(
        F.col(c).isNotNull() &
        (F.col(c) != "1900-01-15") &
        (F.col(c) != "2098-01-15")
    ).select(
        F.min(c).alias("min_real"),
        F.max(c).alias("max_real")
    ).withColumn("coluna", F.lit(c)) \
     .select("coluna", "min_real", "max_real") \
     .show(truncate=False)




+--------------------+----------+----------+
|coluna              |min_real  |max_real  |
+--------------------+----------+----------+
|dateofcredstart_181D|1994-12-04|2020-10-12|
+--------------------+----------+----------+

+--------------------+----------+----------+
|coluna              |min_real  |max_real  |
+--------------------+----------+----------+
|dateofcredstart_739D|1999-09-15|2021-08-13|
+--------------------+----------+----------+

+------------------+----------+----------+
|coluna            |min_real  |max_real  |
+------------------+----------+----------+
|dateofcredend_289D|2004-05-29|2058-01-14|
+------------------+----------+----------+

+------------------+----------+----------+
|coluna            |min_real  |max_real  |
+------------------+----------+----------+
|dateofcredend_353D|1999-02-24|2068-01-15|
+------------------+----------+----------+

+--------------------+----------+----------+
|coluna              |min_real  |max_real  |
+--------------------+----

In [33]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [c for c in df_credit_bureau_a_1.columns if c.endswith("M")]

for c in code_cols:
    outliers = df_credit_bureau_a_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️  {c}: {count} valor(es) fora do padrão")
        outliers.show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  classificationofcontr_13M: 11 valor(es) fora do padrão
+-------------------------+
|classificationofcontr_13M|
+-------------------------+
|be7b251d                 |
|ea6782cc                 |
|1cf4e481                 |
|2c070815                 |
|00135d9c                 |
+-------------------------+
only showing top 5 rows

  ⚠️  classificationofcontr_400M: 389 valor(es) fora do padrão
+--------------------------+
|classificationofcontr_400M|
+--------------------------+
|51590aa9                  |
|acba4f13                  |
|ffee884a                  |
|fa2a66b3                  |
|01938327                  |
+--------------------------+
only showing top 5 rows

  ⚠️  contractst_545M: 51 valor(es) fora do padrão
+---------------+
|contractst_545M|
+---------------+
|83931972       |
|dd67cff0       |
|54132f86       |
|6d2380c9       |
|7640edc3       |
+---------------+
only showing top 5 rows

  ⚠️  contractst_964M: 295 valo

In [12]:
code_cols = [c for c in df_credit_bureau_a_1.columns if c.endswith("M")]

for c in code_cols:
    total_notnull = df_credit_bureau_a_1.filter(F.col(c).isNotNull()).count()
    outliers = df_credit_bureau_a_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull() &
        (F.col(c) != "a55475b1")
    ).count()
    pct = round(outliers / total_notnull * 100, 2) if total_notnull > 0 else 0
    print(f"  {c}: {outliers} hashes fora do padrão ({pct}% dos não-nulos)")


  classificationofcontr_13M: 2656721 hashes fora do padrão (16.67% dos não-nulos)
  classificationofcontr_400M: 6741045 hashes fora do padrão (42.29% dos não-nulos)
  contractst_545M: 2651823 hashes fora do padrão (16.64% dos não-nulos)
  contractst_964M: 6749556 hashes fora do padrão (42.34% dos não-nulos)
  description_351M: 61993 hashes fora do padrão (0.39% dos não-nulos)
  financialinstitution_382M: 4503506 hashes fora do padrão (28.25% dos não-nulos)
  financialinstitution_591M: 1322724 hashes fora do padrão (8.3% dos não-nulos)
  purposeofcred_426M: 2658858 hashes fora do padrão (16.68% dos não-nulos)
  purposeofcred_874M: 6746904 hashes fora do padrão (42.33% dos não-nulos)
  subjectrole_182M: 1301112 hashes fora do padrão (8.16% dos não-nulos)
  subjectrole_93M: 1275537 hashes fora do padrão (8.0% dos não-nulos)


In [34]:
# ── COLUNAS DUPLICADAS (*_xxxL / *_yyyL com mesma semântica) ─────────────────
# Esta tabela tem pares de colunas com nomes similares (ex: credlmt_230A / credlmt_935A)
# Verificar correlação entre pares para identificar redundância
print("=== AMOSTRA DE PARES DE COLUNAS SIMILARES ===")

df_credit_bureau_a_1.select(
    "credlmt_230A", "credlmt_935A",
    "dpdmax_139P", "dpdmax_757P",
    "annualeffectiverate_199L", "annualeffectiverate_63L"
).show(10, truncate=False)


=== AMOSTRA DE PARES DE COLUNAS SIMILARES ===
+------------+------------+-----------+-----------+------------------------+-----------------------+
|credlmt_230A|credlmt_935A|dpdmax_139P|dpdmax_757P|annualeffectiverate_199L|annualeffectiverate_63L|
+------------+------------+-----------+-----------+------------------------+-----------------------+
|NULL        |135806.0    |0.0        |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |0.0        |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL  

In [13]:
from pyspark.sql import functions as F

pairs = [
    ("credlmt_230A",              "credlmt_935A"),
    ("dpdmax_139P",               "dpdmax_757P"),
    ("annualeffectiverate_199L",  "annualeffectiverate_63L"),
    ("instlamount_768A",          "instlamount_852A"),
    ("monthlyinstlamount_332A",   "monthlyinstlamount_674A"),
    ("numberofinstls_229L",       "numberofinstls_320L"),
    ("totalamount_6A",            "totalamount_996A"),
]

print(f"{'Par':<55} {'ambos_preenchidos':>18} {'só_A':>10} {'só_B':>10}")
print("-" * 95)

for a, b in pairs:
    both = df_credit_bureau_a_1.filter(
        F.col(a).isNotNull() & F.col(b).isNotNull()
    ).count()
    only_a = df_credit_bureau_a_1.filter(
        F.col(a).isNotNull() & F.col(b).isNull()
    ).count()
    only_b = df_credit_bureau_a_1.filter(
        F.col(a).isNull() & F.col(b).isNotNull()
    ).count()
    print(f"{a+' / '+b:<55} {both:>18} {only_a:>10} {only_b:>10}")


Par                                                      ambos_preenchidos       só_A       só_B
-----------------------------------------------------------------------------------------------
credlmt_230A / credlmt_935A                                         162430     733992    1092555
dpdmax_139P / dpdmax_757P                                          1762039     881997    4764856
annualeffectiverate_199L / annualeffectiverate_63L                   17750     708883     293493
instlamount_768A / instlamount_852A                                  81266    1157928     512900
monthlyinstlamount_332A / monthlyinstlamount_674A                  1570564    1072078    4721603
numberofinstls_229L / numberofinstls_320L                           800438    5052700     603427
totalamount_6A / totalamount_996A                                   803012    5057869     601279


## 📋 Análise — `train_credit_bureau_a_1` (union de `_0` a `_3`)



### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| annualeffectiverate_199L | double | manter | Taxa percentual, double correto |
| annualeffectiverate_63L | double | manter | Taxa percentual, double correto |
| classificationofcontr_13M | string | manter | Código ofuscado, string correto |
| classificationofcontr_400M | string | manter | Código ofuscado, string correto |
| contractst_545M | string | manter | Código ofuscado, string correto |
| contractst_964M | string | manter | Código ofuscado, string correto |
| contractsum_5085717L | double | manter | Valor monetário, double correto |
| credlmt_230A | double | manter | Limite de crédito, double correto |
| credlmt_935A | double | manter | Limite de crédito, double correto |
| dateofcredend_289D | string | cast → DateType | Data armazenada como string |
| dateofcredend_353D | string | cast → DateType | Data armazenada como string |
| dateofcredstart_181D | string | cast → DateType | Data armazenada como string |
| dateofcredstart_739D | string | cast → DateType | Data armazenada como string |
| dateofrealrepmt_138D | string | cast → DateType | Data armazenada como string |
| debtoutstand_525A | double | manter | Valor monetário, double correto |
| debtoverdue_47A | double | manter | Valor monetário, double correto |
| description_351M | string | manter | Código ofuscado, string correto |
| dpdmax_139P | double | manter | Métrica de atraso, double correto |
| dpdmax_757P | double | manter | Métrica de atraso, double correto |
| dpdmaxdatemonth_442T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| dpdmaxdatemonth_89T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| dpdmaxdateyear_596T | double | cast → IntegerType | Ano, double desnecessário |
| dpdmaxdateyear_896T | double | cast → IntegerType | Ano, double desnecessário |
| financialinstitution_382M | string | manter | Código ofuscado, string correto |
| financialinstitution_591M | string | manter | Código ofuscado, string correto |
| instlamount_768A | double | manter | Valor monetário, double correto |
| instlamount_852A | double | manter | Valor monetário, double correto |
| interestrate_508L | double | manter | Taxa percentual, double correto |
| lastupdate_1112D | string | cast → DateType | Data armazenada como string |
| lastupdate_388D | string | cast → DateType | Data armazenada como string |
| monthlyinstlamount_332A | double | manter | Valor monetário, double correto |
| monthlyinstlamount_674A | double | manter | Valor monetário, double correto |
| nominalrate_281L | double | manter | Taxa percentual, double correto |
| nominalrate_498L | double | manter | Taxa percentual, double correto |
| numberofcontrsvalue_258L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofcontrsvalue_358L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofinstls_229L | double | cast → IntegerType | Contagem de parcelas, double desnecessário |
| numberofinstls_320L | double | cast → IntegerType | Contagem de parcelas, double desnecessário |
| numberofoutstandinstls_520L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoutstandinstls_59L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstlmax_1039L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstlmax_1151L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstlmaxdat_148D | string | cast → DateType | Data armazenada como string |
| numberofoverdueinstlmaxdat_641D | string | cast → DateType | Data armazenada como string |
| numberofoverdueinstls_725L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstls_834L | double | cast → IntegerType | Contagem, double desnecessário |
| outstandingamount_354A | double | manter | Valor monetário, double correto |
| outstandingamount_362A | double | manter | Valor monetário, double correto |
| overdueamount_31A | double | manter | Valor monetário, double correto |
| overdueamount_659A | double | manter | Valor monetário, double correto |
| overdueamountmax2_14A | double | manter | Valor monetário, double correto |
| overdueamountmax2_398A | double | manter | Valor monetário, double correto |
| overdueamountmax2date_1002D | string | cast → DateType | Data armazenada como string |
| overdueamountmax2date_1142D | string | cast → DateType | Data armazenada como string |
| overdueamountmax_155A | double | manter | Valor monetário, double correto |
| overdueamountmax_35A | double | manter | Valor monetário, double correto |
| overdueamountmaxdatemonth_284T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| overdueamountmaxdatemonth_365T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| overdueamountmaxdateyear_2T | double | cast → IntegerType | Ano, double desnecessário |
| overdueamountmaxdateyear_994T | double | cast → IntegerType | Ano, double desnecessário |
| periodicityofpmts_1102L | double | cast → IntegerType | 5 valores distintos — periodicidade categórica |
| periodicityofpmts_837L | double | cast → IntegerType | 5 valores distintos — periodicidade categórica |
| prolongationcount_1120L | double | cast → IntegerType | Contagem de prorrogações, double desnecessário |
| prolongationcount_599L | double | cast → IntegerType | Contagem de prorrogações, double desnecessário |
| purposeofcred_426M | string | manter | Código ofuscado, string correto |
| purposeofcred_874M | string | manter | Código ofuscado, string correto |
| refreshdate_3813885D | string | cast → DateType | Data armazenada como string |
| residualamount_488A | double | manter | Valor monetário, double correto |
| residualamount_856A | double | manter | Valor monetário, double correto |
| subjectrole_182M | string | manter | Código ofuscado, string correto |
| subjectrole_93M | string | manter | Código ofuscado, string correto |
| totalamount_6A | double | manter | Valor monetário, double correto |
| totalamount_996A | double | manter | Valor monetário, double correto |
| totaldebtoverduevalue_178A | double | manter | Valor monetário, double correto |
| totaldebtoverduevalue_718A | double | manter | Valor monetário, double correto |
| totaloutstanddebtvalue_39A | double | manter | Valor monetário, double correto |
| totaloutstanddebtvalue_668A | double | manter | Valor monetário, double correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

### Nulos

Tabela com alto volume de nulos estruturais organizados em **grupos funcionais** — colunas do mesmo grupo têm `null_pct` idêntico, indicando que só existem juntas:

| Grupo | null_pct | Colunas representativas |
|---|---|---|
| Grupo A | ~99% | `prolongationcount_599L`, `interestrate_508L` |
| Grupo B | ~96–98% | `annualeffectiverate_*`, `contractsum_*`, `instlamount_852A` |
| Grupo C | ~91–94% | `credlmt_*`, `debtoutstand_*`, `numberofcontrsvalue_358L` e adjacentes |
| Grupo D | ~83% | `dpdmax_139P`, `dateofcredend_289D`, `numberofoverdueinstls_725L` e adjacentes |
| Grupo E | ~57–63% | `dateofcredend_353D`, `dateofcredstart_181D`, `dpdmax_757P` e adjacentes |
| Baixo nulo | ~30% | `refreshdate_3813885D` |
| Zero nulo | 0% | `case_id`, `num_group1`, todas as colunas `*M`, metadados |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados. Imputação adiada para Gold.

### Sentinelas

**Numéricos** — 35 colunas afetadas. Substituir por `null` na Silver:

```python
sentinel_map = {
    "annualeffectiverate_199L":      ,
    "contractsum_5085717L":          ,
    "credlmt_935A":                  ,
    "debtoutstand_525A":             ,
    "debtoverdue_47A":               ,
    "dpdmax_139P":                   ,
    "dpdmax_757P":                   [99, 999, -1],
    "instlamount_768A":              ,
    "instlamount_852A":              ,
    "monthlyinstlamount_332A":       ,
    "monthlyinstlamount_674A":       ,
    "nominalrate_498L":              ,
    "numberofcontrsvalue_258L":      ,
    "numberofcontrsvalue_358L":      ,
    "numberofinstls_229L":           ,
    "numberofinstls_320L":           ,
    "numberofoutstandinstls_59L":    ,
    "numberofoverdueinstlmax_1039L": ,
    "numberofoverdueinstlmax_1151L": ,
    "numberofoverdueinstls_725L":    ,
    "outstandingamount_354A":        ,
    "outstandingamount_362A":        ,
    "overdueamount_31A":             ,
    "overdueamount_659A":            ,
    "overdueamountmax2_14A":         ,
    "overdueamountmax2_398A":        ,
    "overdueamountmax_155A":         ,
    "overdueamountmax_35A":          ,
    "residualamount_856A":           ,
    "totalamount_6A":                ,
    "totalamount_996A":              ,
    "totaldebtoverduevalue_178A":    ,
    "totaldebtoverduevalue_718A":    ,
    "totaloutstanddebtvalue_39A":    ,
    "totaloutstanddebtvalue_668A":   ,
}
```

⚠️ `dpdmax_757P` é o único campo com sentinela `-1` identificado no projeto até agora.

**Strings** — `a55475b1` detectado em todas as colunas `*M`. Substituir por `null`.

### Cardinalidade
- `case_id`: ~1.34M distintos — cobertura parcial do `train_base` (1.526M), esperado para bureau
- `num_group1`: 518 valores distintos — granularidade muito maior que `train_applprev` (20); representa múltiplos contratos de bureau por cliente
- Colunas monetárias (`outstandingamount_362A`, `totaloutstanddebtvalue_39A`): alta cardinalidade — valores contínuos, esperado
- `residualamount_488A`: 9 valores distintos com 94.43% de nulos — valores monetários legítimos, nenhum sentinela

### Chave Composta
`(case_id, num_group1)` é chave composta única (0 duplicatas em 15.940.537 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição de `num_group1`
Dois regimes distintos identificados:
- **Grupos 0–8**: volume estável ~1.385.691 por grupo — indica 9 tipos fixos de produto/contrato presentes para todos os clientes com bureau
- **Grupo 9 em diante**: queda brusca e decrescente até grupo 517 — registros adicionais para clientes com histórico extenso

⚠️ `num_group1` não é apenas índice sequencial nessa tabela — os primeiros 9 grupos têm semântica fixa. Investigar na Gold se grupos 0–8 representam tipos de produto distintos.

### Range de Datas
Datas com sentinelas conhecidos (`1900-01-15` e `2098-01-15`) detectados em 3 colunas:

| Coluna | Ocorrências anômalas | Range real válido |
|---|---|---|
| `dateofcredend_289D` | 53.722 | 2004-05-29 a 2058-01-14 |
| `dateofcredend_353D` | 20.415 | 1999-02-24 a 2068-01-15 |
| `dateofrealrepmt_138D` | 230 | 2000-01-15 a 2037-08-29 |

⚠️ Datas futuras (`2058`, `2068`) são **legítimas** — contratos de longo prazo (financiamentos, hipotecas). Substituir apenas os sentinelas `1900-01-15` e `2098-01-15` por `null`.

Demais colunas de data dentro de ranges plausíveis. Todos os casts seguros após tratamento de sentinelas.

### Colunas Codificadas (`*M`) — Dois Padrões de Codificação
Diferente das tabelas anteriores, as colunas `*M` desta tabela apresentam **dois esquemas de codificação legítimos**:
- **Padrão `P\d+_\d+_\d+`** — codificação principal
- **Hashes hexadecimais** (`be7b251d`, `ea6782cc`) — codificação alternativa, provavelmente de fonte de bureau distinta

| Coluna | % hashes dos não-nulos |
|---|---|
| `classificationofcontr_400M` | 42.29% |
| `contractst_964M` | 42.34% |
| `purposeofcred_874M` | 42.33% |
| `financialinstitution_382M` | 28.25% |
| `classificationofcontr_13M` | 16.67% |
| `contractst_545M` | 16.64% |
| `purposeofcred_426M` | 16.68% |
| `financialinstitution_591M` | 8.30% |
| `subjectrole_182M` / `subjectrole_93M` | ~8% |
| `description_351M` | 0.39% |

**Ação Silver:** manter ambos os padrões como string. Substituir apenas `a55475b1` por `null`. Registrar presença dos dois padrões no `checks_silver.py`.

### Pares de Colunas Similares
Todas as colunas similares têm sobreposição relevante — **não são mutuamente exclusivas**:

| Par | Ambos preenchidos | Estratégia Gold |
|---|---|---|
| `dpdmax_139P` / `dpdmax_757P` | 1.762.039 | Features independentes (períodos distintos) |
| `monthlyinstlamount_332A` / `monthlyinstlamount_674A` | 1.570.564 | Features independentes |
| `numberofinstls_229L` / `numberofinstls_320L` | 800.438 | Features independentes |
| `totalamount_6A` / `totalamount_996A` | 803.012 | Features independentes |
| `credlmt_230A` / `credlmt_935A` | 162.430 | Avaliar `coalesce(A, B)` |
| `instlamount_768A` / `instlamount_852A` | 81.266 | Avaliar `coalesce(A, B)` |
| `annualeffectiverate_199L` / `annualeffectiverate_63L` | 17.750 | Avaliar `coalesce(A, B)` |

**Ação Silver:** manter todos os pares intactos. Consolidação adiada para Gold.

---

### Ações Silver — Casts

```python
date_cols = [
    "dateofcredend_289D", "dateofcredend_353D", "dateofcredstart_181D",
    "dateofcredstart_739D", "dateofrealrepmt_138D", "lastupdate_1112D",
    "lastupdate_388D", "numberofoverdueinstlmaxdat_148D",
    "numberofoverdueinstlmaxdat_641D", "overdueamountmax2date_1002D",
    "overdueamountmax2date_1142D", "refreshdate_3813885D"
]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

int_cols = [
    "num_group1", "dpdmaxdatemonth_442T", "dpdmaxdatemonth_89T",
    "dpdmaxdateyear_596T", "dpdmaxdateyear_896T",
    "numberofcontrsvalue_258L", "numberofcontrsvalue_358L",
    "numberofinstls_229L", "numberofinstls_320L",
    "numberofoutstandinstls_520L", "numberofoutstandinstls_59L",
    "numberofoverdueinstlmax_1039L", "numberofoverdueinstlmax_1151L",
    "numberofoverdueinstls_725L", "numberofoverdueinstls_834L",
    "overdueamountmaxdatemonth_284T", "overdueamountmaxdatemonth_365T",
    "overdueamountmaxdateyear_2T", "overdueamountmaxdateyear_994T",
    "periodicityofpmts_1102L", "periodicityofpmts_837L",
    "prolongationcount_1120L", "prolongationcount_599L"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — Sentinelas em Datas → null

```python
sentinel_dates = ["1900-01-15", "2098-01-15"]
date_sentinel_cols = [
    "dateofcredend_289D", "dateofcredend_353D",
    "dateofcredstart_181D", "dateofcredstart_739D",
    "dateofrealrepmt_138D"
]
for c in date_sentinel_cols:
    df = df.withColumn(
        c, F.when(F.col(c).isin(sentinel_dates), None).otherwise(F.col(c))
    )
```

### Ações Silver — Sentinelas em Colunas `*M` → null

```python
m_cols = [c for c in df.columns if c.endswith("M")]
for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_a_1: duplicatas na chave (case_id, num_group1)"

# Auditoria de padrões em colunas *M
for c in [col for col in df.columns if col.endswith("M")]:
    total_notnull = df.filter(F.col(c).isNotNull()).count()
    pattern_p   = df.filter(F.col(c).rlike(r"^P\d+_\d+_\d+$")).count()
    pattern_hex = df.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull()
    ).count()
    print(f"{c}: padrão P={pattern_p} | hex={pattern_hex} | total={total_notnull}")
```

**Granularidade:** N linhas por `case_id` (máx. 517+) — chave composta `(case_id, num_group1)`.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.34M de 1.526M).  
**Nulos:** todos preservados — imputação adiada para Gold.  
**Pares similares:** mantidos intactos na Silver — decisão de consolidação (`coalesce` vs features independentes) adiada para Gold.  
**`num_group1` grupos 0–8:** semântica fixa suspeita — investigar na Gold se representam tipos de produto distintos.


## Dataset train_credit_bureau_a_2

In [14]:
df_credit_bureau_a_2 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_a_2")

Partições encontradas para 'train_credit_bureau_a_2':
  train_credit_bureau_a_2_0
  train_credit_bureau_a_2_1
  train_credit_bureau_a_2_10
  train_credit_bureau_a_2_2
  train_credit_bureau_a_2_3
  train_credit_bureau_a_2_4
  train_credit_bureau_a_2_5
  train_credit_bureau_a_2_6
  train_credit_bureau_a_2_7
  train_credit_bureau_a_2_8
  train_credit_bureau_a_2_9


In [15]:
df_credit_bureau_a_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- collater_typofvalofguarant_298M: string (nullable = true)
 |-- collater_typofvalofguarant_407M: string (nullable = true)
 |-- collater_valueofguarantee_1124L: double (nullable = true)
 |-- collater_valueofguarantee_876L: double (nullable = true)
 |-- collaterals_typeofguarante_359M: string (nullable = true)
 |-- collaterals_typeofguarante_669M: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- pmts_dpd_1073P: double (nullable = true)
 |-- pmts_dpd_303P: double (nullable = true)
 |-- pmts_month_158T: double (nullable = true)
 |-- pmts_month_706T: double (nullable = true)
 |-- pmts_overdue_1140A: double (nullable = true)
 |-- pmts_overdue_1152A: double (nullable = true)
 |-- pmts_year_1139T: double (nullable = true)
 |-- pmts_year_507T: double (nullable = true)
 |-- subjectroles_name_541M: string (nullable = true)
 |-- subjectroles_name_838M: string (nullable = true)
 |-- _run_id: s

In [16]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_a_2.count()

null_counts = df_credit_bureau_a_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_a_2.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_a_2.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 188298452

coluna                                          null_count   null_pct
----------------------------------------------------------------------
collater_valueofguarantee_1124L                  185545117     98.54%
collater_valueofguarantee_876L                   181414202     96.34%
pmts_dpd_1073P                                   152991152     81.25%
pmts_overdue_1140A                               152850520     81.17%
pmts_month_158T                                  122168636     64.88%
pmts_year_1139T                                  122168636     64.88%
pmts_dpd_303P                                    113465446     60.26%
pmts_overdue_1152A                               113377640     60.21%
pmts_month_706T                                   29568980      15.7%
pmts_year_507T                                    29568980      15.7%
case_id                                                  0       0.0%
collater_typofvalofguarant_298M                          0   

In [17]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_credit_bureau_a_2.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_credit_bureau_a_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_credit_bureau_a_2.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_credit_bureau_a_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  collater_valueofguarantee_876L: {9999: 3, 99999: 7}
  pmts_dpd_1073P: {99: 1144, 999: 207}
  pmts_dpd_303P: {99: 9490, 999: 2467, -1: 7489}
  pmts_overdue_1140A: {99: 51, 999: 7, 9999: 5}
  pmts_overdue_1152A: {99: 326, 999: 121, 9999: 27}

=== SENTINELAS — Strings ===
  collater_typofvalofguarant_298M: {'a55475b1': 185545117}
  collater_typofvalofguarant_407M: {'a55475b1': 181414202}
  collaterals_typeofguarante_359M: {'a55475b1': 181414626}
  collaterals_typeofguarante_669M: {'a55475b1': 185545250}
  subjectroles_name_541M: {'a55475b1': 181487794}
  subjectroles_name_838M: {'a55475b1': 185621035}


In [18]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_a_2.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_a_2.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
pmts_overdue_1152A                                    3343303
case_id                                               1340900
pmts_overdue_1140A                                    1125902
collater_valueofguarantee_876L                         158409
collater_valueofguarantee_1124L                         85988
pmts_dpd_303P                                            4372
pmts_dpd_1073P                                           4090
num_group1                                                335
num_group2                                                104
pmts_year_507T                                             27
collaterals_typeofguarante_359M                            14
collaterals_typeofguarante_669M                            14
pmts_year_1139T                                            13
pmts_month_158T                     

In [21]:
print("=== VALORES DISTINTOS — pmts_year_507T ===")
df_credit_bureau_a_2.select("pmts_year_507T") \
                    .filter(F.col("pmts_year_507T").isNotNull()) \
                    .distinct() \
                    .orderBy("pmts_year_507T") \
                    .show(30, truncate=False)

print("=== VALORES DISTINTOS — pmts_year_1139T ===")
df_credit_bureau_a_2.select("pmts_year_1139T") \
                    .filter(F.col("pmts_year_1139T").isNotNull()) \
                    .distinct() \
                    .orderBy("pmts_year_1139T") \
                    .show(20, truncate=False)


=== VALORES DISTINTOS — pmts_year_507T ===
+--------------+
|pmts_year_507T|
+--------------+
|2000.0        |
|2001.0        |
|2002.0        |
|2003.0        |
|2004.0        |
|2005.0        |
|2006.0        |
|2007.0        |
|2008.0        |
|2009.0        |
|2010.0        |
|2011.0        |
|2012.0        |
|2013.0        |
|2014.0        |
|2015.0        |
|2016.0        |
|2017.0        |
|2018.0        |
|2019.0        |
|2020.0        |
|2021.0        |
|2025.0        |
|2026.0        |
|2027.0        |
|2028.0        |
+--------------+

=== VALORES DISTINTOS — pmts_year_1139T ===
+---------------+
|pmts_year_1139T|
+---------------+
|2008.0         |
|2009.0         |
|2010.0         |
|2012.0         |
|2013.0         |
|2014.0         |
|2015.0         |
|2016.0         |
|2017.0         |
|2018.0         |
|2019.0         |
|2020.0         |
|2021.0         |
+---------------+



In [19]:
# ── CHAVE COMPOSTA (case_id + num_group1 + num_group2) ───────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===")

unique_keys = df_credit_bureau_a_2.select(
    "case_id", "num_group1", "num_group2"
).distinct().count()
dups = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1, num_group2) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_a_2.groupBy("case_id", "num_group1", "num_group2").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_a_2.groupBy("num_group1").count() \
                    .orderBy("num_group1") \
                    .show(50, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group2 ===")
df_credit_bureau_a_2.groupBy("num_group2").count() \
                    .orderBy("num_group2") \
                    .show(50, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===
  Total de linhas          : 188298452
  Chaves únicas            : 188298452
  Duplicatas               : 0
  ✅ (case_id, num_group1, num_group2) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+--------+
|num_group1|count   |
+----------+--------+
|0         |42903174|
|1         |29641063|
|2         |21502239|
|3         |16576418|
|4         |13333181|
|5         |10990028|
|6         |9107111 |
|7         |7545763 |
|8         |6239007 |
|9         |5166969 |
|10        |4246926 |
|11        |3483897 |
|12        |2864842 |
|13        |2361410 |
|14        |1933195 |
|15        |1590305 |
|16        |1309651 |
|17        |1081678 |
|18        |892716  |
|19        |745440  |
|20        |623243  |
|21        |522688  |
|22        |439851  |
|23        |370744  |
|24        |313399  |
|25        |267231  |
|26        |228871  |
|27        |197394  |
|28        |170826  |
|29        |149790

In [20]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [c for c in df_credit_bureau_a_2.columns if c.endswith("M")]

for c in code_cols:
    total_notnull = df_credit_bureau_a_2.filter(F.col(c).isNotNull()).count()
    outliers = df_credit_bureau_a_2.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull() &
        (F.col(c) != "a55475b1")
    ).count()
    pct = round(outliers / total_notnull * 100, 2) if total_notnull > 0 else 0
    if outliers > 0:
        print(f"  ⚠️  {c}: {outliers} hashes fora do padrão ({pct}% dos não-nulos)")
        df_credit_bureau_a_2.filter(
            ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
            F.col(c).isNotNull() &
            (F.col(c) != "a55475b1")
        ).select(c).distinct().show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  collater_typofvalofguarant_298M: 2753335 hashes fora do padrão (1.46% dos não-nulos)
+-------------------------------+
|collater_typofvalofguarant_298M|
+-------------------------------+
|8fd95e4b                       |
|9a0c095e                       |
|06fb9ba8                       |
|26cf31be                       |
+-------------------------------+

  ⚠️  collater_typofvalofguarant_407M: 6884250 hashes fora do padrão (3.66% dos não-nulos)
+-------------------------------+
|collater_typofvalofguarant_407M|
+-------------------------------+
|8fd95e4b                       |
|9a0c095e                       |
|06fb9ba8                       |
|3cbe86ba                       |
|9276e4bb                       |
+-------------------------------+
only showing top 5 rows

  ⚠️  collaterals_typeofguarante_359M: 6883826 hashes fora do padrão (3.66% dos não-nulos)
+-------------------------------+
|collaterals_typeofguarante_359M|
+----------

## 📋 Análise — `train_credit_bureau_a_2` (union de `_0` a `_10`)

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| num_group2 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| collater_typofvalofguarant_298M | string | manter | Código ofuscado, string correto |
| collater_typofvalofguarant_407M | string | manter | Código ofuscado, string correto |
| collater_valueofguarantee_1124L | double | manter | Valor de garantia, double correto |
| collater_valueofguarantee_876L | double | manter | Valor de garantia, double correto |
| collaterals_typeofguarante_359M | string | manter | Código ofuscado, string correto |
| collaterals_typeofguarante_669M | string | manter | Código ofuscado, string correto |
| pmts_dpd_1073P | double | manter | Métrica de atraso, double correto |
| pmts_dpd_303P | double | manter | Métrica de atraso, double correto |
| pmts_month_158T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| pmts_month_706T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| pmts_overdue_1140A | double | manter | Valor monetário em atraso, double correto |
| pmts_overdue_1152A | double | manter | Valor monetário em atraso, double correto |
| pmts_year_1139T | double | cast → IntegerType | Ano (2008–2021), double desnecessário |
| pmts_year_507T | double | cast → IntegerType | Ano (2000–2028), double desnecessário |
| subjectroles_name_541M | string | manter | Código ofuscado, string correto |
| subjectroles_name_838M | string | manter | Código ofuscado, string correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

### Nulos

| Classificação | Colunas | null_pct | Ação Silver |
|---|---|---|---|
| Estrutural | `collater_valueofguarantee_1124L`, `collater_valueofguarantee_876L` | 96–98% | Manter null — garantia ausente na maioria dos contratos |
| Informativo | `pmts_dpd_1073P`, `pmts_overdue_1140A` | ~81% | Manter null — pagamento sem atraso registrado |
| Informativo | `pmts_month_158T`, `pmts_year_1139T` | ~65% | Manter null — fonte com menor cobertura temporal |
| Informativo | `pmts_dpd_303P`, `pmts_overdue_1152A` | ~60% | Manter null — fonte alternativa com menor cobertura |
| Baixo | `pmts_month_706T`, `pmts_year_507T` | ~16% | Manter null |
| Zero | `case_id`, `num_group1`, `num_group2`, colunas `*M`, metadados | 0% | — |

⚠️ Após substituição de `a55475b1` por `null` nas colunas `*M`, o null_pct real será ~96–99% nessas colunas — alinhado com as colunas numéricas de garantia, confirmando que formam grupos funcionais.  
⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

**Numéricos:**

| Coluna | Valores sentinela |
|---|---|
| `collater_valueofguarantee_876L` | 9999, 99999 |
| `pmts_dpd_1073P` | 99, 999 |
| `pmts_dpd_303P` | 99, 999, **-1** |
| `pmts_overdue_1140A` | 99, 999, 9999 |
| `pmts_overdue_1152A` | 99, 999, 9999 |

⚠️ `pmts_dpd_303P` com sentinela `-1` — segundo campo identificado no projeto com esse padrão (o primeiro foi `dpdmax_757P` em `train_credit_bureau_a_1`).

**Strings:** `a55475b1` detectado em todas as 6 colunas `*M` com volumes massivos (181M–185M ocorrências). Substituir por `null`.

### Cardinalidade
- `case_id`: ~1.34M distintos — mesma cobertura de `train_credit_bureau_a_1`, esperado (mesma fonte de bureau)
- `pmts_overdue_1152A`: ~3.3M distintos — valores monetários contínuos, esperado
- `num_group1`: 335 distintos — range menor que `train_credit_bureau_a_1` (518), reflete que `num_group2` absorve parte da granularidade
- `num_group2`: 104 distintos — representa pagamentos mensais por contrato; explica o volume de 188M de linhas
- `pmts_year_507T`: 27 anos (2000–2028) vs `pmts_year_1139T`: 13 anos (2008–2021) — fontes com coberturas temporais distintas; anos futuros em `pmts_year_507T` (2025–2028) são parcelas futuras planejadas, legítimos
- `pmts_month_*T`: cardinalidade 12 — meses confirmados, cast para `int` seguro
- Colunas `*M`: 5–14 categorias válidas após remoção de `a55475b1`

### Chave Composta
`(case_id, num_group1, num_group2)` é chave composta única (0 duplicatas em 188.298.452 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição
Tabela com **três níveis de granularidade**: `case_id` → `num_group1` (contrato) → `num_group2` (pagamento mensal do contrato).  
- `num_group1`: distribuição decrescente de 0 (42.9M) a 49+ — mesmo padrão de `train_credit_bureau_a_1`
- `num_group2`: até 104 distintos — representa até ~8 anos de histórico mensal por contrato (104 meses ≈ 8.7 anos)
- Volume total de 188M é o maior do projeto — resultado direto da granularidade mensal por contrato

### Colunas Codificadas (`*M`) — Dois Padrões de Codificação
Mesmo padrão identificado em `train_credit_bureau_a_1` — dois esquemas coexistentes:

| Coluna | Hashes fora do padrão | % dos não-nulos |
|---|---|---|
| `collater_typofvalofguarant_407M` | 6.884.250 | 3.66% |
| `collaterals_typeofguarante_359M` | 6.883.826 | 3.66% |
| `collaterals_typeofguarante_669M` | similar | ~3.66% |
| `collater_typofvalofguarant_298M` | 2.753.335 | 1.46% |
| `subjectroles_name_*M` | presente | < 1% |

**Ação Silver:** manter ambos os padrões como string. Substituir apenas `a55475b1` por `null`.

---

### Ações Silver — Casts

```python
int_cols = [
    "num_group1", "num_group2",
    "pmts_month_158T", "pmts_month_706T",
    "pmts_year_1139T", "pmts_year_507T"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "collater_valueofguarantee_876L": ,
    "pmts_dpd_1073P":                 ,
    "pmts_dpd_303P":                  [99, 999, -1],
    "pmts_overdue_1140A":             ,
    "pmts_overdue_1152A":             ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — Sentinelas em Colunas `*M` → null

```python
m_cols = [c for c in df.columns if c.endswith("M")]
for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_a_2: duplicatas na chave (case_id, num_group1, num_group2)"

# Auditoria de padrões em colunas *M
for c in [col for col in df.columns if col.endswith("M")]:
    total_notnull = df.filter(F.col(c).isNotNull()).count()
    pattern_p   = df.filter(F.col(c).rlike(r"^P\d+_\d+_\d+$")).count()
    pattern_hex = df.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull()
    ).count()
    print(f"{c}: padrão P={pattern_p} | hex={pattern_hex} | total={total_notnull}")
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)` representando cliente → contrato → pagamento mensal.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.34M de 1.526M).  
**Volume:** 188M de linhas — maior tabela do projeto. Processar em chunks na Silver se necessário dado o ambiente `local[1]` com 3g de driver memory.  
**Anos futuros em `pmts_year_507T`** (2025–2028): parcelas futuras planejadas — legítimos, não tratar como anomalia.  
**Nulos:** todos preservados — imputação adiada para Gold.


## Dataset train_credit_bureau_b_1

In [22]:
df_credit_bureau_b_1 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_b_1")

Partições encontradas para 'train_credit_bureau_b_1':
  train_credit_bureau_b_1


In [23]:
df_credit_bureau_b_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- amount_1115A: double (nullable = true)
 |-- classificationofcontr_1114M: string (nullable = true)
 |-- contractdate_551D: string (nullable = true)
 |-- contractmaturitydate_151D: string (nullable = true)
 |-- contractst_516M: string (nullable = true)
 |-- contracttype_653M: string (nullable = true)
 |-- credlmt_1052A: double (nullable = true)
 |-- credlmt_228A: double (nullable = true)
 |-- credlmt_3940954A: double (nullable = true)
 |-- credor_3940957M: string (nullable = true)
 |-- credquantity_1099L: double (nullable = true)
 |-- credquantity_984L: double (nullable = true)
 |-- debtpastduevalue_732A: double (nullable = true)
 |-- debtvalue_227A: double (nullable = true)
 |-- dpd_550P: double (nullable = true)
 |-- dpd_733P: double (nullable = true)
 |-- dpdmax_851P: double (nullable = true)
 |-- dpdmaxdatemonth_804T: double (nullable = true)
 |-- dpdmaxdateyear_742T: double (nullable = true)
 |-- installmentamount_644A: double (nullable

In [24]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_b_1.count()

null_counts = df_credit_bureau_b_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_b_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_b_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 85791

coluna                                          null_count   null_pct
----------------------------------------------------------------------
periodicityofpmts_997L                               83764     97.64%
interesteffectiverate_369L                           76285     88.92%
credlmt_228A                                         69661      81.2%
residualamount_1093A                                 69666      81.2%
credlmt_1052A                                        58210     67.85%
residualamount_127A                                  58210     67.85%
interestrateyearly_538L                              56966      66.4%
residualamount_3940956A                              48272     56.27%
credlmt_3940954A                                     47573     55.45%
instlamount_892A                                     42298      49.3%
numberofinstls_810L                                  42298      49.3%
pmtnumpending_403L                                   42111     49

In [25]:
# ── SENTINELAS ───────────────────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_credit_bureau_b_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1")]

for c in numeric_cols:
    counts = df_credit_bureau_b_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_credit_bureau_b_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_credit_bureau_b_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  dpd_550P: {99: 1}
  dpdmax_851P: {99: 6, 9999: 2}
  installmentamount_833A: {9999: 2}
  instlamount_892A: {999: 1}
  maxdebtpduevalodued_3940955A: {99: 3}
  numberofinstls_810L: {99: 8}
  overdueamountmax_950A: {99: 2}
  pmtdaysoverdue_1135P: {99: 6}
  pmtnumpending_403L: {99: 13}
  residualamount_127A: {9999: 2}
  residualamount_3940956A: {9999: 4}

=== SENTINELAS — Strings ===
  classificationofcontr_1114M: {'a55475b1': 4060}
  contractdate_551D: nenhum sentinela detectado
  contractmaturitydate_151D: nenhum sentinela detectado
  contractst_516M: {'a55475b1': 4121}
  contracttype_653M: {'a55475b1': 3892}
  credor_3940957M: {'a55475b1': 3892}
  lastupdate_260D: nenhum sentinela detectado
  periodicityofpmts_997L: nenhum sentinela detectado
  periodicityofpmts_997M: {'a55475b1': 41057}
  pmtmethod_731M: {'a55475b1': 38483}
  purposeofcred_722M: {'a55475b1': 3892}
  subjectrole_326M: {'a55475b1': 32773}
  subjectrole_43M: {'a55475b1': 39563}


In [26]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_b_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_b_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
debtvalue_227A                                          40290
installmentamount_833A                                  39701
case_id                                                 38099
instlamount_892A                                        32457
totalamount_881A                                        28853
totalamount_503A                                        25087
residualamount_3940956A                                 23058
amount_1115A                                            20454
dpdmax_851P                                             19744
residualamount_127A                                     17182
credlmt_1052A                                            9811
credlmt_3940954A                                         9656
debtpastduevalue_732A                                    5770
dpd_550P                            

In [27]:
# ── CHAVE COMPOSTA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_credit_bureau_b_1.select("case_id", "num_group1").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_b_1.groupBy("case_id", "num_group1").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_b_1.groupBy("num_group1").count() \
                    .orderBy("num_group1") \
                    .show(50, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas : 85791
  Chaves únicas   : 85791
  Duplicatas      : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-----+
|num_group1|count|
+----------+-----+
|0         |36500|
|1         |27670|
|2         |12365|
|3         |5506 |
|4         |2185 |
|5         |894  |
|6         |370  |
|7         |163  |
|8         |67   |
|9         |32   |
|10        |11   |
|11        |8    |
|12        |6    |
|13        |4    |
|14        |2    |
|15        |2    |
|16        |2    |
|17        |1    |
|18        |1    |
|19        |1    |
|20        |1    |
+----------+-----+



In [28]:
# ── RANGE DE DATAS ───────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [c for c, t in df_credit_bureau_b_1.dtypes
             if t == "string" and c.endswith("D")
             and c not in ("_run_id", "_source_file")]

df_credit_bureau_b_1.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+---------------------+---------------------+-----------------------------+-----------------------------+-------------------+-------------------+
|contractdate_551D_min|contractdate_551D_max|contractmaturitydate_151D_min|contractmaturitydate_151D_max|lastupdate_260D_min|lastupdate_260D_max|
+---------------------+---------------------+-----------------------------+-----------------------------+-------------------+-------------------+
|2000-01-15           |2020-10-14           |2003-04-11                   |2045-08-17                   |2015-03-30         |2020-10-19         |
+---------------------+---------------------+-----------------------------+-----------------------------+-------------------+-------------------+



In [29]:
# ── COLUNAS *M — PADRÃO E HASHES ─────────────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [c for c in df_credit_bureau_b_1.columns if c.endswith("M")]

for c in code_cols:
    total_notnull = df_credit_bureau_b_1.filter(F.col(c).isNotNull()).count()
    outliers = df_credit_bureau_b_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull() &
        (F.col(c) != "a55475b1")
    ).count()
    pct = round(outliers / total_notnull * 100, 2) if total_notnull > 0 else 0
    if outliers > 0:
        print(f"  ⚠️  {c}: {outliers} hashes ({pct}% dos não-nulos)")
        df_credit_bureau_b_1.filter(
            ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
            F.col(c).isNotNull() &
            (F.col(c) != "a55475b1")
        ).select(c).distinct().show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  classificationofcontr_1114M: 81731 hashes (95.27% dos não-nulos)
+---------------------------+
|classificationofcontr_1114M|
+---------------------------+
|ea6782cc                   |
|436d55c2                   |
|1cf4e481                   |
|90c587b1                   |
|00135d9c                   |
+---------------------------+
only showing top 5 rows

  ⚠️  contractst_516M: 81670 hashes (95.2% dos não-nulos)
+---------------+
|contractst_516M|
+---------------+
|83931972       |
|dd67cff0       |
|54132f86       |
|04bf6e27       |
|7640edc3       |
+---------------+
only showing top 5 rows

  ⚠️  contracttype_653M: 81899 hashes (95.46% dos não-nulos)
+-----------------+
|contracttype_653M|
+-----------------+
|1c9c5356         |
|f4e17141         |
|60e784d6         |
|190917cc         |
|920c2ccd         |
+-----------------+
only showing top 5 rows

  ⚠️  credor_3940957M: 52847 hashes (61.6% dos não-nulos)
+---------------+
|cr

In [30]:
# ── PARES SIMILARES ──────────────────────────────────────────────────────────
print("=== SOBREPOSIÇÃO DE PARES SIMILARES ===")

pairs = [
    ("credlmt_1052A",         "credlmt_228A"),
    ("credlmt_228A",          "credlmt_3940954A"),
    ("dpd_550P",              "dpd_733P"),
    ("installmentamount_644A","installmentamount_833A"),
    ("residualamount_1093A",  "residualamount_127A"),
    ("totalamount_503A",      "totalamount_881A"),
    ("credquantity_1099L",    "credquantity_984L"),
]

print(f"{'Par':<55} {'ambos':>8} {'só_A':>10} {'só_B':>10}")
print("-" * 85)

for a, b in pairs:
    both   = df_credit_bureau_b_1.filter(F.col(a).isNotNull() & F.col(b).isNotNull()).count()
    only_a = df_credit_bureau_b_1.filter(F.col(a).isNotNull() & F.col(b).isNull()).count()
    only_b = df_credit_bureau_b_1.filter(F.col(a).isNull()    & F.col(b).isNotNull()).count()
    print(f"{a+' / '+b:<55} {both:>8} {only_a:>10} {only_b:>10}")


=== SOBREPOSIÇÃO DE PARES SIMILARES ===
Par                                                        ambos       só_A       só_B
-------------------------------------------------------------------------------------
credlmt_1052A / credlmt_228A                               11922      15659       4208
credlmt_228A / credlmt_3940954A                            11922       4208      26296
dpd_550P / dpd_733P                                        39290      13728       6938
installmentamount_644A / installmentamount_833A            39290       6938      13728
residualamount_1093A / residualamount_127A                 11922       4203      15659
totalamount_503A / totalamount_881A                        39290      13728       6938
credquantity_1099L / credquantity_984L                     39290      13728       6938


## 📋 Análise — `train_credit_bureau_b_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Range 0–20, long desnecessário |
| amount_1115A | double | manter | Valor monetário, double correto |
| classificationofcontr_1114M | string | manter | Código ofuscado, string correto |
| contractdate_551D | string | cast → DateType | Data armazenada como string |
| contractmaturitydate_151D | string | cast → DateType | Data armazenada como string |
| contractst_516M | string | manter | Código ofuscado, string correto |
| contracttype_653M | string | manter | Código ofuscado, string correto |
| credlmt_1052A | double | manter | Limite de crédito, double correto |
| credlmt_228A | double | manter | Limite de crédito, double correto |
| credlmt_3940954A | double | manter | Limite de crédito, double correto |
| credor_3940957M | string | manter | Código ofuscado, string correto |
| credquantity_1099L | double | cast → IntegerType | Contagem (14 distintos), double desnecessário |
| credquantity_984L | double | cast → IntegerType | Contagem (70 distintos), double desnecessário |
| debtpastduevalue_732A | double | manter | Valor monetário, double correto |
| debtvalue_227A | double | manter | Valor monetário, double correto |
| dpd_550P | double | manter | Métrica de atraso, double correto |
| dpd_733P | double | manter | Métrica de atraso, double correto |
| dpdmax_851P | double | manter | Métrica de atraso, double correto |
| dpdmaxdatemonth_804T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| dpdmaxdateyear_742T | double | cast → IntegerType | Ano (15 distintos), double desnecessário |
| installmentamount_644A | double | manter | Valor monetário, double correto |
| installmentamount_833A | double | manter | Valor monetário, double correto |
| instlamount_892A | double | manter | Valor monetário, double correto |
| interesteffectiverate_369L | double | manter | Taxa percentual, double correto |
| interestrateyearly_538L | double | manter | Taxa percentual, double correto |
| lastupdate_260D | string | cast → DateType | Data armazenada como string |
| maxdebtpduevalodued_3940955A | double | manter | Valor monetário, double correto |
| numberofinstls_810L | double | cast → IntegerType | Contagem de parcelas, double desnecessário |
| overdueamountmax_950A | double | manter | Valor monetário, double correto |
| overdueamountmaxdatemonth_494T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| overdueamountmaxdateyear_432T | double | cast → IntegerType | Ano (15 distintos), double desnecessário |
| periodicityofpmts_997L | double | cast → IntegerType | 5 valores distintos — periodicidade, double desnecessário |
| periodicityofpmts_997M | string | manter | Mesma semântica que `997L` mas como string — ver nota abaixo |
| pmtdaysoverdue_1135P | double | manter | Dias em atraso, double correto |
| pmtmethod_731M | string | manter | Código ofuscado, string correto |
| pmtnumpending_403L | double | cast → IntegerType | Contagem de parcelas pendentes, double desnecessário |
| purposeofcred_722M | string | manter | Código ofuscado, string correto |
| residualamount_1093A | double | manter | Valor monetário, double correto |
| residualamount_127A | double | manter | Valor monetário, double correto |
| residualamount_3940956A | double | manter | Valor monetário, double correto |
| subjectrole_326M | string | manter | Código ofuscado, string correto |
| subjectrole_43M | string | manter | Código ofuscado, string correto |
| totalamount_503A | double | manter | Valor monetário, double correto |
| totalamount_881A | double | manter | Valor monetário, double correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

⚠️ **`periodicityofpmts_997L` e `periodicityofpmts_997M`**: mesma semântica com sufixos diferentes (`L` = numérico, `M` = código ofuscado). São fontes distintas para o mesmo conceito — manter ambas na Silver. `997L` tem apenas 5 valores e 97.64% de nulos; `997M` tem 9 valores e 3.81% de nulos — cobertura muito diferente, não são redundantes.

⚠️ **`residualamount_1093A`**: apenas 2 valores distintos com 81.2% de nulos — investigar na Gold se é flag binário disfarçado de valor monetário.

### Nulos

| Classificação | null_pct | Colunas representativas |
|---|---|---|
| Estrutural alto | ~97% | `periodicityofpmts_997L` |
| Estrutural alto | ~81–89% | `credlmt_228A`, `residualamount_1093A`, `interesteffectiverate_369L` |
| Informativo | ~55–68% | `credlmt_1052A`, `residualamount_127A`, `interestrateyearly_538L`, `credlmt_3940954A` |
| Grupos funcionais | ~49% | `instlamount_892A`, `numberofinstls_810L`, `amount_1115A`, `debtvalue_227A` (null_pct idêntico) |
| Grupos funcionais | ~38–46% | `credquantity_984L`, `dpd_733P`, `installmentamount_644A`, `totalamount_881A` / `credquantity_1099L`, `dpd_550P`, `installmentamount_833A`, `totalamount_503A` |
| Baixo | ~5% | `debtpastduevalue_732A`, `dpdmax_851P`, `pmtdaysoverdue_1135P` e adjacentes |
| Residual | ~4% | `contractmaturitydate_151D`, `contractdate_551D`, `lastupdate_260D` |
| Zero | 0% | `case_id`, `num_group1`, colunas `*M` base, metadados |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

**Numéricos:**

| Coluna | Valores sentinela |
|---|---|
| `dpd_550P` | 99 |
| `dpdmax_851P` | 99, 9999 |
| `installmentamount_833A` | 9999 |
| `instlamount_892A` | 999 |
| `maxdebtpduevalodued_3940955A` | 99 |
| `numberofinstls_810L` | 99 |
| `overdueamountmax_950A` | 99 |
| `pmtdaysoverdue_1135P` | 99 |
| `pmtnumpending_403L` | 99 |
| `residualamount_127A` | 9999 |
| `residualamount_3940956A` | 9999 |

**Strings:** `a55475b1` em 9 colunas `*M`. Substituir por `null`.

### Cardinalidade
- `case_id`: ~38.099 distintos — cobertura muito menor que demais tabelas de bureau (~1.34M); essa é uma fonte de bureau menor/alternativa
- `residualamount_1093A`: apenas 2 distintos com 81.2% de nulos — investigar na Gold
- `installmentamount_644A`: apenas 85 distintos para valor monetário — possível valor discretizado ou padrão
- `num_group1`: 21 distintos (0–20) — distribuição fortemente concentrada em 0 e 1

### Chave Composta
`(case_id, num_group1)` é chave composta única (0 duplicatas em 85.791 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição de `num_group1`
Distribuição fortemente decrescente: grupo 0 com 36.500 linhas vs grupo 20 com apenas 1. A maioria dos clientes tem 1–2 contratos nessa fonte de bureau.

### Range de Datas
Todos os ranges dentro de domínio plausível:
- `contractdate_551D`: 2000-01-15 a 2020-10-14 ✅
- `contractmaturitydate_151D`: 2003-04-11 a 2045-08-17 ✅ (datas futuras legítimas — contratos de longo prazo)
- `lastupdate_260D`: 2015-03-30 a 2020-10-19 ✅

Nenhum sentinela de data detectado. Casts seguros.

### Colunas `*M` — Dois Padrões de Codificação
A maioria das colunas `*M` tem hashes como **padrão dominante** nessa tabela (95%+ dos não-nulos), invertendo o padrão das tabelas anteriores onde `P\d+_\d+_\d+` era majoritário. Confirma que `train_credit_bureau_b` é uma fonte de bureau diferente com esquema de codificação distinto.

| Coluna | Hashes (% dos não-nulos) |
|---|---|
| `classificationofcontr_1114M` | 95.27% |
| `contractst_516M` | 95.20% |
| demais `*M` | presente em proporções variadas |

**Ação Silver:** manter ambos os padrões como string. Substituir apenas `a55475b1` por `null`.

### Pares de Colunas Similares
Todos os pares têm sobreposição relevante — não são mutuamente exclusivos:

| Par | Ambos | Só A | Só B | Estratégia Gold |
|---|---|---|---|---|
| `dpd_550P` / `dpd_733P` | 39.290 | 13.728 | 6.938 | Features independentes |
| `installmentamount_644A` / `installmentamount_833A` | 39.290 | 6.938 | 13.728 | Features independentes |
| `totalamount_503A` / `totalamount_881A` | 39.290 | 13.728 | 6.938 | Features independentes |
| `credquantity_1099L` / `credquantity_984L` | 39.290 | 13.728 | 6.938 | Features independentes |
| `credlmt_1052A` / `credlmt_228A` | 11.922 | 15.659 | 4.208 | Avaliar `coalesce` |
| `credlmt_228A` / `credlmt_3940954A` | 11.922 | 4.208 | 26.296 | Avaliar `coalesce` |
| `residualamount_1093A` / `residualamount_127A` | 11.922 | 4.203 | 15.659 | Avaliar `coalesce` |

**Ação Silver:** manter todos os pares intactos. Consolidação adiada para Gold.

---

### Ações Silver — Casts

```python
date_cols = ["contractdate_551D", "contractmaturitydate_151D", "lastupdate_260D"]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

int_cols = [
    "num_group1", "credquantity_1099L", "credquantity_984L",
    "dpdmaxdatemonth_804T", "dpdmaxdateyear_742T",
    "numberofinstls_810L", "overdueamountmaxdatemonth_494T",
    "overdueamountmaxdateyear_432T", "periodicityofpmts_997L",
    "pmtnumpending_403L"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "dpd_550P":                     ,
    "dpdmax_851P":                  ,
    "installmentamount_833A":       ,
    "instlamount_892A":             ,
    "maxdebtpduevalodued_3940955A": ,
    "numberofinstls_810L":          ,
    "overdueamountmax_950A":        ,
    "pmtdaysoverdue_1135P":         ,
    "pmtnumpending_403L":           ,
    "residualamount_127A":          ,
    "residualamount_3940956A":      ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — Sentinelas em Colunas `*M` → null

```python
m_cols = [c for c in df.columns if c.endswith("M")]
for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_b_1: duplicatas na chave (case_id, num_group1)"

# Investigar residualamount_1093A — apenas 2 valores distintos
df.filter(F.col("residualamount_1093A").isNotNull()) \
  .select("residualamount_1093A").distinct().show(truncate=False)
```

**Granularidade:** N linhas por `case_id` (máx. 20) — chave composta `(case_id, num_group1)`.  
**Cobertura:** ~38K `case_id` distintos — fonte de bureau menor, cobertura parcial do `train_base`.  
**Join com train_base:** `left join` por `case_id`.  
**`periodicityofpmts_997L` vs `997M`:** fontes distintas para o mesmo conceito — coberturas muito diferentes (97.64% vs 3.81% de nulos), manter ambas.  
**Nulos:** todos preservados — imputação adiada para Gold.


## Dataset train_credit_bureau_b_2

In [5]:
df_credit_bureau_b_2 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_b_2")

Partições encontradas para 'train_credit_bureau_b_2':
  train_credit_bureau_b_2


In [32]:
df_credit_bureau_b_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- pmts_date_1107D: string (nullable = true)
 |-- pmts_dpdvalue_108P: double (nullable = true)
 |-- pmts_pmtsoverdue_635A: double (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [33]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_b_2.count()

null_counts = df_credit_bureau_b_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_b_2.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_b_2.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 1286755

coluna                                          null_count   null_pct
----------------------------------------------------------------------
pmts_dpdvalue_108P                                    5361      0.42%
pmts_pmtsoverdue_635A                                 5361      0.42%
case_id                                                  0       0.0%
num_group1                                               0       0.0%
num_group2                                               0       0.0%
pmts_date_1107D                                          0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [34]:
# ── SENTINELAS ───────────────────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, -1]

numeric_cols = [c for c, t in df_credit_bureau_b_2.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_credit_bureau_b_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  pmts_dpdvalue_108P: {99: 14, 999: 2, 9999: 5, 99999: 1}
  pmts_pmtsoverdue_635A: {99: 33}


In [35]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_b_2.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_b_2.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    print(f"{coluna:<45} {nunique:>15}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
pmts_dpdvalue_108P                                      59690
case_id                                                 37987
pmts_pmtsoverdue_635A                                    3921
pmts_date_1107D                                            57
num_group2                                                 37
num_group1                                                 21


In [36]:
# ── CHAVE COMPOSTA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===")

unique_keys = df_credit_bureau_b_2.select(
    "case_id", "num_group1", "num_group2"
).distinct().count()
dups = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1, num_group2) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_b_2.groupBy("case_id", "num_group1", "num_group2").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_b_2.groupBy("num_group1").count() \
                    .orderBy("num_group1").show(30, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group2 ===")
df_credit_bureau_b_2.groupBy("num_group2").count() \
                    .orderBy("num_group2").show(30, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===
  Total de linhas : 1286755
  Chaves únicas   : 1286755
  Duplicatas      : 0
  ✅ (case_id, num_group1, num_group2) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |723766|
|1         |327250|
|2         |139937|
|3         |58034 |
|4         |22693 |
|5         |8694  |
|6         |3354  |
|7         |1414  |
|8         |780   |
|9         |349   |
|10        |169   |
|11        |101   |
|12        |59    |
|13        |20    |
|14        |23    |
|15        |39    |
|16        |16    |
|17        |5     |
|18        |10    |
|19        |36    |
|20        |6     |
+----------+------+


=== DISTRIBUIÇÃO DE num_group2 ===
+----------+-----+
|num_group2|count|
+----------+-----+
|0         |81889|
|1         |78332|
|2         |73867|
|3         |69081|
|4         |64582|
|5         |60414|
|6         |56217|
|7         |52446|
|8     

In [37]:
# ── RANGE DE DATAS ───────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

df_credit_bureau_b_2.select(
    F.min("pmts_date_1107D").alias("pmts_date_min"),
    F.max("pmts_date_1107D").alias("pmts_date_max")
).show(truncate=False)

# Verificar sentinelas de data conhecidos
print("\n=== SENTINELAS DE DATA ===")
sentinel_dates = ["1900-01-15", "2098-01-15"]
for sd in sentinel_dates:
    count = df_credit_bureau_b_2.filter(F.col("pmts_date_1107D") == sd).count()
    print(f"  {sd}: {count} ocorrências")


=== RANGE DE DATAS ===
+-------------+-------------+
|pmts_date_min|pmts_date_max|
+-------------+-------------+
|2016-01-15   |2020-10-15   |
+-------------+-------------+


=== SENTINELAS DE DATA ===
  1900-01-15: 0 ocorrências
  2098-01-15: 0 ocorrências


## 📋 Análise — `train_credit_bureau_b_2`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| num_group2 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| pmts_date_1107D | string | cast → DateType | Data armazenada como string |
| pmts_dpdvalue_108P | double | manter | Métrica de atraso, double correto |
| pmts_pmtsoverdue_635A | double | manter | Valor monetário em atraso, double correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

### Nulos
Tabela praticamente completa — apenas `pmts_dpdvalue_108P` e `pmts_pmtsoverdue_635A` com 0.42% de nulos cada (5.361 linhas), com null_pct idêntico confirmando que formam um grupo funcional (quando um é nulo, o outro também é). Nenhuma ação necessária.

### Sentinelas

| Coluna | Valores sentinela |
|---|---|
| `pmts_dpdvalue_108P` | 99, 999, 9999, 99999 |
| `pmts_pmtsoverdue_635A` | 99 |

### Cardinalidade
- `case_id`: ~37.987 distintos — mesma cobertura de `train_credit_bureau_b_1` (~38K), confirmando que ambas são da mesma fonte de bureau
- `pmts_date_1107D`: 57 distintos — ~4.75 anos mensais (2016–2020), consistente com o range
- `num_group2`: 37 distintos — até 37 pagamentos mensais por contrato (~3 anos de histórico)
- `num_group1`: 21 distintos (0–20) — alinhado com `train_credit_bureau_b_1`

### Chave Composta
`(case_id, num_group1, num_group2)` é chave composta única (0 duplicatas em 1.286.755 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição
Três níveis: `case_id` → `num_group1` (contrato) → `num_group2` (pagamento mensal).  
`num_group1` fortemente concentrado em 0 (723.766) com queda acentuada — mesmo padrão de `train_credit_bureau_b_1`.  
`num_group2` vai até 37 — histórico de até ~3 anos mensais por contrato, menor que `train_credit_bureau_a_2` (104 meses).

### Range de Datas
`pmts_date_1107D`: 2016-01-15 a 2020-10-15 — range válido e sem sentinelas. Cast seguro.

---

### Ações Silver — Casts

```python
df = df.withColumn("pmts_date_1107D", F.to_date(F.col("pmts_date_1107D"), "yyyy-MM-dd"))
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
df = df.withColumn("num_group2", F.col("num_group2").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "pmts_dpdvalue_108P":    ,
    "pmts_pmtsoverdue_635A": ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_b_2: duplicatas na chave (case_id, num_group1, num_group2)"
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)`.  
**Cobertura:** ~38K `case_id` — mesma fonte de `train_credit_bureau_b_1`.  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** praticamente inexistentes — tabela mais limpa do projeto até agora.


## Dataset train_debitcard_1

In [6]:
df_debitcard_1 = load_logical_dataset(spark, bronze_dir, "train_debitcard_1")


Partições encontradas para 'train_debitcard_1':
  train_debitcard_1


In [7]:
df_debitcard_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- last180dayaveragebalance_704A: double (nullable = true)
 |-- last180dayturnover_1134A: double (nullable = true)
 |-- last30dayturnover_651A: double (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- openingdate_857D: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [8]:
from pyspark.sql import functions as F

null_stats = df_debitcard_1.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_debitcard_1.columns
]).unpivot([], df_debitcard_1.columns, "coluna", "null_count")

total = df_debitcard_1.count()
null_stats.withColumn("null_pct", F.round(F.col("null_count") / total * 100, 2)) \
          .orderBy(F.col("null_count").desc()).show(truncate=False)


+-----------------------------+----------+--------+
|coluna                       |null_count|null_pct|
+-----------------------------+----------+--------+
|last180dayturnover_1134A     |146221    |92.96   |
|last30dayturnover_651A       |146221    |92.96   |
|last180dayaveragebalance_704A|145086    |92.23   |
|openingdate_857D             |12711     |8.08    |
|case_id                      |0         |0.0     |
|num_group1                   |0         |0.0     |
|_run_id                      |0         |0.0     |
|_ingest_date                 |0         |0.0     |
|_source_file                 |0         |0.0     |
+-----------------------------+----------+--------+



In [9]:
# ── NULOS ────────────────────────────────────────────────────────────────────
print("=== NULOS ===")

total = df_debitcard_1.count()

null_stats = df_debitcard_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_debitcard_1.columns
]).first()

results = [(c, null_stats[c], round(null_stats[c] / total * 100, 2))
           for c in df_debitcard_1.columns]
results.sort(key=lambda x: x[1], reverse=True)

for col, cnt, pct in results:
    print(f"  {col}: {cnt} ({pct}%)")



=== NULOS ===
  last180dayturnover_1134A: 146221 (92.96%)
  last30dayturnover_651A: 146221 (92.96%)
  last180dayaveragebalance_704A: 145086 (92.23%)
  openingdate_857D: 12711 (8.08%)
  case_id: 0 (0.0%)
  num_group1: 0 (0.0%)
  _run_id: 0 (0.0%)
  _ingest_date: 0 (0.0%)
  _source_file: 0 (0.0%)


In [12]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_debitcard_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_debitcard_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")


# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_debitcard_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_debitcard_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")



=== SENTINELAS — Numéricos ===
  last180dayturnover_1134A: {999: 1, 9999: 9}
=== SENTINELAS — Strings ===
  openingdate_857D: nenhum sentinela detectado


In [13]:
# ── CARDINALIDADE ─────────────────────────────────────────────────────────────
print("=== CARDINALIDADE ===")

cardinality = df_debitcard_1.select([
    F.approx_count_distinct(c).alias(c)
    for c in df_debitcard_1.columns
]).first()

for c in df_debitcard_1.columns:
    print(f"  {c}: {cardinality[c]}")


=== CARDINALIDADE ===
  case_id: 110053
  last180dayaveragebalance_704A: 2519
  last180dayturnover_1134A: 2552
  last30dayturnover_651A: 502
  num_group1: 67
  openingdate_857D: 1502
  _run_id: 1
  _ingest_date: 1
  _source_file: 1


In [14]:
# ── CHAVE COMPOSTA ────────────────────────────────────────────────────────────
print("=== CHAVE COMPOSTA ===")

total = df_debitcard_1.count()
distinct_key = df_debitcard_1.select("case_id", "num_group1").distinct().count()

print(f"  Total rows:                    {total}")
print(f"  Distinct (case_id, num_group1): {distinct_key}")
print(f"  Chave única:                   {total == distinct_key}")


=== CHAVE COMPOSTA ===
  Total rows:                    157302
  Distinct (case_id, num_group1): 157302
  Chave única:                   True


In [15]:
# ── DISTRIBUIÇÃO num_group1 ───────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO num_group1 ===")

df_debitcard_1.groupBy("num_group1").count() \
              .orderBy("num_group1") \
              .show(20, truncate=False)


=== DISTRIBUIÇÃO num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |111772|
|1         |29309 |
|2         |9172  |
|3         |3231  |
|4         |1412  |
|5         |639   |
|6         |397   |
|7         |279   |
|8         |191   |
|9         |140   |
|10        |108   |
|11        |86    |
|12        |73    |
|13        |62    |
|14        |53    |
|15        |45    |
|16        |40    |
|17        |35    |
|18        |29    |
|19        |28    |
+----------+------+
only showing top 20 rows



In [16]:
# ── RANGE DE DATAS ────────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [c for c in df_debitcard_1.columns if c.endswith("D")]

df_debitcard_1.select([
    F.min(c).alias(f"min_{c}") for c in date_cols
] + [
    F.max(c).alias(f"max_{c}") for c in date_cols
]).show(vertical=True, truncate=False)


=== RANGE DE DATAS ===
-RECORD 0--------------------------
 min_openingdate_857D | 2001-11-19 
 max_openingdate_857D | 2017-07-31 



In [17]:
# ── SENTINELAS DE DATA ────────────────────────────────────────────────────────
print("=== SENTINELAS DE DATA ===")

date_sentinels = ["1900-01-15", "2098-01-15"]
date_cols = [c for c in df_debitcard_1.columns if c.endswith("D")]

for c in date_cols:
    for s in date_sentinels:
        cnt = df_debitcard_1.filter(F.col(c) == s).count()
        print(f"  {c} | sentinel={s} | count={cnt}")


=== SENTINELAS DE DATA ===
  openingdate_857D | sentinel=1900-01-15 | count=0
  openingdate_857D | sentinel=2098-01-15 | count=0


In [18]:
# ── PARES SIMILARES ───────────────────────────────────────────────────────────
print("=== PARES SIMILARES ===")

# Colunas de valor desta tabela
# last180dayaveragebalance_704A  → saldo médio 180 dias
# last180dayturnover_1134A       → giro 180 dias
# last30dayturnover_651A         → giro 30 dias
# Candidato a par: last180dayturnover x last30dayturnover (mesmo conceito, janelas distintas)

pair = ("last180dayturnover_1134A", "last30dayturnover_651A")

df_debitcard_1.select(
    F.corr(pair[0], pair[1]).alias("corr_180d_vs_30d_turnover"),
    F.sum((F.col(pair[0]).isNotNull() & F.col(pair[1]).isNotNull()).cast("int")).alias("ambos_nao_nulos"),
    F.sum((F.col(pair[0]).isNull()    & F.col(pair[1]).isNull()).cast("int")).alias("ambos_nulos"),
    F.sum((F.col(pair[0]).isNull()    & F.col(pair[1]).isNotNull()).cast("int")).alias("so_30d_nulo"),
    F.sum((F.col(pair[0]).isNotNull() & F.col(pair[1]).isNull()).cast("int")).alias("so_180d_nulo"),
).show(vertical=True)


=== PARES SIMILARES ===
-RECORD 0---------------------------------------
 corr_180d_vs_30d_turnover | 0.3348244370318042 
 ambos_nao_nulos           | 11081              
 ambos_nulos               | 146221             
 so_30d_nulo               | 0                  
 so_180d_nulo              | 0                  



## 📋 Análise — `train_debitcard_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| `case_id` | long | manter | Parte da chave composta, tipo correto |
| `last180dayaveragebalance_704A` | double | manter | Valor monetário contínuo, double correto |
| `last180dayturnover_1134A` | double | manter | Valor monetário contínuo, double correto |
| `last30dayturnover_651A` | double | manter | Valor monetário contínuo, double correto |
| `num_group1` | long | cast → IntegerType | Índice ordinal, long desnecessário |
| `openingdate_857D` | string | cast → DateType | Data armazenada como string, domínio validado (2001–2017) |
| `_run_id` | string | propagar | Metadado Bronze, não alterar |
| `_ingest_date` | date | propagar | Metadado Bronze, não alterar |
| `_source_file` | string | propagar | Metadado Bronze, não alterar |

### Nulos

`last180dayturnover_1134A` e `last30dayturnover_651A` compartilham exatamente 92.96% de nulos — grupo funcional confirmado pelos pares similares (`ambos_nulos = 146.221`, `so_30d_nulo = 0`, `so_180d_nulo = 0`): quando um é nulo, o outro também é. `last180dayaveragebalance_704A` forma grupo próprio com 92.23% — levemente diferente, pois saldo médio pode existir mesmo sem movimentação registrada. `openingdate_857D` tem 8.08% de nulos, provavelmente contas mais antigas sem data de abertura registrada. Nenhuma imputação na Silver.

### Sentinelas

| Coluna | Valores sentinela |
|---|---|
| `last180dayturnover_1134A` | 999, 9999 |

### Cardinalidade

- `case_id`: 110.053 distintos — cobertura ampla, até 67 cartões por cliente
- `last180dayaveragebalance_704A`: 2.519 distintos — baixa para valor monetário, possível granularidade truncada
- `last180dayturnover_1134A`: 2.552 distintos — idem
- `last30dayturnover_651A`: **502 distintos** — anomalia: muito baixa para valor monetário contínuo, forte indício de arredondamento ou categorização em faixas fixas pelo bureau; investigar distribuição na Gold
- `num_group1`: 67 distintos — até 67 cartões de débito por cliente
- `openingdate_857D`: 1.502 distintos — razoável para datas entre 2001–2017

### Chave Composta

`(case_id, num_group1)` é chave composta única (157.302 linhas = 157.302 distintos). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição

Um nível de agrupamento: `case_id` → `num_group1` (cartão de débito). `num_group1` fortemente concentrado em 0 (111.772 linhas — 71%), com queda exponencial acentuada — a maioria dos clientes tem apenas 1 cartão registrado. Máximo de 67 cartões distintos por cliente, com poucos casos acima de `num_group1 = 10`.

### Range de Datas

`openingdate_857D`: 2001-11-19 a 2017-07-31 — domínio plausível, sem datas futuras suspeitas e sem sentinelas `1900-01-15` ou `2098-01-15`. Cast para `DateType` seguro.

***

### Ações Silver — Casts

```python
df = df.withColumn("openingdate_857D", F.to_date(F.col("openingdate_857D"), "yyyy-MM-dd"))
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "last180dayturnover_1134A": [999, 9999],
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_debitcard_1: duplicatas na chave (case_id, num_group1)"
```

**Granularidade:** dois níveis — `(case_id, num_group1)`.  
**Cobertura:** 110.053 `case_id` — maior cobertura que as tabelas `credit_bureau_b` (~38K).  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** ~93% nas colunas de movimentação — preservar; tabela esparsa por natureza.



## Dataset train_deposit_1

In [19]:
df_deposit_1 = load_logical_dataset(spark, bronze_dir, "train_deposit_1")

Partições encontradas para 'train_deposit_1':
  train_deposit_1


In [20]:
df_deposit_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- amount_416A: double (nullable = true)
 |-- contractenddate_991D: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- openingdate_313D: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [21]:
# ── NULOS ────────────────────────────────────────────────────────────────────
print("=== NULOS ===")

total = df_deposit_1.count()

null_stats = df_deposit_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_deposit_1.columns
]).first()

results = [(c, null_stats[c], round(null_stats[c] / total * 100, 2))
           for c in df_deposit_1.columns]
results.sort(key=lambda x: x[1], reverse=True)

for col, cnt, pct in results:
    print(f"  {col}: {cnt} ({pct}%)")


=== NULOS ===
  contractenddate_991D: 79682 (54.92%)
  case_id: 0 (0.0%)
  amount_416A: 0 (0.0%)
  num_group1: 0 (0.0%)
  openingdate_313D: 0 (0.0%)
  _run_id: 0 (0.0%)
  _ingest_date: 0 (0.0%)
  _source_file: 0 (0.0%)


In [24]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_deposit_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_deposit_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_deposit_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_deposit_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
=== SENTINELAS — Strings ===
  contractenddate_991D: nenhum sentinela detectado
  openingdate_313D: nenhum sentinela detectado


In [25]:
# ── CARDINALIDADE ─────────────────────────────────────────────────────────────
print("=== CARDINALIDADE ===")

cardinality = df_deposit_1.select([
    F.approx_count_distinct(c).alias(c)
    for c in df_deposit_1.columns
]).first()

for c in df_deposit_1.columns:
    print(f"  {c}: {cardinality[c]}")


=== CARDINALIDADE ===
  case_id: 100651
  amount_416A: 40322
  contractenddate_991D: 1445
  num_group1: 66
  openingdate_313D: 1502
  _run_id: 1
  _ingest_date: 1
  _source_file: 1


In [26]:
# ── CHAVE COMPOSTA ────────────────────────────────────────────────────────────
print("=== CHAVE COMPOSTA ===")

total = df_deposit_1.count()
distinct_key = df_deposit_1.select("case_id", "num_group1").distinct().count()

print(f"  Total rows:                     {total}")
print(f"  Distinct (case_id, num_group1): {distinct_key}")
print(f"  Chave única:                    {total == distinct_key}")


=== CHAVE COMPOSTA ===
  Total rows:                     145086
  Distinct (case_id, num_group1): 145086
  Chave única:                    True


In [27]:
# ── DISTRIBUIÇÃO num_group1 ───────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO num_group1 ===")

df_deposit_1.groupBy("num_group1").count() \
            .orderBy("num_group1") \
            .show(20, truncate=False)


=== DISTRIBUIÇÃO num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |105111|
|1         |26045 |
|2         |7878  |
|3         |2701  |
|4         |1190  |
|5         |568   |
|6         |348   |
|7         |251   |
|8         |173   |
|9         |131   |
|10        |100   |
|11        |74    |
|12        |66    |
|13        |55    |
|14        |49    |
|15        |41    |
|16        |37    |
|17        |31    |
|18        |29    |
|19        |26    |
+----------+------+
only showing top 20 rows



In [28]:
# ── RANGE DE DATAS ────────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [c for c in df_deposit_1.columns if c.endswith("D")]

df_deposit_1.select([
    F.min(c).alias(f"min_{c}") for c in date_cols
] + [
    F.max(c).alias(f"max_{c}") for c in date_cols
]).show(vertical=True, truncate=False)


=== RANGE DE DATAS ===
-RECORD 0------------------------------
 min_contractenddate_991D | 2002-02-27 
 min_openingdate_313D     | 2001-11-19 
 max_contractenddate_991D | 2020-03-18 
 max_openingdate_313D     | 2017-07-31 



In [29]:
# ── SENTINELAS DE DATA ────────────────────────────────────────────────────────
print("=== SENTINELAS DE DATA ===")

date_sentinels = ["1900-01-15", "2098-01-15"]
date_cols = [c for c in df_deposit_1.columns if c.endswith("D")]

for c in date_cols:
    for s in date_sentinels:
        cnt = df_deposit_1.filter(F.col(c) == s).count()
        print(f"  {c} | sentinel={s} | count={cnt}")


=== SENTINELAS DE DATA ===
  contractenddate_991D | sentinel=1900-01-15 | count=0
  contractenddate_991D | sentinel=2098-01-15 | count=0
  openingdate_313D | sentinel=1900-01-15 | count=0
  openingdate_313D | sentinel=2098-01-15 | count=0


In [30]:
# ── PARES SIMILARES ───────────────────────────────────────────────────────────
print("=== PARES SIMILARES ===")

# Candidato: contractenddate_991D x openingdate_313D
# (mesmo contrato — verificar sobreposição/coexistência de nulos)

df_deposit_1.select(
    F.sum((F.col("contractenddate_991D").isNotNull() & F.col("openingdate_313D").isNotNull()).cast("int")).alias("ambos_nao_nulos"),
    F.sum((F.col("contractenddate_991D").isNull()    & F.col("openingdate_313D").isNull()).cast("int")).alias("ambos_nulos"),
    F.sum((F.col("contractenddate_991D").isNull()    & F.col("openingdate_313D").isNotNull()).cast("int")).alias("so_enddate_nulo"),
    F.sum((F.col("contractenddate_991D").isNotNull() & F.col("openingdate_313D").isNull()).cast("int")).alias("so_openingdate_nulo"),
).show(vertical=True)


=== PARES SIMILARES ===
-RECORD 0--------------------
 ambos_nao_nulos     | 65404 
 ambos_nulos         | 0     
 so_enddate_nulo     | 79682 
 so_openingdate_nulo | 0     



## 📋 Análise — `train_deposit_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| `case_id` | long | manter | Parte da chave composta, tipo correto |
| `amount_416A` | double | manter | Valor monetário contínuo, double correto |
| `contractenddate_991D` | string | cast → DateType | Data armazenada como string, domínio validado (2002–2020) |
| `num_group1` | long | cast → IntegerType | Índice ordinal, long desnecessário |
| `openingdate_313D` | string | cast → DateType | Data armazenada como string, domínio validado (2001–2017) |
| `_run_id` | string | propagar | Metadado Bronze, não alterar |
| `_ingest_date` | date | propagar | Metadado Bronze, não alterar |
| `_source_file` | string | propagar | Metadado Bronze, não alterar |

### Nulos

Apenas `contractenddate_991D` tem nulos — 54.92% (79.682 linhas). O padrão de pares similares esclarece a semântica: `so_enddate_nulo = 79.682` e `ambos_nulos = 0`, ou seja, toda linha com `openingdate_313D` preenchida pode ou não ter `contractenddate_991D` — depósitos sem data de encerramento são contratos em aberto ou sem prazo definido. Nenhuma imputação na Silver.

### Sentinelas

Nenhum sentinela numérico ou string detectado em nenhuma coluna. Tabela limpa nessa dimensão.

### Cardinalidade

- `case_id`: 100.651 distintos — cobertura próxima à `train_debitcard_1` (110K), mesma ordem de grandeza
- `amount_416A`: 40.322 distintos — alta cardinalidade para valor monetário, consistente com valores contínuos reais
- `contractenddate_991D`: 1.445 distintos — razoável para datas entre 2002–2020 com 54% de nulos
- `openingdate_313D`: 1.502 distintos — idêntico à `train_debitcard_1`, sugerindo mesma fonte de datas de abertura
- `num_group1`: 66 distintos — até 66 depósitos por cliente, estrutura similar à `train_debitcard_1` (67)

### Chave Composta

`(case_id, num_group1)` é chave composta única (145.086 linhas = 145.086 distintos). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição

Um nível de agrupamento: `case_id` → `num_group1` (depósito). `num_group1` fortemente concentrado em 0 (105.111 linhas — 72%), com queda exponencial acentuada — padrão idêntico à `train_debitcard_1`. Máximo de 66 depósitos distintos por cliente, com volume residual acima de `num_group1 = 10`.

### Range de Datas

`openingdate_313D`: 2001-11-19 a 2017-07-31 — domínio idêntico ao `openingdate_857D` da `train_debitcard_1`, confirmando mesma fonte de dados de abertura. `contractenddate_991D`: 2002-02-27 a 2020-03-18 — datas de encerramento posteriores às de abertura, domínio plausível, sem sentinelas. Ambos os casts para `DateType` são seguros.

***

### Ações Silver — Casts

```python
df = df.withColumn("contractenddate_991D", F.to_date(F.col("contractenddate_991D"), "yyyy-MM-dd"))
df = df.withColumn("openingdate_313D", F.to_date(F.col("openingdate_313D"), "yyyy-MM-dd"))
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
# Nenhum sentinela numérico detectado — bloco não aplicável
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_deposit_1: duplicatas na chave (case_id, num_group1)"
```

**Granularidade:** dois níveis — `(case_id, num_group1)`.  
**Cobertura:** 100.651 `case_id` — cobertura ampla, próxima à `train_debitcard_1`.  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** apenas `contractenddate_991D` com 54.92% — depósitos em aberto ou sem prazo; preservar.


## Dataset train_other_1

In [5]:
df_other_1 = load_logical_dataset(spark, bronze_dir, "train_other_1")

Partições encontradas para 'train_other_1':
  train_other_1


In [6]:
df_other_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- amtdebitincoming_4809443A: double (nullable = true)
 |-- amtdebitoutgoing_4809440A: double (nullable = true)
 |-- amtdepositbalance_4809441A: double (nullable = true)
 |-- amtdepositincoming_4809444A: double (nullable = true)
 |-- amtdepositoutgoing_4809442A: double (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [10]:
# ── NULOS ────────────────────────────────────────────────────────────────────
print("=== NULOS ===")

total = df_other_1.count()

null_stats = df_other_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_other_1.columns
]).first()

results = [(c, null_stats[c], round(null_stats[c] / total * 100, 2))
           for c in df_other_1.columns]
results.sort(key=lambda x: x[1], reverse=True)

for col, cnt, pct in results:
    print(f"  {col}: {cnt} ({pct}%)")


=== NULOS ===
  case_id: 0 (0.0%)
  amtdebitincoming_4809443A: 0 (0.0%)
  amtdebitoutgoing_4809440A: 0 (0.0%)
  amtdepositbalance_4809441A: 0 (0.0%)
  amtdepositincoming_4809444A: 0 (0.0%)
  amtdepositoutgoing_4809442A: 0 (0.0%)
  num_group1: 0 (0.0%)
  _run_id: 0 (0.0%)
  _ingest_date: 0 (0.0%)
  _source_file: 0 (0.0%)


In [11]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_other_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_other_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_other_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

if string_cols:
    for c in string_cols:
        counts = df_other_1.select([
            F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
            for v in sentinel_strs
        ]).first()
        hits = {v: counts[v if v else "empty"] for v in sentinel_strs
                if counts[v if v else "empty"] > 0}
        if hits:
            print(f"  {c}: {hits}")
        else:
            print(f"  {c}: nenhum sentinela detectado")
else:
    print("  Nenhuma coluna string elegível — passo não aplicável")


=== SENTINELAS — Numéricos ===
  amtdebitoutgoing_4809440A: {9999: 2}
  amtdepositoutgoing_4809442A: {99: 2}
=== SENTINELAS — Strings ===
  Nenhuma coluna string elegível — passo não aplicável


In [12]:
# ── CARDINALIDADE ─────────────────────────────────────────────────────────────
print("=== CARDINALIDADE ===")

cardinality = df_other_1.select([
    F.approx_count_distinct(c).alias(c)
    for c in df_other_1.columns
]).first()

for c in df_other_1.columns:
    print(f"  {c}: {cardinality[c]}")


=== CARDINALIDADE ===
  case_id: 47258
  amtdebitincoming_4809443A: 10270
  amtdebitoutgoing_4809440A: 9764
  amtdepositbalance_4809441A: 8529
  amtdepositincoming_4809444A: 2633
  amtdepositoutgoing_4809442A: 5582
  num_group1: 1
  _run_id: 1
  _ingest_date: 1
  _source_file: 1


In [13]:
# ── CHAVE COMPOSTA ────────────────────────────────────────────────────────────
print("=== CHAVE COMPOSTA ===")

total = df_other_1.count()
distinct_key = df_other_1.select("case_id", "num_group1").distinct().count()

print(f"  Total rows:                     {total}")
print(f"  Distinct (case_id, num_group1): {distinct_key}")
print(f"  Chave única:                    {total == distinct_key}")


=== CHAVE COMPOSTA ===
  Total rows:                     51109
  Distinct (case_id, num_group1): 51109
  Chave única:                    True


In [14]:
# ── DISTRIBUIÇÃO num_group1 ───────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO num_group1 ===")

df_other_1.groupBy("num_group1").count() \
           .orderBy("num_group1") \
           .show(20, truncate=False)


=== DISTRIBUIÇÃO num_group1 ===
+----------+-----+
|num_group1|count|
+----------+-----+
|0         |51109|
+----------+-----+



In [15]:
# ── RANGE DE DATAS / SENTINELAS DE DATA ──────────────────────────────────────
print("=== RANGE / SENTINELAS DE DATA ===")

date_cols = [c for c in df_other_1.columns if c.endswith("D")]

if date_cols:
    df_other_1.select([
        F.min(c).alias(f"min_{c}") for c in date_cols
    ] + [
        F.max(c).alias(f"max_{c}") for c in date_cols
    ]).show(vertical=True, truncate=False)

    date_sentinels = ["1900-01-15", "2098-01-15"]
    for c in date_cols:
        for s in date_sentinels:
            cnt = df_other_1.filter(F.col(c) == s).count()
            print(f"  {c} | sentinel={s} | count={cnt}")
else:
    print("  Nenhuma coluna *D — passo não aplicável")


=== RANGE / SENTINELAS DE DATA ===
  Nenhuma coluna *D — passo não aplicável


In [16]:
# ── PARES SIMILARES ───────────────────────────────────────────────────────────
print("=== PARES SIMILARES ===")

# Candidatos semânticos:
# amtdebitincoming_4809443A  x  amtdepositincoming_4809444A  (incoming: débito vs depósito)
# amtdebitoutgoing_4809440A  x  amtdepositoutgoing_4809442A  (outgoing: débito vs depósito)

pairs = [
    ("amtdebitincoming_4809443A",  "amtdepositincoming_4809444A"),
    ("amtdebitoutgoing_4809440A",  "amtdepositoutgoing_4809442A"),
]

for a, b in pairs:
    df_other_1.select(
        F.corr(a, b).alias(f"corr"),
        F.sum((F.col(a).isNotNull() & F.col(b).isNotNull()).cast("int")).alias("ambos_nao_nulos"),
        F.sum((F.col(a).isNull()    & F.col(b).isNull()).cast("int")).alias("ambos_nulos"),
        F.sum((F.col(a).isNull()    & F.col(b).isNotNull()).cast("int")).alias("so_b_nulo"),
        F.sum((F.col(a).isNotNull() & F.col(b).isNull()).cast("int")).alias("so_a_nulo"),
    ).show(vertical=True)
    print(f"  Par: {a} x {b}")


=== PARES SIMILARES ===
-RECORD 0------------------------------
 corr            | 0.16210201712715744 
 ambos_nao_nulos | 51109               
 ambos_nulos     | 0                   
 so_b_nulo       | 0                   
 so_a_nulo       | 0                   

  Par: amtdebitincoming_4809443A x amtdepositincoming_4809444A
-RECORD 0------------------------------
 corr            | 0.13715176739625634 
 ambos_nao_nulos | 51109               
 ambos_nulos     | 0                   
 so_b_nulo       | 0                   
 so_a_nulo       | 0                   

  Par: amtdebitoutgoing_4809440A x amtdepositoutgoing_4809442A


## 📋 Análise — `train_other_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| `case_id` | long | manter | Parte da chave composta, tipo correto |
| `amtdebitincoming_4809443A` | double | manter | Valor monetário contínuo, double correto |
| `amtdebitoutgoing_4809440A` | double | manter | Valor monetário contínuo, double correto |
| `amtdepositbalance_4809441A` | double | manter | Valor monetário contínuo, double correto |
| `amtdepositincoming_4809444A` | double | manter | Valor monetário contínuo, double correto |
| `amtdepositoutgoing_4809442A` | double | manter | Valor monetário contínuo, double correto |
| `num_group1` | long | cast → IntegerType | Índice ordinal, long desnecessário |
| `_run_id` | string | propagar | Metadado Bronze, não alterar |
| `_ingest_date` | date | propagar | Metadado Bronze, não alterar |
| `_source_file` | string | propagar | Metadado Bronze, não alterar |

### Nulos

Tabela completamente preenchida — 0% de nulos em todas as colunas. A tabela mais completa do projeto até agora, junto com os metadados. Nenhuma ação necessária.

### Sentinelas

| Coluna | Valores sentinela |
|---|---|
| `amtdebitoutgoing_4809440A` | 9999 |
| `amtdepositoutgoing_4809442A` | 99 |

### Cardinalidade

- `case_id`: 47.258 distintos — cobertura menor que `train_debitcard_1` (110K) e `train_deposit_1` (100K), indicando população mais restrita de clientes com dados "other"
- `amtdebitincoming_4809443A`: 10.270 distintos — cardinalidade razoável para valor monetário contínuo
- `amtdebitoutgoing_4809440A`: 9.764 distintos — idem
- `amtdepositbalance_4809441A`: 8.529 distintos — ligeiramente menor, consistente com saldo sendo agregado
- `amtdepositincoming_4809444A`: **2.633 distintos** — ⚠️ baixa relativamente às demais colunas monetárias; investigar granularidade na Gold
- `amtdepositoutgoing_4809442A`: 5.582 distintos — intermediária
- `num_group1`: **1 distinto** — 🚨 todos os registros têm `num_group1 = 0`; tabela flat sem hierarquia de contratos

### Chave Composta

`(case_id, num_group1)` é chave composta única (51.109 linhas = 51.109 distintos). Como `num_group1` assume exclusivamente o valor 0, a chave efetiva é simplesmente `case_id` — tabela com granularidade de 1 linha por cliente.

### Granularidade e Distribuição

Tabela completamente flat: `num_group1` assume apenas o valor 0 em todas as 51.109 linhas — não há hierarquia de contratos ou produtos. Cada `case_id` tem exatamente um registro, consolidando métricas agregadas de débito e depósito em nível de cliente. Estrutura distinta de todas as tabelas anteriores do projeto.

### Range de Datas

Nenhuma coluna `*D` presente — passo não aplicável.

### Pares Similares

| Par | Correlação | Comportamento de nulos | Observação |
|---|---|---|---|
| `amtdebitincoming` × `amtdepositincoming` | 0.162 | Sem nulos em nenhum dos lados | Correlação muito baixa — canais distintos de entrada |
| `amtdebitoutgoing` × `amtdepositoutgoing` | 0.137 | Sem nulos em nenhum dos lados | Correlação muito baixa — canais distintos de saída |

Correlações abaixo de 0.17 indicam que débito e depósito são produtos financeiros independentes, não redundantes. Manter todos os pares na Silver — eventual consolidação por tipo de fluxo (incoming/outgoing) adiada para Gold.

***

### Ações Silver — Casts

```python
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "amtdebitoutgoing_4809440A":  [9999],
    "amtdepositoutgoing_4809442A": [99],
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_other_1: duplicatas na chave (case_id, num_group1)"
```

**Granularidade:** um nível — `case_id` efetivo (num_group1 sempre = 0).  
**Cobertura:** 47.258 `case_id` — população restrita, menor cobertura dentre as tabelas analisadas.  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** zero — tabela completamente preenchida, mais limpa do projeto.



## Dataset train_person_1

In [18]:
df_person_1 = load_logical_dataset(spark, bronze_dir, "train_person_1")

Partições encontradas para 'train_person_1':
  train_person_1


In [19]:
df_person_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- birth_259D: string (nullable = true)
 |-- birthdate_87D: string (nullable = true)
 |-- childnum_185L: double (nullable = true)
 |-- contaddr_district_15M: string (nullable = true)
 |-- contaddr_matchlist_1032L: boolean (nullable = true)
 |-- contaddr_smempladdr_334L: boolean (nullable = true)
 |-- contaddr_zipcode_807M: string (nullable = true)
 |-- education_927M: string (nullable = true)
 |-- empl_employedfrom_271D: string (nullable = true)
 |-- empl_employedtotal_800L: string (nullable = true)
 |-- empl_industry_691L: string (nullable = true)
 |-- empladdr_district_926M: string (nullable = true)
 |-- empladdr_zipcode_114M: string (nullable = true)
 |-- familystate_447L: string (nullable = true)
 |-- gender_992L: string (nullable = true)
 |-- housetype_905L: string (nullable = true)
 |-- housingtype_772L: string (nullable = true)
 |-- incometype_1044T: string (nullable = true)
 |-- isreference_387L: boolean (nullable = true)
 |-- languag

In [20]:
# ── NULOS ────────────────────────────────────────────────────────────────────
print("=== NULOS ===")

total = df_person_1.count()

null_stats = df_person_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_person_1.columns
]).first()

results = [(c, null_stats[c], round(null_stats[c] / total * 100, 2))
           for c in df_person_1.columns]
results.sort(key=lambda x: x[1], reverse=True)

for col, cnt, pct in results:
    print(f"  {col}: {cnt} ({pct}%)")


=== NULOS ===
  housingtype_772L: 2964176 (99.67%)
  childnum_185L: 2964084 (99.67%)
  maritalst_703L: 2962646 (99.62%)
  birthdate_87D: 2949075 (99.16%)
  gender_992L: 2949075 (99.16%)
  isreference_387L: 2949075 (99.16%)
  role_993L: 2949075 (99.16%)
  housetype_905L: 2873173 (96.61%)
  empl_industry_691L: 2451755 (82.44%)
  empl_employedtotal_800L: 2445676 (82.24%)
  empl_employedfrom_271D: 2407290 (80.94%)
  familystate_447L: 2245378 (75.5%)
  relationshiptoclient_415T: 2168942 (72.93%)
  remitter_829L: 2168942 (72.93%)
  relationshiptoclient_642T: 2168049 (72.9%)
  contaddr_matchlist_1032L: 1447773 (48.68%)
  contaddr_smempladdr_334L: 1447773 (48.68%)
  safeguarantyflag_411L: 1447334 (48.67%)
  birth_259D: 1447332 (48.67%)
  incometype_1044T: 1447332 (48.67%)
  mainoccupationinc_384A: 1447332 (48.67%)
  sex_738L: 1447332 (48.67%)
  personindex_1023L: 642283 (21.6%)
  persontype_792L: 642283 (21.6%)
  persontype_1072L: 6117 (0.21%)
  role_1084L: 6117 (0.21%)
  type_25L: 6117 (0.21%

In [21]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_person_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_person_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_person_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_person_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
=== SENTINELAS — Strings ===
  birth_259D: nenhum sentinela detectado
  birthdate_87D: nenhum sentinela detectado
  contaddr_district_15M: {'a55475b1': 1449674}
  contaddr_zipcode_807M: {'a55475b1': 1474844}
  education_927M: {'a55475b1': 2234573}
  empl_employedfrom_271D: nenhum sentinela detectado
  empl_employedtotal_800L: nenhum sentinela detectado
  empl_industry_691L: nenhum sentinela detectado
  empladdr_district_926M: {'a55475b1': 2448480}
  empladdr_zipcode_114M: {'a55475b1': 2448349}
  familystate_447L: nenhum sentinela detectado
  gender_992L: nenhum sentinela detectado
  housetype_905L: nenhum sentinela detectado
  housingtype_772L: nenhum sentinela detectado
  incometype_1044T: nenhum sentinela detectado
  language1_981M: {'a55475b1': 1505452}
  maritalst_703L: nenhum sentinela detectado
  registaddr_district_1083M: {'a55475b1': 1447423}
  registaddr_zipcode_184M: {'a55475b1': 1515676}
  relationshiptoclient_415T: nenhum sentinela detectado
 

In [22]:
# ── CARDINALIDADE ─────────────────────────────────────────────────────────────
print("=== CARDINALIDADE ===")

cardinality = df_person_1.select([
    F.approx_count_distinct(c).alias(c)
    for c in df_person_1.columns
]).first()

for c in df_person_1.columns:
    print(f"  {c}: {cardinality[c]}")


=== CARDINALIDADE ===
  case_id: 1540952
  birth_259D: 706
  birthdate_87D: 671
  childnum_185L: 11
  contaddr_district_15M: 1004
  contaddr_matchlist_1032L: 1
  contaddr_smempladdr_334L: 2
  contaddr_zipcode_807M: 3336
  education_927M: 6
  empl_employedfrom_271D: 8291
  empl_employedtotal_800L: 3
  empl_industry_691L: 25
  empladdr_district_926M: 214
  empladdr_zipcode_114M: 3203
  familystate_447L: 5
  gender_992L: 2
  housetype_905L: 6
  housingtype_772L: 6
  incometype_1044T: 9
  isreference_387L: 2
  language1_981M: 3
  mainoccupationinc_384A: 6360
  maritalst_703L: 5
  num_group1: 10
  personindex_1023L: 7
  persontype_1072L: 3
  persontype_792L: 3
  registaddr_district_1083M: 1004
  registaddr_zipcode_184M: 3318
  relationshiptoclient_415T: 10
  relationshiptoclient_642T: 10
  remitter_829L: 1
  role_1084L: 3
  role_993L: 1
  safeguarantyflag_411L: 2
  sex_738L: 2
  type_25L: 8
  _run_id: 1
  _ingest_date: 1
  _source_file: 1


In [23]:
# ── CHAVE COMPOSTA ────────────────────────────────────────────────────────────
print("=== CHAVE COMPOSTA ===")

total = df_person_1.count()
distinct_key = df_person_1.select("case_id", "num_group1").distinct().count()

print(f"  Total rows:                     {total}")
print(f"  Distinct (case_id, num_group1): {distinct_key}")
print(f"  Chave única:                    {total == distinct_key}")


=== CHAVE COMPOSTA ===
  Total rows:                     2973991
  Distinct (case_id, num_group1): 2973991
  Chave única:                    True


In [24]:
# ── DISTRIBUIÇÃO num_group1 ───────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO num_group1 ===")

df_person_1.groupBy("num_group1").count() \
            .orderBy("num_group1") \
            .show(20, truncate=False)


=== DISTRIBUIÇÃO num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |1526659|
|1         |757320 |
|2         |484214 |
|3         |181768 |
|4         |22453  |
|5         |1466   |
|6         |99     |
|7         |8      |
|8         |2      |
|9         |2      |
+----------+-------+



In [25]:
# ── RANGE DE DATAS / SENTINELAS DE DATA ──────────────────────────────────────
print("=== RANGE / SENTINELAS DE DATA ===")

date_cols = [c for c in df_person_1.columns if c.endswith("D")]

df_person_1.select([
    F.min(c).alias(f"min_{c}") for c in date_cols
] + [
    F.max(c).alias(f"max_{c}") for c in date_cols
]).show(vertical=True, truncate=False)

date_sentinels = ["1900-01-15", "2098-01-15"]
for c in date_cols:
    for s in date_sentinels:
        cnt = df_person_1.filter(F.col(c) == s).count()
        print(f"  {c} | sentinel={s} | count={cnt}")


=== RANGE / SENTINELAS DE DATA ===
-RECORD 0--------------------------------
 min_birth_259D             | 1943-03-01 
 min_birthdate_87D          | 1911-12-01 
 min_empl_employedfrom_271D | 1963-06-15 
 max_birth_259D             | 1999-10-01 
 max_birthdate_87D          | 2015-12-01 
 max_empl_employedfrom_271D | 2020-09-15 

  birth_259D | sentinel=1900-01-15 | count=0
  birth_259D | sentinel=2098-01-15 | count=0
  birthdate_87D | sentinel=1900-01-15 | count=0
  birthdate_87D | sentinel=2098-01-15 | count=0
  empl_employedfrom_271D | sentinel=1900-01-15 | count=0
  empl_employedfrom_271D | sentinel=2098-01-15 | count=0


In [26]:
# ── PARES SIMILARES ───────────────────────────────────────────────────────────
print("=== PARES SIMILARES ===")

# Pares candidatos identificados no schema:
# birth_259D          x birthdate_87D           (data de nascimento duplicada?)
# contaddr_district_15M x empladdr_district_926M x registaddr_district_1083M (distrito em 3 endereços)
# contaddr_zipcode_807M x empladdr_zipcode_114M  x registaddr_zipcode_184M   (CEP em 3 endereços)
# relationshiptoclient_415T x relationshiptoclient_642T                      (relacionamento duplicado?)
# role_1084L          x role_993L                                            (papel duplicado?)
# persontype_1072L    x persontype_792L                                      (tipo de pessoa duplicado?)
# sex_738L            x gender_992L                                          (sexo/gênero duplicado?)

pairs = [
    ("birth_259D",               "birthdate_87D"),
    ("relationshiptoclient_415T","relationshiptoclient_642T"),
    ("role_1084L",               "role_993L"),
    ("persontype_1072L",         "persontype_792L"),
    ("sex_738L",                 "gender_992L"),
]

for a, b in pairs:
    result = df_person_1.select(
        F.sum((F.col(a).isNotNull() & F.col(b).isNotNull()).cast("int")).alias("ambos_nao_nulos"),
        F.sum((F.col(a).isNull()    & F.col(b).isNull()).cast("int")).alias("ambos_nulos"),
        F.sum((F.col(a).isNull()    & F.col(b).isNotNull()).cast("int")).alias("so_b_preenchido"),
        F.sum((F.col(a).isNotNull() & F.col(b).isNull()).cast("int")).alias("so_a_preenchido"),
    ).first()
    print(f"\n  Par: {a} x {b}")
    print(f"    ambos_nao_nulos : {result['ambos_nao_nulos']}")
    print(f"    ambos_nulos     : {result['ambos_nulos']}")
    print(f"    so_{a} : {result['so_a_preenchido']}")
    print(f"    so_{b} : {result['so_b_preenchido']}")


=== PARES SIMILARES ===

  Par: birth_259D x birthdate_87D
    ambos_nao_nulos : 12458
    ambos_nulos     : 1434874
    so_birth_259D : 1514201
    so_birthdate_87D : 12458

  Par: relationshiptoclient_415T x relationshiptoclient_642T
    ambos_nao_nulos : 358128
    ambos_nulos     : 1721128
    so_relationshiptoclient_415T : 446921
    so_relationshiptoclient_642T : 447814

  Par: role_1084L x role_993L
    ambos_nao_nulos : 18799
    ambos_nulos     : 0
    so_role_1084L : 2949075
    so_role_993L : 6117

  Par: persontype_1072L x persontype_792L
    ambos_nao_nulos : 2331708
    ambos_nulos     : 6117
    so_persontype_1072L : 636166
    so_persontype_792L : 0

  Par: sex_738L x gender_992L
    ambos_nao_nulos : 12458
    ambos_nulos     : 1434874
    so_sex_738L : 1514201
    so_gender_992L : 12458


## 📋 Análise — `train_person_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| `case_id` | long | manter | Parte da chave composta, tipo correto |
| `birth_259D` | string | cast → DateType | Data armazenada como string, domínio validado (1943–1999) |
| `birthdate_87D` | string | cast → DateType | Data armazenada como string, domínio validado (1911–2015) |
| `childnum_185L` | double | cast → IntegerType | Contagem de filhos, nunca fracionária |
| `contaddr_district_15M` | string | manter | Categórica de distrito, string correto |
| `contaddr_matchlist_1032L` | boolean | manter | Flag booleana, tipo correto |
| `contaddr_smempladdr_334L` | boolean | manter | Flag booleana, tipo correto |
| `contaddr_zipcode_807M` | string | manter | CEP, string correto |
| `education_927M` | string | manter | Categórica de educação, string correto |
| `empl_employedfrom_271D` | string | cast → DateType | Data armazenada como string, domínio validado (1963–2020) |
| `empl_employedtotal_800L` | string | manter | Categórica ordinal de tempo empregado, string correto |
| `empl_industry_691L` | string | manter | Categórica de indústria, string correto |
| `empladdr_district_926M` | string | manter | Categórica de distrito, string correto |
| `empladdr_zipcode_114M` | string | manter | CEP, string correto |
| `familystate_447L` | string | manter | Categórica de estado familiar, string correto |
| `gender_992L` | string | manter | Categórica de gênero, string correto |
| `housetype_905L` | string | manter | Categórica de tipo de moradia, string correto |
| `housingtype_772L` | string | manter | Categórica de tipo de habitação, string correto |
| `incometype_1044T` | string | manter | Categórica de tipo de renda, string correto |
| `isreference_387L` | boolean | manter | Flag booleana, tipo correto |
| `language1_981M` | string | manter | Categórica de idioma, string correto |
| `mainoccupationinc_384A` | double | manter | Valor monetário de renda, double correto |
| `maritalst_703L` | string | manter | Categórica de estado civil, string correto |
| `num_group1` | long | cast → IntegerType | Índice ordinal, long desnecessário |
| `personindex_1023L` | double | cast → IntegerType | Índice de pessoa, nunca fracionário |
| `persontype_1072L` | double | cast → IntegerType | Tipo de pessoa codificado, nunca fracionário |
| `persontype_792L` | double | cast → IntegerType | Tipo de pessoa codificado, nunca fracionário |
| `registaddr_district_1083M` | string | manter | Categórica de distrito, string correto |
| `registaddr_zipcode_184M` | string | manter | CEP, string correto |
| `relationshiptoclient_415T` | string | manter | Categórica de relacionamento, string correto |
| `relationshiptoclient_642T` | string | manter | Categórica de relacionamento, string correto |
| `remitter_829L` | boolean | manter | Flag booleana, tipo correto |
| `role_1084L` | string | manter | Categórica de papel, string correto |
| `role_993L` | string | manter | Categórica de papel, string correto |
| `safeguarantyflag_411L` | boolean | manter | Flag booleana, tipo correto |
| `sex_738L` | string | manter | Categórica de sexo, string correto |
| `type_25L` | string | manter | Categórica de tipo, string correto |
| `_run_id` | string | propagar | Metadado Bronze, não alterar |
| `_ingest_date` | date | propagar | Metadado Bronze, não alterar |
| `_source_file` | string | propagar | Metadado Bronze, não alterar |

### Nulos

A tabela apresenta cinco grupos funcionais bem definidos por null_pct idêntico. **Grupo A (~99.6–99.7%):** `housingtype_772L`, `childnum_185L`, `maritalst_703L` — atributos pessoais opcionais, quase ausentes. **Grupo B (99.16%):** `birthdate_87D`, `gender_992L`, `isreference_387L`, `role_993L` — todas ligadas a pessoas secundárias (num_group1 > 0), confirmado pela distribuição. **Grupo C (~80–82%):** `empl_industry_691L`, `empl_employedtotal_800L`, `empl_employedfrom_271D` — dados de emprego ausentes para desempregados ou não informados. **Grupo D (~72–75%):** `familystate_447L`, `relationshiptoclient_415T`, `remitter_829L`, `relationshiptoclient_642T` — atributos de pessoas relacionadas ao requerente. **Grupo E (~48.67%):** `contaddr_matchlist_1032L`, `contaddr_smempladdr_334L`, `safeguarantyflag_411L`, `birth_259D`, `incometype_1044T`, `mainoccupationinc_384A`, `sex_738L` — grupo do requerente principal, preenchido em ~51% dos casos. Nenhuma imputação na Silver.

### Sentinelas

| Coluna | Valores sentinela |
|---|---|
| `contaddr_district_15M` | `a55475b1` (1.449.674) |
| `contaddr_zipcode_807M` | `a55475b1` (1.474.844) |
| `education_927M` | `a55475b1` (2.234.573) |
| `empladdr_district_926M` | `a55475b1` (2.448.480) |
| `empladdr_zipcode_114M` | `a55475b1` (2.448.349) |
| `language1_981M` | `a55475b1` (1.505.452) |
| `registaddr_district_1083M` | `a55475b1` (1.447.423) |
| `registaddr_zipcode_184M` | `a55475b1` (1.515.676) |

Nenhum sentinela numérico detectado. Todas as colunas `*M` contêm `a55475b1` — padrão global confirmado. Substituir por `null` na Silver.

### Cardinalidade

- `case_id`: 1.540.952 distintos — maior cobertura do projeto, tabela central de pessoas
- `birth_259D` / `birthdate_87D`: 706 / 671 distintos — datas truncadas ao mês (padrão `YYYY-MM-01`), não ao dia
- `contaddr_zipcode_807M`: 3.336 distintos — alta granularidade geográfica de endereço de contato
- `registaddr_zipcode_184M`: 3.318 distintos — idem para endereço de registro
- `empladdr_zipcode_114M`: 3.203 distintos — idem para endereço de emprego
- `mainoccupationinc_384A`: 6.360 distintos — cardinalidade razoável para renda
- `empl_employedfrom_271D`: 8.291 distintos — maior cardinalidade de data, granularidade diária
- `contaddr_matchlist_1032L`: **1 distinto** — 🚨 coluna constante (apenas `false` ou apenas `true`); sem valor preditivo, candidata a drop na Gold
- `remitter_829L`: **1 distinto** — 🚨 idem
- `role_993L`: **1 distinto** — 🚨 idem
- `empl_employedtotal_800L`: 3 distintos — categórica ordinal simples (ex: curto/médio/longo)
- `language1_981M`: 3 distintos — poucos idiomas registrados

### Chave Composta

`(case_id, num_group1)` é chave composta única. Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição

Dois níveis: `case_id` → `num_group1` (pessoa relacionada). `num_group1 = 0` representa o requerente principal (1.526.659 linhas — 51%), com queda acentuada nos índices seguintes — a maioria dos casos registra até 2 pessoas adicionais (`num_group1` 1 e 2 somam ~41%). Máximo de `num_group1 = 9`, com apenas 2 registros em cada extremo — dados residuais.

### Range de Datas

`birth_259D`: 1943-03-01 a 1999-10-01 — domínio plausível para datas de nascimento do requerente, granularidade mensal. `birthdate_87D`: 1911-12-01 a 2015-12-01 — range mais amplo, inclui pessoas mais velhas e potencialmente menores (pessoas relacionadas); `max = 2015` merece atenção na Gold para filtrar menores. `empl_employedfrom_271D`: 1963-06-15 a 2020-09-15 — domínio plausível para início de emprego, sem sentinelas. Todos os casts para `DateType` são seguros.

### Pares Similares

| Par | ambos_preenchidos | só_A | só_B | ambos_nulos | Interpretação |
|---|---|---|---|---|---|
| `birth_259D` × `birthdate_87D` | 12.458 | 1.514.201 | 12.458 | 1.434.874 | Complementares — `birth_259D` é do requerente principal, `birthdate_87D` de pessoas secundárias; não redundantes |
| `sex_738L` × `gender_992L` | 12.458 | 1.514.201 | 12.458 | 1.434.874 | Mesmo padrão exato de `birth_259D` × `birthdate_87D` — mesma segmentação requerente vs secundário |
| `relationshiptoclient_415T` × `_642T` | 358.128 | 446.921 | 447.814 | 1.721.128 | Parcialmente sobrepostos — populações ligeiramente distintas; manter ambos na Silver |
| `role_1084L` × `role_993L` | 18.799 | 2.949.075 | 6.117 | 0 | `role_993L` quase sempre nulo (99.16%) com 1 distinto — candidato a drop na Gold |
| `persontype_1072L` × `persontype_792L` | 2.331.708 | 636.166 | 0 | 6.117 | `persontype_792L` nunca preenchido sem `persontype_1072L` — hierarquia clara; manter ambos na Silver |

***

### Ações Silver — Casts

```python
df = df.withColumn("birth_259D",            F.to_date(F.col("birth_259D"),            "yyyy-MM-dd"))
df = df.withColumn("birthdate_87D",         F.to_date(F.col("birthdate_87D"),         "yyyy-MM-dd"))
df = df.withColumn("empl_employedfrom_271D",F.to_date(F.col("empl_employedfrom_271D"),"yyyy-MM-dd"))
df = df.withColumn("num_group1",      F.col("num_group1").cast("int"))
df = df.withColumn("childnum_185L",   F.col("childnum_185L").cast("int"))
df = df.withColumn("personindex_1023L", F.col("personindex_1023L").cast("int"))
df = df.withColumn("persontype_1072L",  F.col("persontype_1072L").cast("int"))
df = df.withColumn("persontype_792L",   F.col("persontype_792L").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
# Nenhum sentinela numérico detectado — bloco não aplicável
```

### Ações Silver — Sentinelas *M → null

```python
m_cols = [
    "contaddr_district_15M",
    "contaddr_zipcode_807M",
    "education_927M",
    "empladdr_district_926M",
    "empladdr_zipcode_114M",
    "language1_981M",
    "registaddr_district_1083M",
    "registaddr_zipcode_184M",
]

for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_person_1: duplicatas na chave (case_id, num_group1)"
```

**Granularidade:** dois níveis — `(case_id, num_group1)` onde `num_group1 = 0` é o requerente principal.  
**Cobertura:** 1.540.952 `case_id` — maior cobertura do projeto.  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** estruturais e esperados — refletem segmentação entre requerente principal e pessoas relacionadas; preservar todos.  
**Atenção Gold:** `contaddr_matchlist_1032L`, `remitter_829L` e `role_993L` com 1 distinto — candidatos a drop; `birthdate_87D` com max 2015 — filtrar menores se necessário.



## Dataset train_person_2

In [30]:
df_person_2 = load_logical_dataset(spark, bronze_dir, "train_person_2")

Partições encontradas para 'train_person_2':
  train_person_2


In [31]:
df_person_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- addres_district_368M: string (nullable = true)
 |-- addres_role_871L: string (nullable = true)
 |-- addres_zip_823M: string (nullable = true)
 |-- conts_role_79M: string (nullable = true)
 |-- empls_economicalst_849M: string (nullable = true)
 |-- empls_employedfrom_796D: string (nullable = true)
 |-- empls_employer_name_740M: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- relatedpersons_role_762T: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [32]:
# ── NULOS ────────────────────────────────────────────────────────────────────
print("=== NULOS ===")

total = df_person_2.count()

null_stats = df_person_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_person_2.columns
]).first()

results = [(c, null_stats[c], round(null_stats[c] / total * 100, 2))
           for c in df_person_2.columns]
results.sort(key=lambda x: x[1], reverse=True)

for col, cnt, pct in results:
    print(f"  {col}: {cnt} ({pct}%)")


=== NULOS ===
  empls_employedfrom_796D: 1637653 (99.65%)
  relatedpersons_role_762T: 1614684 (98.25%)
  addres_role_871L: 1575736 (95.88%)
  case_id: 0 (0.0%)
  addres_district_368M: 0 (0.0%)
  addres_zip_823M: 0 (0.0%)
  conts_role_79M: 0 (0.0%)
  empls_economicalst_849M: 0 (0.0%)
  empls_employer_name_740M: 0 (0.0%)
  num_group1: 0 (0.0%)
  num_group2: 0 (0.0%)
  _run_id: 0 (0.0%)
  _ingest_date: 0 (0.0%)
  _source_file: 0 (0.0%)


In [33]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_person_2.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

if numeric_cols:
    for c in numeric_cols:
        counts = df_person_2.select([
            F.sum((F.col(c) == v).cast("int")).alias(str(v))
            for v in sentinel_vals
        ]).first()
        hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
        if hits:
            print(f"  {c}: {hits}")
else:
    print("  Nenhuma coluna numérica elegível — passo não aplicável")
# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_person_2.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_person_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  Nenhuma coluna numérica elegível — passo não aplicável
=== SENTINELAS — Strings ===
  addres_district_368M: {'a55475b1': 1582872}
  addres_role_871L: nenhum sentinela detectado
  addres_zip_823M: {'a55475b1': 1576370}
  conts_role_79M: {'a55475b1': 1587829}
  empls_economicalst_849M: {'a55475b1': 1618686}
  empls_employedfrom_796D: nenhum sentinela detectado
  empls_employer_name_740M: {'a55475b1': 1636009}
  relatedpersons_role_762T: nenhum sentinela detectado


In [34]:
# ── CARDINALIDADE ─────────────────────────────────────────────────────────────
print("=== CARDINALIDADE ===")

cardinality = df_person_2.select([
    F.approx_count_distinct(c).alias(c)
    for c in df_person_2.columns
]).first()

for c in df_person_2.columns:
    print(f"  {c}: {cardinality[c]}")


=== CARDINALIDADE ===
  case_id: 1446340
  addres_district_368M: 524
  addres_role_871L: 8
  addres_zip_823M: 1899
  conts_role_79M: 11
  empls_economicalst_849M: 10
  empls_employedfrom_796D: 857
  empls_employer_name_740M: 6815
  num_group1: 5
  num_group2: 32
  relatedpersons_role_762T: 10
  _run_id: 1
  _ingest_date: 1
  _source_file: 1


In [35]:
# ── CHAVE COMPOSTA ────────────────────────────────────────────────────────────
print("=== CHAVE COMPOSTA ===")

total = df_person_2.count()
distinct_key = df_person_2.select("case_id", "num_group1", "num_group2").distinct().count()

print(f"  Total rows:                                  {total}")
print(f"  Distinct (case_id, num_group1, num_group2):  {distinct_key}")
print(f"  Chave única:                                 {total == distinct_key}")


=== CHAVE COMPOSTA ===
  Total rows:                                  1643410
  Distinct (case_id, num_group1, num_group2):  1643410
  Chave única:                                 True


In [36]:
# ── DISTRIBUIÇÃO num_group1 ───────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO num_group1 ===")
df_person_2.groupBy("num_group1").count() \
            .orderBy("num_group1") \
            .show(20, truncate=False)

print("=== DISTRIBUIÇÃO num_group2 ===")
df_person_2.groupBy("num_group2").count() \
            .orderBy("num_group2") \
            .show(20, truncate=False)


=== DISTRIBUIÇÃO num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |1463928|
|1         |175805 |
|2         |3529   |
|3         |145    |
|4         |3      |
+----------+-------+

=== DISTRIBUIÇÃO num_group2 ===
+----------+-------+
|num_group2|count  |
+----------+-------+
|0         |1561280|
|1         |44507  |
|2         |11110  |
|3         |8411   |
|4         |5356   |
|5         |4213   |
|6         |2831   |
|7         |1979   |
|8         |1232   |
|9         |860    |
|10        |523    |
|11        |341    |
|12        |225    |
|13        |150    |
|14        |97     |
|15        |74     |
|16        |58     |
|17        |39     |
|18        |27     |
|19        |20     |
+----------+-------+
only showing top 20 rows



In [37]:
# ── RANGE DE DATAS / SENTINELAS DE DATA ──────────────────────────────────────
print("=== RANGE / SENTINELAS DE DATA ===")

date_cols = [c for c in df_person_2.columns if c.endswith("D")]

df_person_2.select([
    F.min(c).alias(f"min_{c}") for c in date_cols
] + [
    F.max(c).alias(f"max_{c}") for c in date_cols
]).show(vertical=True, truncate=False)

date_sentinels = ["1900-01-15", "2098-01-15"]
for c in date_cols:
    for s in date_sentinels:
        cnt = df_person_2.filter(F.col(c) == s).count()
        print(f"  {c} | sentinel={s} | count={cnt}")


=== RANGE / SENTINELAS DE DATA ===
-RECORD 0---------------------------------
 min_empls_employedfrom_796D | 1972-08-15 
 max_empls_employedfrom_796D | 2020-01-19 

  empls_employedfrom_796D | sentinel=1900-01-15 | count=0
  empls_employedfrom_796D | sentinel=2098-01-15 | count=0


In [38]:
# ── PARES SIMILARES ───────────────────────────────────────────────────────────
print("=== PARES SIMILARES ===")

# Candidatos identificados no schema:
# addres_district_368M x addres_zip_823M  (mesmo endereço — distrito vs CEP)
# empls_economicalst_849M x empls_employedfrom_796D x empls_employer_name_740M (mesmo emprego)
# conts_role_79M x relatedpersons_role_762T (papel de contato vs papel de pessoa relacionada)

pairs = [
    ("conts_role_79M", "relatedpersons_role_762T"),
]

for a, b in pairs:
    result = df_person_2.select(
        F.sum((F.col(a).isNotNull() & F.col(b).isNotNull()).cast("int")).alias("ambos_nao_nulos"),
        F.sum((F.col(a).isNull()    & F.col(b).isNull()).cast("int")).alias("ambos_nulos"),
        F.sum((F.col(a).isNull()    & F.col(b).isNotNull()).cast("int")).alias("so_b_preenchido"),
        F.sum((F.col(a).isNotNull() & F.col(b).isNull()).cast("int")).alias("so_a_preenchido"),
    ).first()
    print(f"\n  Par: {a} x {b}")
    print(f"    ambos_nao_nulos  : {result['ambos_nao_nulos']}")
    print(f"    ambos_nulos      : {result['ambos_nulos']}")
    print(f"    so_{a}: {result['so_a_preenchido']}")
    print(f"    so_{b}: {result['so_b_preenchido']}")


=== PARES SIMILARES ===

  Par: conts_role_79M x relatedpersons_role_762T
    ambos_nao_nulos  : 28726
    ambos_nulos      : 0
    so_conts_role_79M: 1614684
    so_relatedpersons_role_762T: 0


## 📋 Análise — `train_person_2`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| `case_id` | long | manter | Parte da chave composta, tipo correto |
| `addres_district_368M` | string | manter | Categórica de distrito, string correto |
| `addres_role_871L` | string | manter | Categórica de papel do endereço, string correto |
| `addres_zip_823M` | string | manter | CEP, string correto |
| `conts_role_79M` | string | manter | Categórica de papel do contato, string correto |
| `empls_economicalst_849M` | string | manter | Categórica de status econômico, string correto |
| `empls_employedfrom_796D` | string | cast → DateType | Data armazenada como string, domínio validado (1972–2020) |
| `empls_employer_name_740M` | string | manter | Nome do empregador, string correto |
| `num_group1` | long | cast → IntegerType | Índice ordinal, long desnecessário |
| `num_group2` | long | cast → IntegerType | Índice ordinal, long desnecessário |
| `relatedpersons_role_762T` | string | manter | Categórica de papel de pessoa relacionada, string correto |
| `_run_id` | string | propagar | Metadado Bronze, não alterar |
| `_ingest_date` | date | propagar | Metadado Bronze, não alterar |
| `_source_file` | string | propagar | Metadado Bronze, não alterar |

### Nulos

Três grupos funcionais identificados:

- **Grupo A (~99.65%):** `empls_employedfrom_796D` — data de início de emprego ausente na quase totalidade dos registros; preenchida apenas para empregados com data conhecida
- **Grupo B (~98.25%):** `relatedpersons_role_762T` — papel de pessoa relacionada ausente na maioria; preenchido apenas quando `num_group2` representa uma pessoa relacionada específica
- **Grupo C (~95.88%):** `addres_role_871L` — papel do endereço ausente para a maioria; preenchido apenas para endereços com papel explícito

Colunas `addres_district_368M`, `addres_zip_823M`, `conts_role_79M`, `empls_economicalst_849M`, `empls_employer_name_740M` e chaves com 0% de nulos — mas atenção: volume real de dados válidos é muito menor após remoção dos `a55475b1`. Nenhuma imputação na Silver.

### Sentinelas

| Coluna | Valores sentinela |
|---|---|
| `addres_district_368M` | `a55475b1` (1.582.872) |
| `addres_zip_823M` | `a55475b1` (1.576.370) |
| `conts_role_79M` | `a55475b1` (1.587.829) |
| `empls_economicalst_849M` | `a55475b1` (1.618.686) |
| `empls_employer_name_740M` | `a55475b1` (1.636.009) |

Nenhum sentinela numérico detectado. `addres_role_871L`, `empls_employedfrom_796D` e `relatedpersons_role_762T` sem sentinelas — seus nulos são nulos reais. Atenção: `empls_employer_name_740M` não termina em `M` mas contém `a55475b1` — substituir igualmente na Silver.

### Cardinalidade

- `case_id`: 1.446.340 distintos — cobertura próxima à `train_person_1` (1.54M), esperado para tabela de pessoas
- `empls_employer_name_740M`: 6.815 distintos — alta cardinalidade para nome de empregador; após remoção de `a55475b1` (~1.6M linhas), dados válidos são escassos
- `addres_zip_823M`: 1.899 distintos — cobertura geográfica razoável
- `addres_district_368M`: 524 distintos — menor granularidade que CEP, esperado
- `empls_employedfrom_796D`: 857 distintos — baixa para data (99.65% nulos explicam)
- `conts_role_79M`: 11 distintos; `relatedpersons_role_762T`: 10; `addres_role_871L`: 8 — categóricas de baixa cardinalidade, coerente
- `num_group1`: 5 distintos (0–4) — até 4 pessoas relacionadas por caso
- `num_group2`: 32 distintos (0–31+) — até 31 endereços/contatos por pessoa

### Chave Composta

`(case_id, num_group1, num_group2)` é chave composta única (1.643.410 linhas = 1.643.410 distintos). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição

Três níveis: `case_id` → `num_group1` (pessoa) → `num_group2` (endereço/contato da pessoa). `num_group1` concentrado em 0 (1.463.928 — 89%), com queda abrupta — a maioria dos casos tem apenas uma pessoa registrada. `num_group2` concentrado em 0 (1.561.280 — 95%) com cauda longa até 31 — cada pessoa pode ter múltiplos endereços ou contatos registrados, mas o volume é residual acima de `num_group2 = 5`.

### Range de Datas

`empls_employedfrom_796D`: 1972-08-15 a 2020-01-19 — domínio plausível para data de início de emprego; sem sentinelas `1900-01-15` ou `2098-01-15`. Cast para `DateType` seguro.

### Pares Similares

| Par | ambos_preenchidos | comportamento | Estratégia |
|---|---|---|---|
| `conts_role_79M` × `relatedpersons_role_762T` | 28.726 | `conts_role_79M` presente em 1.614.684 linhas onde `relatedpersons_role_762T` é nulo — `conts_role_79M` é o campo primário | Manter ambos na Silver — semânticas distintas (papel do contato vs papel da pessoa relacionada); consolidar na Gold |

***

### Ações Silver — Casts

```python
df = df.withColumn("empls_employedfrom_796D", F.to_date(F.col("empls_employedfrom_796D"), "yyyy-MM-dd"))
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
df = df.withColumn("num_group2", F.col("num_group2").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
# Nenhum sentinela numérico detectado — bloco não aplicável
```

### Ações Silver — Sentinelas *M → null

```python
m_cols = [
    "addres_district_368M",
    "addres_zip_823M",
    "conts_role_79M",
    "empls_economicalst_849M",
    "empls_employer_name_740M",
]

for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_person_2: duplicatas na chave (case_id, num_group1, num_group2)"
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)`.  
**Cobertura:** 1.446.340 `case_id` — cobertura quase total, próxima à `train_person_1`.  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** três grupos funcionais refletindo preenchimento condicional por tipo de registro — preservar todos.


## Dataset train_static_0

In [6]:
df_static_0 = load_logical_dataset(spark, bronze_dir, "train_static_0")

Partições encontradas para 'train_static_0':
  train_static_0_0
  train_static_0_1


In [40]:
df_static_0.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- actualdpdtolerance_344P: double (nullable = true)
 |-- amtinstpaidbefduel24m_4187115A: double (nullable = true)
 |-- annuity_780A: double (nullable = true)
 |-- annuitynextmonth_57A: double (nullable = true)
 |-- applicationcnt_361L: double (nullable = true)
 |-- applications30d_658L: double (nullable = true)
 |-- applicationscnt_1086L: double (nullable = true)
 |-- applicationscnt_464L: double (nullable = true)
 |-- applicationscnt_629L: double (nullable = true)
 |-- applicationscnt_867L: double (nullable = true)
 |-- avgdbddpdlast24m_3658932P: double (nullable = true)
 |-- avgdbddpdlast3m_4187120P: double (nullable = true)
 |-- avgdbdtollast24m_4525197P: double (nullable = true)
 |-- avgdpdtolclosure24_3658938P: double (nullable = true)
 |-- avginstallast24m_3658937A: double (nullable = true)
 |-- avglnamtstart24m_4525187A: double (nullable = true)
 |-- avgmaxdpdlast9m_3716943P: double (nullable = true)
 |-- avgoutstandbalancel6m_4187114

In [7]:
from __future__ import annotations
from pyspark.sql import functions as F

total = df_static_0.count()
print(f"Total de linhas: {total}\n")

null_counts = df_static_0.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_static_0.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_static_0.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<50} {'null_count':>12} {'null_pct':>10}")
print("-" * 75)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<50} {null_count:>12} {null_pct:>9}%")


Total de linhas: 1526659

coluna                                               null_count   null_pct
---------------------------------------------------------------------------
clientscnt_136L                                         1526242     99.97%
lastrepayingdate_696D                                   1524282     99.84%
lastotherinc_902A                                       1523603      99.8%
lastotherlnsexpense_631A                                1523601      99.8%
payvacationpostpone_4187118D                            1517330     99.39%
isbidproductrequest_292L                                1514201     99.18%
interestrategrace_34L                                   1505776     98.63%
lastdependentsnum_448L                                  1497142     98.07%
equalityempfrom_62L                                     1488847     97.52%
maxannuity_4075009A                                     1450726     95.03%
equalitydataagreement_891L                              1448632     94.89

In [8]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_static_0.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id",)]

for c in numeric_cols:
    counts = df_static_0.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]

string_cols = [c for c, t in df_static_0.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_static_0.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")


=== SENTINELAS — Numéricos ===
  actualdpdtolerance_344P: {99: 4}
  amtinstpaidbefduel24m_4187115A: {9999: 12}
  annuity_780A: {999: 64, 9999: 10}
  annuitynextmonth_57A: {999: 33, 9999: 1}
  applicationscnt_1086L: {99: 3}
  applicationscnt_464L: {99: 53}
  applicationscnt_867L: {99: 1}
  avgdbddpdlast24m_3658932P: {99: 30, 999: 10, -1: 73885}
  avgdbddpdlast3m_4187120P: {99: 8, 999: 5, -1: 61517}
  avgdbdtollast24m_4525197P: {99: 27, 999: 5, -1: 45814}
  avgdpdtolclosure24_3658938P: {99: 68, 999: 19}
  avginstallast24m_3658937A: {999: 22, 9999: 7}
  avgmaxdpdlast9m_3716943P: {99: 9}
  avgoutstandbalancel6m_4187114A: {999: 2, 9999: 5, 99999: 1, -1: 110}
  avgpmtlast12m_4525200A: {999: 12, 9999: 6}
  clientscnt_493L: {99: 1}
  clientscnt_887L: {99: 28, 999: 1}
  credamount_770A: {9999: 8}
  currdebt_22A: {999: 1}
  currdebtcredtyperange_828A: {999: 2}
  daysoverduetolerancedd_3976961L: {99: 576, 999: 37}
  disbursedcredamount_1113A: {9999: 9}
  downpmt_116A: {99: 3}
  inittransactionamo

In [9]:
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_static_0.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_static_0.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<50} {'approx_nunique':>15}")
print("-" * 67)
for coluna, nunique in rows_card:
    flag = " ← CONSTANTE ⚠️" if nunique == 1 else ""
    print(f"{coluna:<50} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                              approx_nunique
-------------------------------------------------------------------
case_id                                                    1540952
totalsettled_863A                                           746865
amtinstpaidbefduel24m_4187115A                              623650
avgoutstandbalancel6m_4187114A                              607297
maxoutstandbalancel12m_4187113A                             475009
maxdebt4_972A                                               446234
maxannuity_159A                                             358515
currdebt_22A                                                354460
totaldebt_9A                                                353084
sumoutstandtotal_3546847A                                   346460
sumoutstandtotalest_4493215A                                246557
maxlnamtstart6m_4525199A                                    214539
maxinstallast24m_3658928A   

In [10]:
print("=== UNICIDADE DA CHAVE — case_id ===")

unique_ids = df_static_0.select("case_id").distinct().count()
dups = total - unique_ids

print(f"  Total de linhas : {total}")
print(f"  case_id únicos  : {unique_ids}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ case_id é chave primária única")
else:
    print("  ⚠️ Duplicatas encontradas — investigar antes da Silver")
    df_static_0.groupBy("case_id").count() \
        .filter(F.col("count") > 1) \
        .orderBy(F.col("count").desc()) \
        .show(10, truncate=False)


=== UNICIDADE DA CHAVE — case_id ===
  Total de linhas : 1526659
  case_id únicos  : 1526659
  Duplicatas      : 0
  ✅ case_id é chave primária única


In [11]:
print("=== RANGE DE DATAS ===")

date_cols = [
    "datefirstoffer_1144D", "datelastinstal40dpd_247D", "datelastunpaid_3546854D",
    "dtlastpmtallstes_4499206D", "firstclxcampaign_1125D", "firstdatedue_489D",
    "lastactivateddate_801D", "lastapplicationdate_877D", "lastapprdate_640D",
    "lastdelinqdate_224D", "lastrejectdate_50D", "lastrepayingdate_696D",
    "maxdpdinstldate_3546855D", "payvacationpostpone_4187118D", "validfrom_1069D"
]

df_static_0.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+------------------------+------------------------+----------------------------+----------------------------+---------------------------+---------------------------+-----------------------------+-----------------------------+--------------------------+--------------------------+---------------------+---------------------+--------------------------+--------------------------+----------------------------+----------------------------+---------------------+---------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------------+-------------------------+----------------------------+----------------------------+--------------------------------+--------------------------------+-------------------+-------------------+
|datefirstoffer_1144D_min|datefirstoffer_1144D_max|datelastinstal40dpd_247D_min|datelastinstal40dpd_247D_max|datelastunpaid_3546854D_min|datelastunpaid_3546854D_max|dtlastpmtallstes_44992

In [12]:
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [
    "lastapprcommoditycat_1041M", "lastapprcommoditytypec_5251766M",
    "lastcancelreason_561M", "lastrejectcommoditycat_161M",
    "lastrejectcommodtypec_5251769M", "lastrejectreason_759M",
    "lastrejectreasonclient_4145040M", "previouscontdistrict_112M"
]

for c in code_cols:
    outliers = df_static_0.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️ {c}: {count} valor(es) fora do padrão")
        outliers.show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️ lastapprcommoditycat_1041M: 1 valor(es) fora do padrão
+--------------------------+
|lastapprcommoditycat_1041M|
+--------------------------+
|a55475b1                  |
+--------------------------+

  ⚠️ lastapprcommoditytypec_5251766M: 1 valor(es) fora do padrão
+-------------------------------+
|lastapprcommoditytypec_5251766M|
+-------------------------------+
|a55475b1                       |
+-------------------------------+

  ⚠️ lastcancelreason_561M: 1 valor(es) fora do padrão
+---------------------+
|lastcancelreason_561M|
+---------------------+
|a55475b1             |
+---------------------+

  ⚠️ lastrejectcommoditycat_161M: 1 valor(es) fora do padrão
+---------------------------+
|lastrejectcommoditycat_161M|
+---------------------------+
|a55475b1                   |
+---------------------------+

  ⚠️ lastrejectcommodtypec_5251769M: 1 valor(es) fora do padrão
+------------------------------+
|lastrejectcommodtypec_525176

In [13]:
print("=== VALORES FRACIONÁRIOS EM COLUNAS *L (double) ===\n")

l_cols_double = [c for c, t in df_static_0.dtypes
                 if t == "double" and c.endswith("L")]

for c in l_cols_double:
    frac_count = df_static_0.filter(
        F.col(c).isNotNull() & ((F.col(c) % 1) != 0)
    ).count()
    if frac_count > 0:
        print(f"  ⚠️ {c}: {frac_count} valores fracionários")
        df_static_0.filter(
            F.col(c).isNotNull() & ((F.col(c) % 1) != 0)
        ).select(c).distinct().show(5, truncate=False)
    else:
        print(f"  ✅ {c}: sem fracionários")


=== VALORES FRACIONÁRIOS EM COLUNAS *L (double) ===

  ✅ applicationcnt_361L: sem fracionários
  ✅ applications30d_658L: sem fracionários
  ✅ applicationscnt_1086L: sem fracionários
  ✅ applicationscnt_464L: sem fracionários
  ✅ applicationscnt_629L: sem fracionários
  ✅ applicationscnt_867L: sem fracionários
  ✅ clientscnt12m_3712952L: sem fracionários
  ✅ clientscnt3m_3712950L: sem fracionários
  ✅ clientscnt6m_3712949L: sem fracionários
  ✅ clientscnt_100L: sem fracionários
  ✅ clientscnt_1022L: sem fracionários
  ✅ clientscnt_1071L: sem fracionários
  ✅ clientscnt_1130L: sem fracionários
  ✅ clientscnt_136L: sem fracionários
  ✅ clientscnt_157L: sem fracionários
  ✅ clientscnt_257L: sem fracionários
  ✅ clientscnt_304L: sem fracionários
  ✅ clientscnt_360L: sem fracionários
  ✅ clientscnt_493L: sem fracionários
  ✅ clientscnt_533L: sem fracionários
  ✅ clientscnt_887L: sem fracionários
  ✅ clientscnt_946L: sem fracionários
  ✅ cntincpaycont9m_3716944L: sem fracionários
  ✅ cntpmts2

In [14]:
print("=== RANGE DAS COLUNAS PCT* ===")

pct_cols = [c for c in df_static_0.columns if c.startswith("pct")]

df_static_0.select([
    val
    for c in pct_cols
    for val in (
        F.min(c).alias(f"{c}_min"),
        F.max(c).alias(f"{c}_max"),
        F.percentile_approx(c, 0.5).alias(f"{c}_median")
    )
]).show(truncate=False)


=== RANGE DAS COLUNAS PCT* ===
+-------------------------------+-------------------------------+----------------------------------+-------------------------------+-------------------------------+----------------------------------+-----------------------------------+-----------------------------------+--------------------------------------+-----------------------------------+-----------------------------------+--------------------------------------+-----------------------------------+-----------------------------------+--------------------------------------+
|pctinstlsallpaidearl3d_427L_min|pctinstlsallpaidearl3d_427L_max|pctinstlsallpaidearl3d_427L_median|pctinstlsallpaidlat10d_839L_min|pctinstlsallpaidlat10d_839L_max|pctinstlsallpaidlat10d_839L_median|pctinstlsallpaidlate1d_3546856L_min|pctinstlsallpaidlate1d_3546856L_max|pctinstlsallpaidlate1d_3546856L_median|pctinstlsallpaidlate4d_3546849L_min|pctinstlsallpaidlate4d_3546849L_max|pctinstlsallpaidlate4d_3546849L_median|pctinstlsallpai

In [15]:
print("=== DISTRIBUIÇÃO BOOLEANOS (null / false / true) ===\n")

bool_cols = [c for c, t in df_static_0.dtypes if t == "boolean"]

for c in bool_cols:
    counts = df_static_0.groupBy(c).count().collect()
    dist = {str(row[c]): row["count"] for row in counts}
    null_pct = round(dist.get("None", 0) / total * 100, 2)
    print(f"  {c}:")
    print(f"    True  : {dist.get('True', 0)}")
    print(f"    False : {dist.get('False', 0)}")
    print(f"    null  : {dist.get('None', 0)} ({null_pct}%)")


=== DISTRIBUIÇÃO BOOLEANOS (null / false / true) ===

  equalitydataagreement_891L:
    True  : 73064
    False : 4963
    null  : 1448632 (94.89%)
  equalityempfrom_62L:
    True  : 36784
    False : 1028
    null  : 1488847 (97.52%)
  isbidproduct_1095L:
    True  : 176610
    False : 1350049
    null  : 0 (0.0%)
  isbidproductrequest_292L:
    True  : 1
    False : 12457
    null  : 1514201 (99.18%)
  isdebitcard_729L:
    True  : 0
    False : 192302
    null  : 1334357 (87.4%)
  opencred_647L:
    True  : 50812
    False : 1170710
    null  : 305137 (19.99%)


In [16]:
print("=== VALORES NEGATIVOS — Colunas *P (DPD) ===\n")

dpd_cols = [c for c, t in df_static_0.dtypes
            if t == "double" and c.endswith("P")]

for c in dpd_cols:
    neg_count = df_static_0.filter(F.col(c) < 0).count()
    if neg_count > 0:
        print(f"  ⚠️ {c}: {neg_count} negativos")
    else:
        print(f"  ✅ {c}: sem negativos")

print("\n=== VALORES NEGATIVOS — Colunas *A (monetárias) ===\n")

a_cols = [c for c, t in df_static_0.dtypes
          if t == "double" and c.endswith("A")]

for c in a_cols:
    neg_count = df_static_0.filter(F.col(c) < 0).count()
    if neg_count > 0:
        print(f"  ⚠️ {c}: {neg_count} negativos")
    else:
        print(f"  ✅ {c}: sem negativos")


=== VALORES NEGATIVOS — Colunas *P (DPD) ===

  ✅ actualdpdtolerance_344P: sem negativos
  ⚠️ avgdbddpdlast24m_3658932P: 734651 negativos
  ⚠️ avgdbddpdlast3m_4187120P: 445711 negativos
  ⚠️ avgdbdtollast24m_4525197P: 461572 negativos
  ✅ avgdpdtolclosure24_3658938P: sem negativos
  ✅ avgmaxdpdlast9m_3716943P: sem negativos
  ⚠️ maxdbddpdlast1m_3658939P: 360717 negativos
  ⚠️ maxdbddpdtollast12m_3658940P: 234717 negativos
  ⚠️ maxdbddpdtollast6m_4187119P: 242197 negativos
  ✅ maxdpdfrom6mto36m_3546853P: sem negativos
  ✅ maxdpdinstlnum_3546846P: sem negativos
  ✅ maxdpdlast12m_727P: sem negativos
  ✅ maxdpdlast24m_143P: sem negativos
  ✅ maxdpdlast3m_392P: sem negativos
  ✅ maxdpdlast6m_474P: sem negativos
  ✅ maxdpdlast9m_1059P: sem negativos
  ✅ maxdpdtolerance_374P: sem negativos
  ⚠️ mindbddpdlast24m_3658935P: 867792 negativos
  ⚠️ mindbdtollast24m_4525191P: 527289 negativos
  ✅ posfpd10lastmonth_333P: sem negativos
  ✅ posfpd30lastmonth_3976960P: sem negativos
  ✅ posfstqpd30lastm

## 📋 Análise — `train_static_0`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Chave primária, tipo correto |
| actualdpdtolerance_344P | double | manter | Métrica de tolerância DPD, double correto |
| amtinstpaidbefduel24m_4187115A | double | manter | Valor monetário, double correto |
| annuity_780A | double | manter | Valor monetário, double correto |
| annuitynextmonth_57A | double | manter | Valor monetário, double correto |
| applicationcnt_361L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| applications30d_658L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| applicationscnt_1086L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| applicationscnt_464L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| applicationscnt_629L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| applicationscnt_867L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| avgdbddpdlast24m_3658932P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| avgdbddpdlast3m_4187120P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| avgdbdtollast24m_4525197P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| avgdpdtolclosure24_3658938P | double | manter | Sem negativos legítimos, double correto |
| avginstallast24m_3658937A | double | manter | Valor monetário, double correto |
| avglnamtstart24m_4525187A | double | manter | Valor monetário, double correto |
| avgmaxdpdlast9m_3716943P | double | manter | Métrica DPD, double correto |
| avgoutstandbalancel6m_4187114A | double | manter | Saldo pode ser negativo (legítimo), double correto |
| avgpmtlast12m_4525200A | double | manter | Valor monetário, double correto |
| bankacctype_710L | string | **DROP** | Constante — `approx_nunique = 1` |
| cardtype_51L | string | manter | Categórico de baixa cardinalidade (2 valores) |
| clientscnt12m_3712952L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt3m_3712950L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt6m_3712949L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_100L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_1022L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_1071L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_1130L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_136L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_157L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_257L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_304L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_360L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_493L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_533L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_887L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| clientscnt_946L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| cntincpaycont9m_3716944L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| cntpmts24_3658933L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| commnoinclast6m_3546845L | double | **DROP** | Constante — `approx_nunique = 1` |
| credamount_770A | double | manter | Valor monetário, double correto |
| credtype_322L | string | manter | Categórico (3 valores), string correto |
| currdebt_22A | double | manter | Valor monetário, double correto |
| currdebtcredtyperange_828A | double | manter | Valor monetário, double correto |
| datefirstoffer_1144D | string | cast → DateType | Data armazenada como string |
| datelastinstal40dpd_247D | string | cast → DateType | Data armazenada como string |
| datelastunpaid_3546854D | string | cast → DateType | Data armazenada como string |
| daysoverduetolerancedd_3976961L | double | cast → IntegerType | Contagem de dias, double desnecessário |
| deferredmnthsnum_166L | double | **DROP** | Constante — `approx_nunique = 1` |
| disbursedcredamount_1113A | double | manter | Valor monetário, double correto |
| disbursementtype_67L | string | manter | Categórico (3 valores), string correto |
| downpmt_116A | double | manter | Valor monetário, double correto |
| dtlastpmtallstes_4499206D | string | cast → DateType | Data armazenada como string |
| eir_270L | double | **manter como double** | Confirmado fracionário (taxa efetiva de juros) — sufixo `L` enganoso, semântica é rate |
| equalitydataagreement_891L | boolean | manter | Booleano, nullable semântico (não respondido ≠ false) |
| equalityempfrom_62L | boolean | manter | Booleano, nullable semântico (não respondido ≠ false) |
| firstclxcampaign_1125D | string | cast → DateType | Data armazenada como string |
| firstdatedue_489D | string | cast → DateType | Data armazenada como string |
| homephncnt_628L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| inittransactionamount_650A | double | manter | Valor monetário, double correto |
| inittransactioncode_186L | string | manter | Categórico (3 valores), string correto |
| interestrate_311L | double | **manter como double** | Confirmado fracionário (taxa de juros) — sufixo `L` enganoso, semântica é rate |
| interestrategrace_34L | double | **DROP** | Constante — `approx_nunique = 1` |
| isbidproduct_1095L | boolean | manter | Sem nulos, booleano correto |
| isbidproductrequest_292L | boolean | manter | Nullable semântico — 99.18% null, apenas 1 `True` em 12.458 preenchidos |
| isdebitcard_729L | boolean | **DROP** | Constante — `approx_nunique = 1`, todos `False` |
| lastactivateddate_801D | string | cast → DateType | Data armazenada como string |
| lastapplicationdate_877D | string | cast → DateType | Data armazenada como string |
| lastapprcommoditycat_1041M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastapprcommoditytypec_5251766M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastapprcredamount_781A | double | manter | Valor monetário, double correto |
| lastapprdate_640D | string | cast → DateType | Data armazenada como string |
| lastcancelreason_561M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastdelinqdate_224D | string | cast → DateType | Data armazenada como string |
| lastdependentsnum_448L | double | cast → IntegerType | Número de dependentes, double desnecessário |
| lastotherinc_902A | double | manter | Valor monetário, double correto |
| lastotherlnsexpense_631A | double | manter | Valor monetário, double correto |
| lastrejectcommoditycat_161M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastrejectcommodtypec_5251769M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastrejectcredamount_222A | double | manter | Valor monetário, double correto |
| lastrejectdate_50D | string | cast → DateType | Data armazenada como string |
| lastrejectreason_759M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastrejectreasonclient_4145040M | string | manter + tratar `a55475b1` | Código ofuscado; `a55475b1` → null |
| lastrepayingdate_696D | string | cast → DateType | Data armazenada como string |
| lastst_736L | string | manter | Categórico (11 valores), string correto |
| maininc_215A | double | manter | Renda, double correto |
| mastercontrelectronic_519L | double | **DROP** | Constante — `approx_nunique = 1` |
| mastercontrexist_109L | double | **DROP** | Constante — `approx_nunique = 1` |
| maxannuity_159A | double | manter | Valor monetário, double correto |
| maxannuity_4075009A | double | manter | Valor monetário, double correto |
| maxdbddpdlast1m_3658939P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| maxdbddpdtollast12m_3658940P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| maxdbddpdtollast6m_4187119P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| maxdebt4_972A | double | manter | Valor monetário, double correto |
| maxdpdfrom6mto36m_3546853P | double | manter | Sem negativos, double correto |
| maxdpdinstldate_3546855D | string | cast → DateType | Data armazenada como string |
| maxdpdinstlnum_3546846P | double | cast → IntegerType | Número de parcelas, double desnecessário |
| maxdpdlast12m_727P | double | manter | Métrica DPD, double correto |
| maxdpdlast24m_143P | double | manter | Métrica DPD, double correto |
| maxdpdlast3m_392P | double | manter | Métrica DPD, double correto |
| maxdpdlast6m_474P | double | manter | Métrica DPD, double correto |
| maxdpdlast9m_1059P | double | manter | Métrica DPD, double correto |
| maxdpdtolerance_374P | double | manter | Métrica DPD, double correto |
| maxinstallast24m_3658928A | double | manter | Valor monetário, double correto |
| maxlnamtstart6m_4525199A | double | manter | Valor monetário, double correto |
| maxoutstandbalancel12m_4187113A | double | manter + tratar -1 | Saldo — negativos residuais (4148); -1 é sentinela → null |
| maxpmtlast3m_4525190A | double | manter | Valor monetário, double correto |
| mindbddpdlast24m_3658935P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| mindbdtollast24m_4525191P | double | manter + tratar -1 | Double correto; -1 é sentinela → null |
| mobilephncnt_593L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| monthsannuity_845L | double | cast → IntegerType | Número de meses, double desnecessário |
| numactivecreds_622L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numactivecredschannel_414L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numactiverelcontr_750L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numcontrs3months_479L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numincomingpmts_3546848L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstlallpaidearly3d_817L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstls_657L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstlsallpaid_934L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstlswithdpd10_728L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstlswithdpd5_4187116L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstlswithoutdpd_562L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstmatpaidtearly2d_4499204L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaid_4499208L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearly3d_3546850L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearly3dest_4493216L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearly5d_1087L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearly5dest_4493211L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearly5dobd_4499205L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearly_338L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidearlyest_4493214L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidlastcontr_4325080L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstpaidlate1d_3546852L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstregularpaid_973L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstregularpaidest_4493210L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinsttopaygr_769L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinsttopaygrest_4493213L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstunpaidmax_3546851L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numinstunpaidmaxest_4493212L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numnotactivated_1143L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numpmtchanneldd_318L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| numrejects9m_859L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| opencred_647L | boolean | manter | Nullable semântico — 20% null (contrato sem informação) |
| paytype1st_925L | string | **DROP** | Constante — `approx_nunique = 1` |
| paytype_783L | string | **DROP** | Constante — `approx_nunique = 1` |
| payvacationpostpone_4187118D | string | cast → DateType | Data armazenada como string |
| pctinstlsallpaidearl3d_427L | double | **manter como double** | Confirmado fracionário [0, 23] — sufixo `L` enganoso, semântica é proporção/percentual |
| pctinstlsallpaidlat10d_839L | double | **manter como double** | Confirmado fracionário [0, 1.11] — sufixo `L` enganoso, semântica é proporção |
| pctinstlsallpaidlate1d_3546856L | double | **manter como double** | Confirmado fracionário [0, 1] — proporção |
| pctinstlsallpaidlate4d_3546849L | double | **manter como double** | Confirmado fracionário [0, 2] — proporção |
| pctinstlsallpaidlate6d_3546844L | double | **manter como double** | Confirmado fracionário [0, 1] — proporção |
| pmtnum_254L | double | cast → IntegerType | Número de parcelas, double desnecessário |
| posfpd10lastmonth_333P | double | manter | Binária (0/1 como double) — sem negativos |
| posfpd30lastmonth_3976960P | double | manter | Binária (0/1 como double) — sem negativos |
| posfstqpd30lastmonth_3976962P | double | manter | Binária (0/1 como double) — sem negativos |
| previouscontdistrict_112M | string | manter | Código ofuscado, string correto |
| price_1097A | double | manter | Valor monetário, double correto |
| sellerplacecnt_915L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| sellerplacescnt_216L | double | cast → IntegerType | Contagem inteira, double desnecessário |
| sumoutstandtotal_3546847A | double | manter | Saldo — negativos residuais (5); manter double |
| sumoutstandtotalest_4493215A | double | manter | Saldo estimado — negativos residuais (5); manter double |
| totaldebt_9A | double | manter | Valor monetário, double correto |
| totalsettled_863A | double | manter | Valor monetário, double correto |
| totinstallast1m_4525188A | double | manter | Valor monetário, double correto |
| twobodfilling_608L | string | manter | Binário string (2 valores), string correto |
| typesuite_864L | string | **DROP** | Constante — `approx_nunique = 1` |
| validfrom_1069D | string | cast → DateType | Data armazenada como string |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

---

### Nulos

O dataset apresenta perfil de nulos em 4 grupos distintos:

| Grupo | Critério | Colunas representativas | Ação Silver |
|---|---|---|---|
| Estrutural alto (≥ 87%) | Feature ausente para a maioria dos casos | `clientscnt_136L`, `lastrepayingdate_696D`, `isbidproductrequest_292L`, `isdebitcard_729L` | Manter — ausência é informação; candidatas a drop na modelagem |
| Estrutural médio (30–87%) | Feature disponível para subconjunto de casos | colunas `numinstl*`, `pct*`, `avgdpd*`, `dtlastpmt*` | Manter — codificar ausência na modelagem |
| Informativo baixo (< 30%) | Pequena fração sem preenchimento | `maininc_215A`, `firstdatedue_489D`, `pmtnum_254L` | Manter — investigar padrão na modelagem |
| Sem nulos | Completas | `case_id`, `isbidproduct_1095L`, `disbursementtype_67L`, etc. | Nenhuma ação |

> ⚠️ `clientscnt_136L` (99.97% nulo) com apenas 5 valores distintos — candidata a **DROP** na Silver.

---

### Sentinelas

Sentinelas **numéricos** detectados em larga escala. Nenhum sentinela string encontrado.

**Padrão `-1` (pseudo-nulo em métricas DPD/saldo):**

| Coluna | Ocorrências de `-1` |
|---|---|
| `avgdbddpdlast24m_3658932P` | 73.885 |
| `avgdbddpdlast3m_4187120P` | 61.517 |
| `avgdbdtollast24m_4525197P` | 45.814 |
| `maxdbddpdlast1m_3658939P` | 62.857 |
| `maxdbddpdtollast12m_3658940P` | 70.818 |
| `maxdbddpdtollast6m_4187119P` | 67.108 |
| `mindbddpdlast24m_3658935P` | 29.200 |
| `mindbdtollast24m_4525191P` | 17.348 |
| `avgoutstandbalancel6m_4187114A` | 110 |
| `maxoutstandbalancel12m_4187113A` | 11 |

> ⚠️ `-1` é sentinela explícito neste dataset (DPD nunca pode ser negativo). Converter para `null` na Silver em todas as colunas acima.

**Padrão `99/999/9999` (valores truncados/desconhecidos):**
Presentes em 50+ colunas, com contagens residuais (< 1.500 ocorrências por coluna na maioria dos casos). Exceções notáveis: `monthsannuity_845L` (860 ocorrências de `99`), `numincomingpmts_3546848L` (1.585 de `99`), `numinstlswithoutdpd_562L` (1.479 de `99`). Converter para `null` na Silver.

---

### Cardinalidade

- **Constantes confirmadas (DROP):** `bankacctype_710L`, `commnoinclast6m_3546845L`, `deferredmnthsnum_166L`, `interestrategrace_34L`, `isdebitcard_729L`, `mastercontrelectronic_519L`, `mastercontrexist_109L`, `paytype1st_925L`, `paytype_783L`, `typesuite_864L` — 10 colunas a remover.
- **Alta cardinalidade (contínuas):** `totalsettled_863A`, `amtinstpaidbefduel24m_4187115A`, `avgoutstandbalancel6m_4187114A` e demais `*A` — comportamento esperado para valores monetários.
- **Datas:** `*D` com ~4.000–5.500 distintos — coerente com ~20 meses de janela temporal.
- **Categóricos baixa cardinalidade:** `credtype_322L` (3), `disbursementtype_67L` (3), `inittransactioncode_186L` (3), `cardtype_51L` (2), `twobodfilling_608L` (2) — candidatos a `StringType` mantido ou encoding na modelagem.

---

### Chave Primária

`case_id` é chave primária única (0 duplicatas). Nenhuma deduplicação necessária na Silver.

---

### Range de Datas

Todas as 15 colunas `*D` apresentam ranges coerentes com o período esperado do dataset (2006–2020). Nenhuma data impossível detectada. Casts seguros para `DateType`.

> ℹ️ `payvacationpostpone_4187118D` e `validfrom_1069D` apresentam range mais restrito (2018–2020), consistente com features introduzidas posteriormente no sistema de origem.

---

### Sufixo `*L` com Semântica Não-Inteira

Três grupos de colunas com sufixo `*L` que **não devem ser castadas para IntegerType**:

| Colunas | Range Observado | Semântica real |
|---|---|---|
| `eir_270L`, `interestrate_311L` | fracionário (ex: 0.0012 – 0.445) | Taxa de juros (rate) |
| `pctinstlsallpaid*` (5 colunas) | [0, 23] / [0, 2] / [0, 1] | Proporção/percentual de parcelas |

> ⚠️ O sufixo `L` é enganoso para estas 7 colunas. Manter como `double` na Silver e documentar no catálogo de dados.

> ⚠️ `pctinstlsallpaidearl3d_427L` apresenta `max = 23.0`, sugerindo que parte dos registros usa escala `[0–100]` e outra `[0–1]`. Investigar distribuição completa antes da modelagem.

---

### Booleanos — Análise de `null` vs `false`

| Coluna | True | False | null (%) | Interpretação |
|---|---|---|---|---|
| `equalitydataagreement_891L` | 73.064 | 4.963 | 94,89% | `null` = não respondeu (≠ `false`) |
| `equalityempfrom_62L` | 36.784 | 1.028 | 97,52% | `null` = não respondeu (≠ `false`) |
| `isbidproduct_1095L` | 176.610 | 1.350.049 | 0% | Booleano completo ✅ |
| `isbidproductrequest_292L` | 1 | 12.457 | 99,18% | Feature quase vazia — candidata a DROP na modelagem |
| `isdebitcard_729L` | 0 | 192.302 | 87,4% | **Constante `false`** — DROP na Silver |
| `opencred_647L` | 50.812 | 1.170.710 | 19,99% | `null` = contrato sem informação de crédito aberto |

> ⚠️ Para `equalitydataagreement_891L` e `equalityempfrom_62L`, **não** imputar `null` como `false` — o `null` tem semântica distinta na modelagem.

---

### Valores Negativos em Colunas `*P` (DPD)

Colunas com prefixo `avg*/min*/max*` e sufixo `*P` que envolvem **janelas de comparação temporal** (ex: `avgdbddpdlast24m` vs `avgdbdtollast24m`) podem apresentar negativos como resultado de diferenças entre períodos. Esses negativos **não são erros** — representam melhora no comportamento de pagamento (DPD atual menor que DPD de referência).

| Coluna | Negativos | Interpretação |
|---|---|---|
| `avgdbddpdlast24m_3658932P` | 734.651 | Diferença temporal — legítimo |
| `avgdbddpdlast3m_4187120P` | 445.711 | Diferença temporal — legítimo |
| `avgdbdtollast24m_4525197P` | 461.572 | Diferença temporal — legítimo |
| `maxdbddpdlast1m_3658939P` | 360.717 | Diferença temporal — legítimo |
| `maxdbddpdtollast12m_3658940P` | 234.717 | Diferença temporal — legítimo |
| `maxdbddpdtollast6m_4187119P` | 242.197 | Diferença temporal — legítimo |
| `mindbddpdlast24m_3658935P` | 867.792 | Diferença temporal — legítimo |
| `mindbdtollast24m_4525191P` | 527.289 | Diferença temporal — legítimo |

> ✅ Nenhuma ação de truncamento na Silver. Negativos são legítimos nestas colunas. Documentar no catálogo.

---

### Valores Negativos em Colunas `*A` (Monetárias)

| Coluna | Negativos | Ação |
|---|---|---|
| `avgoutstandbalancel6m_4187114A` | 13.169 | Legítimo — saldo médio pode ser negativo |
| `maxoutstandbalancel12m_4187113A` | 4.148 | Legítimo — saldo máximo histórico pode ser negativo |
| `sumoutstandtotal_3546847A` | 5 | Residual — investigar, mas não truncar |
| `sumoutstandtotalest_4493215A` | 5 | Residual — investigar, mas não truncar |

> ✅ Nenhum truncamento na Silver. Manter negativos e documentar.

---

### Ações Silver — Resumo Executivo

```python
from pyspark.sql import functions as F

# ── DROP — Colunas constantes ─────────────────────────────────────────────────
cols_to_drop = [
    "bankacctype_710L", "commnoinclast6m_3546845L", "deferredmnthsnum_166L",
    "interestrategrace_34L", "isdebitcard_729L", "mastercontrelectronic_519L",
    "mastercontrexist_109L", "paytype1st_925L", "paytype_783L", "typesuite_864L"
]
df = df.drop(*cols_to_drop)

# ── CAST — Datas (string → DateType) ─────────────────────────────────────────
date_cols = [
    "datefirstoffer_1144D", "datelastinstal40dpd_247D", "datelastunpaid_3546854D",
    "dtlastpmtallstes_4499206D", "firstclxcampaign_1125D", "firstdatedue_489D",
    "lastactivateddate_801D", "lastapplicationdate_877D", "lastapprdate_640D",
    "lastdelinqdate_224D", "lastrejectdate_50D", "lastrepayingdate_696D",
    "maxdpdinstldate_3546855D", "payvacationpostpone_4187118D", "validfrom_1069D"
]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

# ── CAST — Contagens inteiras (double → IntegerType) ─────────────────────────
int_cols = [
    "applicationcnt_361L", "applications30d_658L", "applicationscnt_1086L",
    "applicationscnt_464L", "applicationscnt_629L", "applicationscnt_867L",
    "clientscnt12m_3712952L", "clientscnt3m_3712950L", "clientscnt6m_3712949L",
    "clientscnt_100L", "clientscnt_1022L", "clientscnt_1071L", "clientscnt_1130L",
    "clientscnt_136L", "clientscnt_157L", "clientscnt_257L", "clientscnt_304L",
    "clientscnt_360L", "clientscnt_493L", "clientscnt_533L", "clientscnt_887L",
    "clientscnt_946L", "cntincpaycont9m_3716944L", "cntpmts24_3658933L",
    "daysoverduetolerancedd_3976961L", "homephncnt_628L", "lastdependentsnum_448L",
    "maxdpdinstlnum_3546846P", "mobilephncnt_593L", "monthsannuity_845L",
    "numactivecreds_622L", "numactivecredschannel_414L", "numactiverelcontr_750L",
    "numcontrs3months_479L", "numincomingpmts_3546848L", "numinstlallpaidearly3d_817L",
    "numinstls_657L", "numinstlsallpaid_934L", "numinstlswithdpd10_728L",
    "numinstlswithdpd5_4187116L", "numinstlswithoutdpd_562L",
    "numinstmatpaidtearly2d_4499204L", "numinstpaid_4499208L",
    "numinstpaidearly3d_3546850L", "numinstpaidearly3dest_4493216L",
    "numinstpaidearly5d_1087L", "numinstpaidearly5dest_4493211L",
    "numinstpaidearly5dobd_4499205L", "numinstpaidearly_338L",
    "numinstpaidearlyest_4493214L", "numinstpaidlastcontr_4325080L",
    "numinstpaidlate1d_3546852L", "numinstregularpaid_973L",
    "numinstregularpaidest_4493210L", "numinsttopaygr_769L",
    "numinsttopaygrest_4493213L", "numinstunpaidmax_3546851L",
    "numinstunpaidmaxest_4493212L", "numnotactivated_1143L",
    "numpmtchanneldd_318L", "numrejects9m_859L", "pmtnum_254L",
    "sellerplacecnt_915L", "sellerplacescnt_216L"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))

# ── SENTINELAS → null (padrão -1 em DPD/saldo) ───────────────────────────────
sentinel_minus1_cols = [
    "avgdbddpdlast24m_3658932P", "avgdbddpdlast3m_4187120P",
    "avgdbdtollast24m_4525197P", "maxdbddpdlast1m_3658939P",
    "maxdbddpdtollast12m_3658940P", "maxdbddpdtollast6m_4187119P",
    "mindbddpdlast24m_3658935P", "mindbdtollast24m_4525191P",
    "avgoutstandbalancel6m_4187114A", "maxoutstandbalancel12m_4187113A"
]
for c in sentinel_minus1_cols:
    df = df.withColumn(c, F.when(F.col(c) == -1, None).otherwise(F.col(c)))

# ── SENTINELAS → null (padrão 99/999/9999 — ver tabela de sentinelas) ─────────
# Aplicar coluna a coluna conforme tabela de sentinelas detectados acima

# ── COLUNAS *M — a55475b1 → null ─────────────────────────────────────────────
m_cols = [
    "lastapprcommoditycat_1041M", "lastapprcommoditytypec_5251766M",
    "lastcancelreason_561M", "lastrejectcommoditycat_161M",
    "lastrejectcommodtypec_5251769M", "lastrejectreason_759M",
    "lastrejectreasonclient_4145040M"
]
for c in m_cols:
    df = df.withColumn(c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c)))


## Dataset train_static_cb_0

In [17]:
df_static_cb = load_logical_dataset(spark, bronze_dir, "train_static_cb")

Partições encontradas para 'train_static_cb':
  train_static_cb_0


In [18]:
df_static_cb.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- assignmentdate_238D: string (nullable = true)
 |-- assignmentdate_4527235D: string (nullable = true)
 |-- assignmentdate_4955616D: string (nullable = true)
 |-- birthdate_574D: string (nullable = true)
 |-- contractssum_5085716L: double (nullable = true)
 |-- dateofbirth_337D: string (nullable = true)
 |-- dateofbirth_342D: string (nullable = true)
 |-- days120_123L: double (nullable = true)
 |-- days180_256L: double (nullable = true)
 |-- days30_165L: double (nullable = true)
 |-- days360_512L: double (nullable = true)
 |-- days90_310L: double (nullable = true)
 |-- description_5085714M: string (nullable = true)
 |-- education_1103M: string (nullable = true)
 |-- education_88M: string (nullable = true)
 |-- firstquarter_103L: double (nullable = true)
 |-- for3years_128L: double (nullable = true)
 |-- for3years_504L: double (nullable = true)
 |-- for3years_584L: double (nullable = true)
 |-- formonth_118L: double (nullable = true)
 |-- for

In [19]:
from __future__ import annotations
from pyspark.sql import functions as F

total = df_static_cb.count()
print(f"Total de linhas: {total}\n")

null_counts = df_static_cb.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_static_cb.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_static_cb.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<50} {'null_count':>12} {'null_pct':>10}")
print("-" * 75)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<50} {null_count:>12} {null_pct:>9}%")


Total de linhas: 1500476

coluna                                               null_count   null_pct
---------------------------------------------------------------------------
dateofbirth_342D                                        1463976     97.57%
for3years_128L                                          1463962     97.57%
for3years_504L                                          1463962     97.57%
for3years_584L                                          1463962     97.57%
formonth_118L                                           1463962     97.57%
formonth_206L                                           1463962     97.57%
formonth_535L                                           1463962     97.57%
forquarter_1017L                                        1463962     97.57%
forquarter_462L                                         1463962     97.57%
forquarter_634L                                         1463962     97.57%
fortoday_1092L                                          1463962     97.57

In [20]:
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_static_cb.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id",)]

for c in numeric_cols:
    counts = df_static_cb.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]

string_cols = [c for c, t in df_static_cb.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_static_cb.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")


=== SENTINELAS — Numéricos ===
  contractssum_5085716L: {9999: 1}
  pmtscount_423L: {99: 1}
  pmtssum_45A: {99: 2, 999: 1, 9999: 2}

=== SENTINELAS — Strings ===


In [21]:
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_static_cb.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_static_cb.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<50} {'approx_nunique':>15}")
print("-" * 67)
for coluna, nunique in rows_card:
    flag = " ← CONSTANTE ⚠️" if nunique == 1 else ""
    print(f"{coluna:<50} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                              approx_nunique
-------------------------------------------------------------------
case_id                                                    1473037
pmtssum_45A                                                 264322
contractssum_5085716L                                       138905
pmtaverage_3A                                                38327
pmtaverage_4955615A                                          32284
pmtaverage_4527227A                                          28649
assignmentdate_238D                                           8916
assignmentdate_4955616D                                       8036
dateofbirth_337D                                               739
birthdate_574D                                                 694
dateofbirth_342D                                               694
responsedate_4527233D                                          379
assignmentdate_4527235D     

In [22]:
print("=== UNICIDADE DA CHAVE — case_id ===")

unique_ids = df_static_cb.select("case_id").distinct().count()
dups = total - unique_ids

print(f"  Total de linhas : {total}")
print(f"  case_id únicos  : {unique_ids}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ case_id é chave primária única")
else:
    print("  ⚠️ Duplicatas encontradas — investigar antes da Silver")
    df_static_cb.groupBy("case_id").count() \
        .filter(F.col("count") > 1) \
        .orderBy(F.col("count").desc()) \
        .show(10, truncate=False)


=== UNICIDADE DA CHAVE — case_id ===
  Total de linhas : 1500476
  case_id únicos  : 1500476
  Duplicatas      : 0
  ✅ case_id é chave primária única


In [23]:
print("=== RANGE DE DATAS ===")

date_cols = [
    "assignmentdate_238D", "assignmentdate_4527235D", "assignmentdate_4955616D",
    "birthdate_574D", "dateofbirth_337D", "dateofbirth_342D",
    "responsedate_1012D", "responsedate_4527233D", "responsedate_4917613D"
]

df_static_cb.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+-----------------------+-----------------------+---------------------------+---------------------------+---------------------------+---------------------------+------------------+------------------+--------------------+--------------------+--------------------+--------------------+----------------------+----------------------+-------------------------+-------------------------+-------------------------+-------------------------+
|assignmentdate_238D_min|assignmentdate_238D_max|assignmentdate_4527235D_min|assignmentdate_4527235D_max|assignmentdate_4955616D_min|assignmentdate_4955616D_max|birthdate_574D_min|birthdate_574D_max|dateofbirth_337D_min|dateofbirth_337D_max|dateofbirth_342D_min|dateofbirth_342D_max|responsedate_1012D_min|responsedate_1012D_max|responsedate_4527233D_min|responsedate_4527233D_max|responsedate_4917613D_min|responsedate_4917613D_max|
+-----------------------+-----------------------+---------------------------+---------------------------+----

In [24]:
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [
    "description_5085714M", "education_1103M", "education_88M",
    "maritalst_385M", "maritalst_893M"
]

for c in code_cols:
    outliers = df_static_cb.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️ {c}: {count} valor(es) fora do padrão")
        outliers.show(10, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️ description_5085714M: 2 valor(es) fora do padrão
+--------------------+
|description_5085714M|
+--------------------+
|2fc785b2            |
|a55475b1            |
+--------------------+

  ⚠️ education_1103M: 5 valor(es) fora do padrão
+---------------+
|education_1103M|
+---------------+
|6b2ae0fa       |
|39a0853f       |
|c8e1a1d0       |
|717ddd49       |
|a55475b1       |
+---------------+

  ⚠️ education_88M: 5 valor(es) fora do padrão
+-------------+
|education_88M|
+-------------+
|6b2ae0fa     |
|c8e1a1d0     |
|717ddd49     |
|a55475b1     |
|a34a13c8     |
+-------------+

  ⚠️ maritalst_385M: 6 valor(es) fora do padrão
+--------------+
|maritalst_385M|
+--------------+
|a7fcb6e5      |
|3439d993      |
|ecd83604      |
|b6cabe76      |
|38c061ee      |
|a55475b1      |
+--------------+

  ⚠️ maritalst_893M: 6 valor(es) fora do padrão
+--------------+
|maritalst_893M|
+--------------+
|46b968c3      |
|ecd83604      |
|977b2a

In [25]:
print("=== VALORES FRACIONÁRIOS — days*, for*, pmtcount* ===\n")

check_cols = [c for c, t in df_static_cb.dtypes
              if t == "double" and c not in ("case_id",)
              and any(c.startswith(p) for p in ("days", "for", "pmtcount", "pmtscount",
                                                 "firstquarter", "secondquarter",
                                                 "thirdquarter", "fourthquarter",
                                                 "numberofqueries", "contractssum"))]

for c in check_cols:
    frac_count = df_static_cb.filter(
        F.col(c).isNotNull() & ((F.col(c) % 1) != 0)
    ).count()
    if frac_count > 0:
        print(f"  ⚠️ {c}: {frac_count} valores fracionários")
        df_static_cb.filter(
            F.col(c).isNotNull() & ((F.col(c) % 1) != 0)
        ).select(c).distinct().show(5, truncate=False)
    else:
        print(f"  ✅ {c}: sem fracionários")


=== VALORES FRACIONÁRIOS — days*, for*, pmtcount* ===

  ⚠️ contractssum_5085716L: 122527 valores fracionários
+---------------------+
|contractssum_5085716L|
+---------------------+
|286596.17            |
|156449.4             |
|719271.82            |
|749563.23            |
|917349.72            |
+---------------------+
only showing top 5 rows

  ✅ days120_123L: sem fracionários
  ✅ days180_256L: sem fracionários
  ✅ days30_165L: sem fracionários
  ✅ days360_512L: sem fracionários
  ✅ days90_310L: sem fracionários
  ✅ firstquarter_103L: sem fracionários
  ✅ for3years_128L: sem fracionários
  ✅ for3years_504L: sem fracionários
  ✅ for3years_584L: sem fracionários
  ✅ formonth_118L: sem fracionários
  ✅ formonth_206L: sem fracionários
  ✅ formonth_535L: sem fracionários
  ✅ forquarter_1017L: sem fracionários
  ✅ forquarter_462L: sem fracionários
  ✅ forquarter_634L: sem fracionários
  ✅ fortoday_1092L: sem fracionários
  ✅ forweek_1077L: sem fracionários
  ✅ forweek_528L: sem fracio

In [26]:
print("=== CONCORDÂNCIA — Pares de colunas com semântica similar ===\n")

# Datas de nascimento
total_non_null = df_static_cb.filter(
    F.col("birthdate_574D").isNotNull() &
    F.col("dateofbirth_337D").isNotNull()
).count()
diverge = df_static_cb.filter(
    F.col("birthdate_574D").isNotNull() &
    F.col("dateofbirth_337D").isNotNull() &
    (F.col("birthdate_574D") != F.col("dateofbirth_337D"))
).count()
print(f"  birthdate_574D vs dateofbirth_337D:")
print(f"    Ambas preenchidas: {total_non_null} | Divergentes: {diverge}")

total_non_null = df_static_cb.filter(
    F.col("birthdate_574D").isNotNull() &
    F.col("dateofbirth_342D").isNotNull()
).count()
diverge = df_static_cb.filter(
    F.col("birthdate_574D").isNotNull() &
    F.col("dateofbirth_342D").isNotNull() &
    (F.col("birthdate_574D") != F.col("dateofbirth_342D"))
).count()
print(f"  birthdate_574D vs dateofbirth_342D:")
print(f"    Ambas preenchidas: {total_non_null} | Divergentes: {diverge}")

# Estado civil
total_non_null = df_static_cb.filter(
    F.col("maritalst_385M").isNotNull() &
    F.col("maritalst_893M").isNotNull()
).count()
diverge = df_static_cb.filter(
    F.col("maritalst_385M").isNotNull() &
    F.col("maritalst_893M").isNotNull() &
    (F.col("maritalst_385M") != F.col("maritalst_893M"))
).count()
print(f"\n  maritalst_385M vs maritalst_893M:")
print(f"    Ambas preenchidas: {total_non_null} | Divergentes: {diverge}")

# Educação
total_non_null = df_static_cb.filter(
    F.col("education_1103M").isNotNull() &
    F.col("education_88M").isNotNull()
).count()
diverge = df_static_cb.filter(
    F.col("education_1103M").isNotNull() &
    F.col("education_88M").isNotNull() &
    (F.col("education_1103M") != F.col("education_88M"))
).count()
print(f"\n  education_1103M vs education_88M:")
print(f"    Ambas preenchidas: {total_non_null} | Divergentes: {diverge}")

# Response dates
total_non_null = df_static_cb.filter(
    F.col("responsedate_4527233D").isNotNull() &
    F.col("responsedate_4917613D").isNotNull()
).count()
diverge = df_static_cb.filter(
    F.col("responsedate_4527233D").isNotNull() &
    F.col("responsedate_4917613D").isNotNull() &
    (F.col("responsedate_4527233D") != F.col("responsedate_4917613D"))
).count()
print(f"\n  responsedate_4527233D vs responsedate_4917613D:")
print(f"    Ambas preenchidas: {total_non_null} | Divergentes: {diverge}")


=== CONCORDÂNCIA — Pares de colunas com semântica similar ===

  birthdate_574D vs dateofbirth_337D:
    Ambas preenchidas: 534180 | Divergentes: 767
  birthdate_574D vs dateofbirth_342D:
    Ambas preenchidas: 10693 | Divergentes: 35

  maritalst_385M vs maritalst_893M:
    Ambas preenchidas: 1500476 | Divergentes: 840517

  education_1103M vs education_88M:
    Ambas preenchidas: 1500476 | Divergentes: 628750

  responsedate_4527233D vs responsedate_4917613D:
    Ambas preenchidas: 5408 | Divergentes: 261


In [27]:
print("=== INSPEÇÃO — riskassesment_302T (string) ===")
df_static_cb.groupBy("riskassesment_302T").count() \
    .orderBy(F.col("count").desc()) \
    .show(20, truncate=False)

print("\n=== RANGE — riskassesment_940T (double) ===")
df_static_cb.select(
    F.min("riskassesment_940T").alias("min"),
    F.max("riskassesment_940T").alias("max"),
    F.avg("riskassesment_940T").alias("mean"),
    F.percentile_approx("riskassesment_940T", 0.5).alias("median"),
    F.percentile_approx("riskassesment_940T", [0.25, 0.75]).alias("q1_q3")
).show(truncate=False)

frac_count = df_static_cb.filter(
    F.col("riskassesment_940T").isNotNull() &
    ((F.col("riskassesment_940T") % 1) != 0)
).count()
print(f"  Valores fracionários em riskassesment_940T: {frac_count}")


=== INSPEÇÃO — riskassesment_302T (string) ===
+------------------+-------+
|riskassesment_302T|count  |
+------------------+-------+
|NULL              |1446917|
|3% - 4%           |6445   |
|2% - 3%           |6147   |
|4% - 6%           |5681   |
|2% - 2%           |4948   |
|6% - 8%           |4561   |
|67% - 100%        |4549   |
|1% - 1%           |4103   |
|8% - 11%          |3922   |
|11% - 15%         |2989   |
|15% - 19%         |2244   |
|20% - 25%         |1569   |
|33% - 41%         |1556   |
|26% - 33%         |1495   |
|41% - 49%         |1370   |
|50% - 58%         |1187   |
|59% - 66%         |793    |
+------------------+-------+


=== RANGE — riskassesment_940T (double) ===
+----------+---------+-------------------+----------+------------------------+
|min       |max      |mean               |median    |q1_q3                   |
+----------+---------+-------------------+----------+------------------------+
|-3.6704228|2.1191318|0.22596803014593497|0.37183386|[-0.2279

## 📋 Análise — `train_static_cb_0`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Chave primária, tipo correto |
| assignmentdate_238D | string | cast → DateType | Data fiscal, armazenada como string |
| assignmentdate_4527235D | string | cast → DateType | Data fiscal, armazenada como string |
| assignmentdate_4955616D | string | cast → DateType | Data fiscal, armazenada como string |
| birthdate_574D | string | cast → DateType | Data de nascimento (bureau), armazenada como string |
| contractssum_5085716L | double | **manter como double** | Confirmado fracionário (soma monetária de contratos) — sufixo `L` enganoso |
| dateofbirth_337D | string | cast → DateType + tratar sentinelas | Data de nascimento; `1900-01-01` e `2020-01-01` são sentinelas → null |
| dateofbirth_342D | string | cast → DateType | Data de nascimento (97.57% nulo), cast seguro |
| days120_123L | double | cast → IntegerType | Contagem de consultas ao bureau, sem fracionários |
| days180_256L | double | cast → IntegerType | Contagem de consultas ao bureau, sem fracionários |
| days30_165L | double | cast → IntegerType | Contagem de consultas ao bureau, sem fracionários |
| days360_512L | double | cast → IntegerType | Contagem de consultas ao bureau, sem fracionários |
| days90_310L | double | cast → IntegerType | Contagem de consultas ao bureau, sem fracionários |
| description_5085714M | string | manter + tratar `a55475b1`, `2fc785b2` | Categorização bureau; ambos os valores → null |
| education_1103M | string | manter + tratar hash codes | Educação (fonte externa); hashes sem padrão `P*` → null |
| education_88M | string | manter + tratar hash codes | Educação (fonte interna); hashes sem padrão `P*` → null |
| firstquarter_103L | double | cast → IntegerType | Contagem de resultados bureau por trimestre, sem fracionários |
| for3years_128L | double | cast → IntegerType | Nº rejeições em 3 anos, sem fracionários |
| for3years_504L | double | cast → IntegerType | Histórico de crédito em 3 anos, sem fracionários |
| for3years_584L | double | cast → IntegerType | Nº cancelamentos em 3 anos, sem fracionários |
| formonth_118L | double | **DROP** | Constante — `approx_nunique = 1` |
| formonth_206L | double | cast → IntegerType | Nº cancelamentos no mês anterior, sem fracionários |
| formonth_535L | double | cast → IntegerType | Histórico de crédito no último mês, sem fracionários |
| forquarter_462L | double | **DROP** | Constante — `approx_nunique = 1` |
| forquarter_1017L | double | cast → IntegerType | Nº cancelamentos no último trimestre, sem fracionários |
| forquarter_634L | double | cast → IntegerType | Histórico de crédito no último trimestre, sem fracionários |
| fortoday_1092L | double | cast → IntegerType | Histórico de crédito hoje, sem fracionários |
| forweek_528L | double | cast → IntegerType | Histórico de crédito na última semana, sem fracionários |
| forweek_601L | double | **DROP** | Constante — `approx_nunique = 1` |
| forweek_1077L | double | cast → IntegerType | Nº cancelamentos na última semana, sem fracionários |
| foryear_618L | double | cast → IntegerType | Nº rejeições no último ano, sem fracionários |
| foryear_818L | double | cast → IntegerType | Nº cancelamentos no último ano, sem fracionários |
| foryear_850L | double | cast → IntegerType | Histórico de crédito no último ano, sem fracionários |
| fourthquarter_440L | double | cast → IntegerType | Contagem de resultados bureau no 4º trimestre, sem fracionários |
| maritalst_385M | string | manter | Estado civil, string correto |
| maritalst_893M | string | manter | Estado civil (fonte distinta), string correto |
| numberofqueries_373L | double | cast → IntegerType | Número de consultas ao bureau, sem fracionários |
| pmtaverage_3A | double | manter | Média de deduções fiscais, double correto |
| pmtaverage_4527227A | double | manter | Média de deduções fiscais (fonte 2), double correto |
| pmtaverage_4955615A | double | manter | Média de deduções fiscais (fonte 3), double correto |
| pmtcount_4527229L | double | cast → IntegerType | Nº de deduções fiscais (fonte 2), sem fracionários |
| pmtcount_4955617L | double | cast → IntegerType | Nº de deduções fiscais (fonte 3), sem fracionários |
| pmtcount_693L | double | cast → IntegerType | Nº de deduções fiscais (fonte 1), sem fracionários |
| pmtscount_423L | double | cast → IntegerType | Nº de pagamentos de deduções fiscais, sem fracionários |
| pmtssum_45A | double | manter + tratar sentinelas | Soma de deduções fiscais; sentinelas `99/999/9999` → null |
| requesttype_4525192L | string | manter | Tipo de requisição fiscal, string correto |
| responsedate_1012D | string | cast → DateType | Data de resposta fiscal (fonte 1), armazenada como string |
| responsedate_4527233D | string | cast → DateType | Data de resposta fiscal (fonte 2), armazenada como string |
| responsedate_4917613D | string | cast → DateType | Data de resposta fiscal (fonte 3), armazenada como string |
| riskassesment_302T | string | manter | Score de default em faixas percentuais — string categórica ordenada |
| riskassesment_940T | double | manter como double | Score contínuo de creditworthiness; range [-3.67, 2.12] confirma escala de score (logit/zscore) |
| secondquarter_766L | double | cast → IntegerType | Contagem de resultados bureau no 2º trimestre, sem fracionários |
| thirdquarter_1082L | double | cast → IntegerType | Contagem de resultados bureau no 3º trimestre, sem fracionários |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

---

### Nulos

| Grupo | null_pct | Colunas representativas | Interpretação |
|---|---|---|---|
| Estrutural alto (≥ 90%) | 90–98% | `for*` (17 cols), `riskassesment_*`, `assignmentdate_4*`, `pmtaverage_4*`, `contractssum` | Fontes fiscais/bureau ausentes para a maioria dos casos |
| Estrutural médio (52–90%) | 52–90% | `pmtscount`, `pmtssum`, `birthdate_574D`, `responsedate_*`, `requesttype` | Subconjunto de casos com dados fiscais ou de bureau disponíveis |
| Baixo (< 8%) | 7.65% | `dateofbirth_337D`, `days*`, `*quarter*`, `numberofqueries_373L` | Bloco coeso — mesmo percentual sugere que são preenchidos em conjunto pela mesma fonte |
| Sem nulos | 0% | `case_id`, `maritalst_385M`, `maritalst_893M`, `education_1103M`, `education_88M` | Completos ✅ |

> ℹ️ O bloco de 7.65% com exatamente o mesmo percentual nulo (`dateofbirth_337D`, `days*`, `*quarter*`) confirma que essas colunas vêm de uma **única fonte bureau** que ou responde completa ou não responde. Ausência é estrutural — não imputar.

---

### Sentinelas

Sentinelas numéricos residuais em apenas 3 colunas:

| Coluna | Sentinelas detectados | Ação |
|---|---|---|
| `contractssum_5085716L` | `{9999: 1}` | → null |
| `pmtscount_423L` | `{99: 1}` | → null |
| `pmtssum_45A` | `{99: 2, 999: 1, 9999: 2}` | → null |

Nenhum sentinela string detectado.

---

### Cardinalidade

- **`contractssum_5085716L`**: 138.905 distintos — alta cardinalidade confirmada como monetária contínua ✅
- **Datas de nascimento**: `birthdate_574D` e `dateofbirth_342D` com ~694 distintos cada — coerente (datas de nascimento discretizadas por mês/ano nos dados do bureau)
- **`riskassesment_940T`**: 202 distintos — score contínuo com granularidade moderada ✅
- **`riskassesment_302T`**: 16 faixas percentuais + null — categórico ordinal ✅
- **Constantes confirmadas (DROP):** `formonth_118L`, `forquarter_462L`, `forweek_601L`

---

### Chave Primária

`case_id` é chave primária única. Nenhuma deduplicação necessária na Silver.

---

### Range de Datas

| Coluna | Min | Max | Status |
|---|---|---|---|
| `assignmentdate_238D` | 1974-05-30 | 2019-10-22 | ✅ Range coerente |
| `assignmentdate_4527235D` | 2019-09-13 | 2020-10-19 | ✅ Janela recente — fonte mais nova |
| `assignmentdate_4955616D` | 1984-12-05 | 2020-09-22 | ✅ Range coerente |
| `birthdate_574D` | 1943-03-01 | 1998-09-01 | ✅ Datas de nascimento plausíveis |
| `dateofbirth_337D` | **1900-01-01** | **2020-01-01** | ⚠️ Sentinelas de data — converter para null |
| `dateofbirth_342D` | 1915-01-01 | 2015-09-01 | ⚠️ 1915 pode ser sentinela — monitorar |
| `responsedate_1012D` | 2019-01-02 | 2019-10-22 | ✅ Janela restrita — fonte com período curto |
| `responsedate_4527233D` | 2019-09-13 | 2020-10-19 | ✅ Coerente |
| `responsedate_4917613D` | 2020-03-26 | 2020-10-19 | ✅ Janela mais recente — fonte introduzida em 2020 |

> ⚠️ `dateofbirth_337D` com `min = 1900-01-01` e `max = 2020-01-01` são datas redondas impossíveis como nascimento real — tratar como sentinelas de data → null na Silver.

---

### Colunas `*M` — Valores Fora do Padrão

Todos os valores fora do padrão `P\d+_\d+_\d+` são **hashes ofuscados** sem semântica decodificável, tratados uniformemente como null:

| Coluna | Valores fora do padrão | Ação |
|---|---|---|
| `description_5085714M` | `a55475b1`, `2fc785b2` | → null |
| `education_1103M` | `6b2ae0fa`, `39a0853f`, `c8e1a1d0`, `717ddd49`, `a55475b1` | → null |
| `education_88M` | `6b2ae0fa`, `1a19667c`, `a55475b1` (+ outros) | → null |
| `maritalst_385M` | verificar output completo | → null se fora do padrão |
| `maritalst_893M` | verificar output completo | → null se fora do padrão |

---

### `contractssum_5085716L` — Sufixo `*L` com Semântica Não-Inteira

`contractssum_5085716L` apresentou **122.527 valores fracionários** (ex: `286596.17`, `749563.23`). Apesar do sufixo `L` (tipicamente contagem inteira), o dicionário confirma que é a **soma total de valores monetários de contratos** — double correto, não deve ser castado para inteiro.

> ⚠️ Documentar no catálogo: sufixo `L` enganoso para esta coluna.

---

### Concordância entre Pares de Colunas

| Par | Ambas preenchidas | Divergentes | Taxa de divergência | Interpretação |
|---|---|---|---|---|
| `birthdate_574D` vs `dateofbirth_337D` | 534.180 | 767 | 0,14% | ✅ Alta concordância — divergências residuais, provavelmente erros de digitação |
| `birthdate_574D` vs `dateofbirth_342D` | 10.693 | 35 | 0,33% | ✅ Alta concordância — fonte rara, divergências residuais |
| `maritalst_385M` vs `maritalst_893M` | 1.500.476 | 840.517 | **56%** | ⚠️ Divergência massiva — fontes com granularidades ou momentos distintos; manter ambas |
| `education_1103M` vs `education_88M` | 1.500.476 | 628.750 | **42%** | ⚠️ Divergência alta — fonte externa vs. interna divergem frequentemente; manter ambas |
| `responsedate_4527233D` vs `responsedate_4917613D` | 5.408 | 261 | 4,8% | ✅ Fontes distintas com datas diferentes por design — esperado |

> ⚠️ A divergência de 56% entre `maritalst_385M` e `maritalst_893M` e de 42% entre `education_1103M` e `education_88M` confirma que **não são duplicatas** — representam fontes de dados independentes com visões distintas do cliente. Manter ambas e deixar para a modelagem decidir qual usar ou como combinar.

---

### `riskassesment_302T` e `riskassesment_940T`

**`riskassesment_302T`** é um **score categórico ordinal** expresso como faixas percentuais de probabilidade de default (ex: `"3% - 4%"`, `"67% - 100%"`). Presente em apenas 3,57% dos casos (53.559 registros). Na Silver, manter como string — encoding ordinal adiado para a modelagem.

**`riskassesment_940T`** é um **score contínuo de creditworthiness** em escala de logit/z-score, com range `[-3.67, 2.12]` e mediana `0.37`. Valores negativos são esperados e legítimos nessa escala — sem ação de truncamento na Silver.

> ℹ️ Apesar de 96,43% de nulos em ambas as colunas `riskassesment_*`, quando presentes são features de alta densidade de informação sobre risco — manter obrigatoriamente.

---

### Ações Silver — Resumo Executivo

```python
from pyspark.sql import functions as F

# ── DROP — Colunas constantes ─────────────────────────────────────────────────
cols_to_drop = ["formonth_118L", "forquarter_462L", "forweek_601L"]
df = df.drop(*cols_to_drop)

# ── CAST — Datas (string → DateType) ─────────────────────────────────────────
date_cols = [
    "assignmentdate_238D", "assignmentdate_4527235D", "assignmentdate_4955616D",
    "birthdate_574D", "dateofbirth_337D", "dateofbirth_342D",
    "responsedate_1012D", "responsedate_4527233D", "responsedate_4917613D"
]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

# ── SENTINELAS DE DATA → null ────────────────────────────────────────────────
df = df.withColumn("dateofbirth_337D",
    F.when(F.col("dateofbirth_337D").isin(
        F.lit("1900-01-01").cast("date"),
        F.lit("2020-01-01").cast("date")
    ), None).otherwise(F.col("dateofbirth_337D")))

# ── CAST — Contagens inteiras (double → IntegerType) ─────────────────────────
int_cols = [
    "days120_123L", "days180_256L", "days30_165L", "days360_512L", "days90_310L",
    "firstquarter_103L", "secondquarter_766L", "thirdquarter_1082L", "fourthquarter_440L",
    "for3years_128L", "for3years_504L", "for3years_584L",
    "formonth_206L", "formonth_535L", "forquarter_1017L", "forquarter_634L",
    "fortoday_1092L", "forweek_528L", "forweek_1077L",
    "foryear_618L", "foryear_818L", "foryear_850L",
    "numberofqueries_373L",
    "pmtcount_693L", "pmtcount_4527229L", "pmtcount_4955617L", "pmtscount_423L"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))

# ── SENTINELAS → null ─────────────────────────────────────────────────────────
df = df.withColumn("contractssum_5085716L",
    F.when(F.col("contractssum_5085716L") == 9999, None)
     .otherwise(F.col("contractssum_5085716L")))

df = df.withColumn("pmtscount_423L",
    F.when(F.col("pmtscount_423L") == 99, None)
     .otherwise(F.col("pmtscount_423L")))

for val in :
    df = df.withColumn("pmtssum_45A",
        F.when(F.col("pmtssum_45A") == val, None)
         .otherwise(F.col("pmtssum_45A")))

# ── COLUNAS *M — hashes ofuscados → null ─────────────────────────────────────
hash_cols = ["description_5085714M", "education_1103M", "education_88M",
             "maritalst_385M", "maritalst_893M"]
for c in hash_cols:
    df = df.withColumn(c,
        F.when(~F.col(c).rlike(r"^P\d+_\d+_\d+$"), None)
         .otherwise(F.col(c)))


## Dataset train_tax_registry_a_1

In [4]:
df_tax_registry_a_1 = load_logical_dataset(spark, bronze_dir, "train_tax_registry_a_1")

Partições encontradas para 'train_tax_registry_a_1':
  train_tax_registry_a_1


In [5]:
df_tax_registry_a_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- amount_4527230A: double (nullable = true)
 |-- name_4527232M: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- recorddate_4527225D: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [6]:
from __future__ import annotations
from pyspark.sql import functions as F

total = df_tax_registry_a_1.count()
print(f"Total de linhas: {total}\n")

null_counts = df_tax_registry_a_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_tax_registry_a_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_tax_registry_a_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 3275770

coluna                                          null_count   null_pct
----------------------------------------------------------------------
case_id                                                  0       0.0%
amount_4527230A                                          0       0.0%
name_4527232M                                            0       0.0%
num_group1                                               0       0.0%
recorddate_4527225D                                      0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [7]:
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

for c in ["amount_4527230A"]:
    counts = df_tax_registry_a_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  ✅ {c}: sem sentinelas")

print("\n=== SENTINELAS — Strings ===")
sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]
for c in ["name_4527232M"]:
    counts = df_tax_registry_a_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  ✅ {c}: sem sentinelas string")


=== SENTINELAS — Numéricos ===
  amount_4527230A: {99: 133, 999: 100, 9999: 6}

=== SENTINELAS — Strings ===
  ✅ name_4527232M: sem sentinelas string


In [8]:
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_tax_registry_a_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_tax_registry_a_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← CONSTANTE ⚠️" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                                472480
name_4527232M                                          144052
amount_4527230A                                         88585
recorddate_4527225D                                       379
num_group1                                                103


In [9]:
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_tax_registry_a_1.select("case_id", "num_group1").distinct().count()
dups = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️ Duplicatas encontradas — investigar")
    df_tax_registry_a_1.groupBy("case_id", "num_group1").count() \
        .filter(F.col("count") > 1) \
        .orderBy(F.col("count").desc()) \
        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_tax_registry_a_1.groupBy("num_group1").count() \
    .orderBy("num_group1") \
    .show(30, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas : 3275770
  Chaves únicas   : 3275770
  Duplicatas      : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |457934|
|1         |441416|
|2         |426953|
|3         |409203|
|4         |388239|
|5         |341022|
|6         |174157|
|7         |133804|
|8         |112012|
|9         |95505 |
|10        |77635 |
|11        |60956 |
|12        |38268 |
|13        |27735 |
|14        |20538 |
|15        |15831 |
|16        |12321 |
|17        |9460  |
|18        |6630  |
|19        |5088  |
|20        |3929  |
|21        |3124  |
|22        |2507  |
|23        |2044  |
|24        |1578  |
|25        |1279  |
|26        |1029  |
|27        |853   |
|28        |693   |
|29        |562   |
+----------+------+
only showing top 30 rows



In [10]:
print("=== RANGE DE DATAS — recorddate_4527225D ===")

df_tax_registry_a_1.select(
    F.min("recorddate_4527225D").alias("min"),
    F.max("recorddate_4527225D").alias("max"),
    F.approx_count_distinct("recorddate_4527225D").alias("distintos")
).show(truncate=False)


=== RANGE DE DATAS — recorddate_4527225D ===
+----------+----------+---------+
|min       |max       |distintos|
+----------+----------+---------+
|2019-09-13|2020-10-19|379      |
+----------+----------+---------+



In [11]:
print("=== VALORES FORA DO PADRÃO — name_4527232M ===")

outliers = df_tax_registry_a_1.filter(
    ~F.col("name_4527232M").rlike(r"^P\d+_\d+_\d+$") &
    F.col("name_4527232M").isNotNull()
).select("name_4527232M").distinct()

count = outliers.count()
if count > 0:
    print(f"  ⚠️ {count} valor(es) fora do padrão")
    outliers.show(10, truncate=False)
else:
    print("  ✅ Todos no padrão esperado")


=== VALORES FORA DO PADRÃO — name_4527232M ===
  ⚠️ 146789 valor(es) fora do padrão
+-------------+
|name_4527232M|
+-------------+
|942f85e5     |
|8974a132     |
|0f571b99     |
|daf3f011     |
|52b3209c     |
|506c4bd0     |
|745fc1aa     |
|fa830c53     |
|7266fed8     |
|bfad72be     |
+-------------+
only showing top 10 rows



In [12]:
print("=== VALORES NEGATIVOS — amount_4527230A ===")

neg_count = df_tax_registry_a_1.filter(F.col("amount_4527230A") < 0).count()
if neg_count > 0:
    print(f"  ⚠️ {neg_count} valores negativos")
    df_tax_registry_a_1.filter(F.col("amount_4527230A") < 0) \
        .select("amount_4527230A") \
        .describe() \
        .show(truncate=False)
else:
    print("  ✅ Sem valores negativos")


=== VALORES NEGATIVOS — amount_4527230A ===
  ✅ Sem valores negativos


In [13]:
print("=== TOP 20 — name_4527232M (concentração de empregadores) ===")

total_nn = df_tax_registry_a_1.filter(F.col("name_4527232M").isNotNull()).count()

df_tax_registry_a_1.groupBy("name_4527232M").count() \
    .withColumn("pct", F.round(F.col("count") / total_nn * 100, 2)) \
    .orderBy(F.col("count").desc()) \
    .show(20, truncate=False)


=== TOP 20 — name_4527232M (concentração de empregadores) ===
+-------------+------+----+
|name_4527232M|count |pct |
+-------------+------+----+
|5e180ef0     |204371|6.24|
|P114_118_163 |15751 |0.48|
|74ca9587     |13196 |0.4 |
|7444479d     |12313 |0.38|
|P157_88_183  |10025 |0.31|
|a409d8fa     |9601  |0.29|
|f10df922     |9408  |0.29|
|6bd6aa12     |9239  |0.28|
|e304888c     |8957  |0.27|
|3d4aae4c     |8510  |0.26|
|f1e4f355     |8209  |0.25|
|6a3d9351     |7900  |0.24|
|7e3f5b77     |7220  |0.22|
|d580dfef     |7138  |0.22|
|6d01cb7c     |6964  |0.21|
|d014aa37     |6880  |0.21|
|3613fb71     |6832  |0.21|
|36a9355c     |6759  |0.21|
|9a211a98     |6509  |0.2 |
|eede7a57     |6238  |0.19|
+-------------+------+----+
only showing top 20 rows



## 📋 Análise — `tax_registry_a_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| amount_4527230A | double | manter + tratar sentinelas | Valor monetário de dedução fiscal, double correto |
| name_4527232M | string | manter + tratar hashes | Nome de empregador ofuscado, string correto |
| num_group1 | long | cast → IntegerType | Range 0–102, long desnecessário |
| recorddate_4527225D | string | cast → DateType | Data de registro fiscal, armazenada como string |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

---

### Nulos

Nenhuma coluna apresenta nulos. Dataset completo em todas as colunas. Nenhuma ação necessária na Silver.

---

### Sentinelas

Sentinelas numéricos detectados em `amount_4527230A`:

| Valor sentinela | Ocorrências | Ação |
|---|---|---|
| `99` | 133 | → null |
| `999` | 100 | → null |
| `9999` | 6 | → null |

Nenhum sentinela string detectado. Total de registros afetados: 239 (< 0,01% do dataset) — residual mas semanticamente incorretos como valores monetários.

---

### Cardinalidade

- **`case_id`**: ~472K distintos — o dataset cobre ~31% dos `case_id` do `train_base` (1.526.659), ou seja, nem todos os clientes têm registros fiscais nesta fonte
- **`name_4527232M`**: ~144K distintos — alta cardinalidade de empregadores, coerente com dado real
- **`amount_4527230A`**: ~88K distintos — cardinalidade contínua esperada para valor monetário
- **`recorddate_4527225D`**: 379 distintos — coerente com janela temporal de ~13 meses (set/2019–out/2020)
- **`num_group1`**: 103 distintos (range 0–102) — até 103 deduções fiscais por cliente

---

### Chave Composta

`(case_id, num_group1)` é chave composta única (0 duplicatas). Nenhuma deduplicação necessária na Silver.

---

### Granularidade — `num_group1`

A distribuição de `num_group1` revela o histórico de deduções fiscais por cliente:

- **num_group1 = 0**: 457.934 registros — todos os clientes com ao menos 1 dedução
- Decaimento gradual até **num_group1 = 5** (~341K) seguido de queda abrupta no **num_group1 = 6** (~174K)
- Cauda longa até **num_group1 = 102** — poucos clientes com histórico fiscal muito extenso

> ℹ️ A quebra entre `num_group1 = 5` e `= 6` sugere que a maioria dos clientes tem até 6 registros fiscais nesta fonte. Registros com `num_group1 ≥ 6` representam clientes com histórico fiscal acima da mediana — potencialmente relevante como feature de engajamento fiscal na modelagem.

---

### Range de Datas

`recorddate_4527225D` cobre **2019-09-13 a 2020-10-19** (379 datas distintas em ~13 meses). Range consistente com o período do dataset e com as `responsedate_*` do `train_static_cb`. Cast seguro para `DateType`.

---

### Coluna `*M` — `name_4527232M`

**146.789 valores distintos** fora do padrão `P\d+_\d+_\d+` — todos são hashes hexadecimais ofuscados (ex: `942f85e5`, `8974a132`). Este é o formato real dos nomes de empregadores neste dataset: a coluna usa **dois esquemas de ofuscação simultâneos**:

| Formato | Exemplo | Ocorrência |
|---|---|---|
| Padrão `P\d+_\d+_\d+` | `P114_118_163`, `P157_88_183` | Minoria |
| Hash hexadecimal | `5e180ef0`, `74ca9587` | Maioria (~144K distintos) |

> ⚠️ Ambos os formatos são **válidos** — não converter hashes para null. O padrão `P*` não é o único esquema de ofuscação desta coluna. Ajustar o check de `*M` para `name_4527232M` na Silver: **não aplicar** o filtro `~rlike(P\d+_\d+_\d+$)` → null.

---

### Concentração de Empregadores

O empregador `5e180ef0` domina com **204.371 registros (6,24%)**, muito acima do segundo colocado (0,48%). Esse valor isolado sugere um **empregador âncora** (grande empresa ou entidade pública) com volume desproporcional. Os demais empregadores têm distribuição bastante uniforme (0,19%–0,48%).

> ℹ️ Nenhuma ação na Silver. Relevante para engenharia de features na modelagem — considerar flag `is_anchor_employer` ou encoding por frequência.

---

### Ações Silver

```python
from pyspark.sql import functions as F

# ── CAST — Data (string → DateType) ──────────────────────────────────────────
df = df.withColumn("recorddate_4527225D",
    F.to_date(F.col("recorddate_4527225D"), "yyyy-MM-dd"))

# ── CAST — num_group1 (long → IntegerType) ───────────────────────────────────
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))

# ── SENTINELAS → null ─────────────────────────────────────────────────────────
for val in :
    df = df.withColumn("amount_4527230A",
        F.when(F.col("amount_4527230A") == val, None)
         .otherwise(F.col("amount_4527230A")))

# ── name_4527232M — NÃO aplicar filtro P\d+_\d+_\d+ → null ──────────────────
# Hashes hexadecimais são formato válido de ofuscação nesta coluna.
# Apenas converter a55475b1 se confirmado como sentinela em análise futura.


## Dataset train_tax_registry_b_1

In [14]:
df_tax_registry_b_1 = load_logical_dataset(spark, bronze_dir, "train_tax_registry_b_1")

Partições encontradas para 'train_tax_registry_b_1':
  train_tax_registry_b_1


In [15]:
df_tax_registry_b_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- amount_4917619A: double (nullable = true)
 |-- deductiondate_4917603D: string (nullable = true)
 |-- name_4917606M: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [16]:
from __future__ import annotations
from pyspark.sql import functions as F

total = df_tax_registry_b_1.count()
print(f"Total de linhas: {total}\n")

null_counts = df_tax_registry_b_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_tax_registry_b_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_tax_registry_b_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 1107933

coluna                                          null_count   null_pct
----------------------------------------------------------------------
case_id                                                  0       0.0%
amount_4917619A                                          0       0.0%
deductiondate_4917603D                                   0       0.0%
name_4917606M                                            0       0.0%
num_group1                                               0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [17]:
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

for c in ["amount_4917619A"]:
    counts = df_tax_registry_b_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  ✅ {c}: sem sentinelas")

print("\n=== SENTINELAS — Strings ===")
sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]
for c in ["name_4917606M"]:
    counts = df_tax_registry_b_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  ✅ {c}: sem sentinelas string")


=== SENTINELAS — Numéricos ===
  amount_4917619A: {99: 1, 999: 1}

=== SENTINELAS — Strings ===
  ✅ name_4917606M: sem sentinelas string


In [18]:
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_tax_registry_b_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_tax_registry_b_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← CONSTANTE ⚠️" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
amount_4917619A                                        188727
case_id                                                161809
name_4917606M                                           56050
deductiondate_4917603D                                    250
num_group1                                                104


In [19]:
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_tax_registry_b_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_tax_registry_b_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← CONSTANTE ⚠️" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
amount_4917619A                                        188727
case_id                                                161809
name_4917606M                                           56050
deductiondate_4917603D                                    250
num_group1                                                104


In [20]:
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_tax_registry_b_1.select("case_id", "num_group1").distinct().count()
dups = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️ Duplicatas encontradas — investigar")
    df_tax_registry_b_1.groupBy("case_id", "num_group1").count() \
        .filter(F.col("count") > 1) \
        .orderBy(F.col("count").desc()) \
        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_tax_registry_b_1.groupBy("num_group1").count() \
    .orderBy("num_group1") \
    .show(30, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas : 1107933
  Chaves únicas   : 1107933
  Duplicatas      : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |150732|
|1         |149210|
|2         |146224|
|3         |141814|
|4         |135063|
|5         |116602|
|6         |59418 |
|7         |42568 |
|8         |34229 |
|9         |28762 |
|10        |23851 |
|11        |19213 |
|12        |13151 |
|13        |9813  |
|14        |7690  |
|15        |6191  |
|16        |4993  |
|17        |3917  |
|18        |2855  |
|19        |2218  |
|20        |1781  |
|21        |1430  |
|22        |1132  |
|23        |920   |
|24        |670   |
|25        |548   |
|26        |448   |
|27        |361   |
|28        |299   |
|29        |258   |
+----------+------+
only showing top 30 rows



In [21]:
print("=== RANGE DE DATAS — deductiondate_4917603D ===")

df_tax_registry_b_1.select(
    F.min("deductiondate_4917603D").alias("min"),
    F.max("deductiondate_4917603D").alias("max"),
    F.approx_count_distinct("deductiondate_4917603D").alias("distintos")
).show(truncate=False)


=== RANGE DE DATAS — deductiondate_4917603D ===
+----------+----------+---------+
|min       |max       |distintos|
+----------+----------+---------+
|2019-09-27|2020-10-16|250      |
+----------+----------+---------+



In [25]:
print("=== INSPEÇÃO DE FORMATOS — name_4917606M ===")

# Contagem por formato
df_tax_registry_b_1.withColumn(
    "formato",
    F.when(F.col("name_4917606M").isNull(), "null")
     .when(F.col("name_4917606M").rlike(r"^P\d+_\d+_\d+$"), "padrao_P")
     .when(F.col("name_4917606M").rlike(r"^[0-9a-f]{8}$"), "hash_hex")
     .otherwise("outro")
).groupBy("formato").count() \
 .orderBy(F.col("count").desc()) \
 .show(truncate=False)

# Amostras do formato "outro", se existir
print("\n=== AMOSTRAS — valores fora de padrao_P e hash_hex ===")
df_tax_registry_b_1.filter(
    F.col("name_4917606M").isNotNull() &
    ~F.col("name_4917606M").rlike(r"^P\d+_\d+_\d+$") &
    ~F.col("name_4917606M").rlike(r"^[0-9a-f]{8}$")
).select("name_4917606M").distinct().show(10, truncate=False)


=== INSPEÇÃO DE FORMATOS — name_4917606M ===
+--------+-------+
|formato |count  |
+--------+-------+
|hash_hex|1087493|
|padrao_P|20440  |
+--------+-------+


=== AMOSTRAS — valores fora de padrao_P e hash_hex ===
+-------------+
|name_4917606M|
+-------------+
+-------------+



In [26]:
print("=== VALORES NEGATIVOS — amount_4917619A ===")

neg_count = df_tax_registry_b_1.filter(F.col("amount_4917619A") < 0).count()
if neg_count > 0:
    print(f"  ⚠️ {neg_count} valores negativos")
    df_tax_registry_b_1.filter(F.col("amount_4917619A") < 0) \
        .select("amount_4917619A") \
        .describe() \
        .show(truncate=False)
else:
    print("  ✅ Sem valores negativos")


=== VALORES NEGATIVOS — amount_4917619A ===
  ✅ Sem valores negativos


In [24]:
print("=== COBERTURA — case_id únicos em tax_registry_b_1 ===")

unique_cases_b = df_tax_registry_b_1.select("case_id").distinct().count()
print(f"  case_id únicos em tax_registry_b_1 : {unique_cases_b}")

# Sobreposição com tax_registry_a_1
overlap = df_tax_registry_b_1.select("case_id").distinct() \
    .join(df_tax_registry_a_1.select("case_id").distinct(),
          on="case_id", how="inner") \
    .count()

only_b = unique_cases_b - overlap
print(f"  Presentes em ambas as fontes       : {overlap}")
print(f"  Exclusivos de tax_registry_b_1     : {only_b}")


=== COBERTURA — case_id únicos em tax_registry_b_1 ===
  case_id únicos em tax_registry_b_1 : 150732
  Presentes em ambas as fontes       : 4246
  Exclusivos de tax_registry_b_1     : 146486


## 📋 Análise — `tax_registry_b_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| amount_4917619A | double | manter + tratar sentinelas | Valor monetário de dedução fiscal, double correto |
| deductiondate_4917603D | string | cast → DateType | Data de dedução fiscal, armazenada como string |
| name_4917606M | string | manter | Nome de empregador ofuscado — ambos os formatos válidos |
| num_group1 | long | cast → IntegerType | Range 0–103, long desnecessário |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

---

### Nulos

Nenhuma coluna apresenta nulos. Dataset completo em todas as colunas. Nenhuma ação necessária na Silver.

---

### Sentinelas

Sentinelas numéricos residuais em `amount_4917619A`:

| Valor sentinela | Ocorrências | Ação |
|---|---|---|
| `99` | 1 | → null |
| `999` | 1 | → null |

Nenhum sentinela string detectado. Volume residual (2 registros), mas semanticamente incorretos como valor monetário.

---

### Cardinalidade

- **`case_id`**: ~161K distintos — cobre ~10,6% dos `case_id` do `train_base` (1.526.659), cobertura bem menor que `tax_registry_a_1` (~31%)
- **`name_4917606M`**: ~56K distintos — cardinalidade menor que `tax_registry_a_1` (~144K), consistente com base de clientes menor
- **`amount_4917619A`**: ~188K distintos — alta cardinalidade contínua esperada para valor monetário
- **`deductiondate_4917603D`**: 250 distintos — janela de ~12 meses (set/2019–out/2020)
- **`num_group1`**: 104 distintos (range 0–103) — até 104 deduções fiscais por cliente

---

### Chave Composta

`(case_id, num_group1)` é chave composta única (0 duplicatas). Nenhuma deduplicação necessária na Silver.

---

### Granularidade — `num_group1`

A distribuição replica o padrão observado em `tax_registry_a_1`:

- Decaimento gradual de `num_group1 = 0` (150.732) até `= 5` (~116K)
- Queda abrupta no `num_group1 = 6` (~59K) — mesma quebra estrutural da fonte A
- Cauda longa até `num_group1 = 103`

> ℹ️ A quebra em `num_group1 = 6` é consistente entre as duas fontes fiscais (`a_1` e `b_1`), sugerindo que esse limiar reflete uma característica real do sistema de origem — possivelmente um limite de janela temporal ou de registros reportados por ciclo fiscal.

---

### Range de Datas

`deductiondate_4917603D` cobre **2019-09-27 a 2020-10-16** (250 datas distintas em ~12 meses). Range coerente e ligeiramente mais restrito que `tax_registry_a_1` (379 distintos). Cast seguro para `DateType`.

---

### Coluna `*M` — `name_4917606M`

| Formato | Ocorrências | Interpretação |
|---|---|---|
| `hash_hex` (`[0-9a-f]{8}`) | 1.087.493 | Formato dominante — ofuscação hexadecimal |
| `padrao_P` (`P\d+_\d+_\d+`) | 20.440 | Formato alternativo — mesmo padrão de outras tabelas |
| `outro` | 0 | ✅ Nenhum valor fora dos dois formatos esperados |

Ambos os formatos são válidos, confirmando o padrão já identificado em `tax_registry_a_1`. **Não aplicar** filtro `~rlike(P\d+_\d+_\d+$)` → null para esta coluna.

---

### Cobertura Comparada entre Fontes Fiscais

| Métrica | Valor |
|---|---|
| `case_id` únicos em `tax_registry_a_1` | ~472.480 |
| `case_id` únicos em `tax_registry_b_1` | ~150.732 |
| Presentes em **ambas** as fontes | 4.246 |
| Exclusivos de `tax_registry_b_1` | 146.486 |

> ⚠️ As duas fontes fiscais são **quase completamente disjuntas** — apenas 4.246 clientes (2,8% do `tax_registry_b_1`) aparecem nas duas. Isso confirma que `tax_registry_a_1` e `tax_registry_b_1` representam **autoridades fiscais distintas**, não versões redundantes da mesma fonte. Na modelagem, features agregadas de cada fonte devem ser tratadas como sinais independentes.

---

### Ações Silver

```python
from pyspark.sql import functions as F

# ── CAST — Data (string → DateType) ──────────────────────────────────────────
df = df.withColumn("deductiondate_4917603D",
    F.to_date(F.col("deductiondate_4917603D"), "yyyy-MM-dd"))

# ── CAST — num_group1 (long → IntegerType) ───────────────────────────────────
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))

# ── SENTINELAS → null ─────────────────────────────────────────────────────────
for val in :
    df = df.withColumn("amount_4917619A",
        F.when(F.col("amount_4917619A") == val, None)
         .otherwise(F.col("amount_4917619A")))

# ── name_4917606M — NÃO aplicar filtro P\d+_\d+_\d+ → null ──────────────────
# Hashes hexadecimais e padrão P* são ambos formatos válidos de ofuscação.
# Nenhum valor fora desses dois formatos foi detectado.


## Dataset train_tax_registry_c_1

In [27]:
df_tax_registry_c_1 = load_logical_dataset(spark, bronze_dir, "train_tax_registry_c_1")

Partições encontradas para 'train_tax_registry_c_1':
  train_tax_registry_c_1


In [28]:
df_tax_registry_c_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- employername_160M: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- pmtamount_36A: double (nullable = true)
 |-- processingdate_168D: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [29]:
from __future__ import annotations
from pyspark.sql import functions as F

total = df_tax_registry_c_1.count()
print(f"Total de linhas: {total}\n")

null_counts = df_tax_registry_c_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_tax_registry_c_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_tax_registry_c_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 3343800

coluna                                          null_count   null_pct
----------------------------------------------------------------------
case_id                                                  0       0.0%
employername_160M                                        0       0.0%
num_group1                                               0       0.0%
pmtamount_36A                                            0       0.0%
processingdate_168D                                      0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [30]:
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

for c in ["pmtamount_36A"]:
    counts = df_tax_registry_c_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  ✅ {c}: sem sentinelas")

print("\n=== SENTINELAS — Strings ===")
sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]
for c in ["employername_160M"]:
    counts = df_tax_registry_c_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  ✅ {c}: sem sentinelas string")


=== SENTINELAS — Numéricos ===
  pmtamount_36A: {99: 95, 999: 89, 9999: 3}

=== SENTINELAS — Strings ===
  ✅ employername_160M: sem sentinelas string


In [31]:
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_tax_registry_c_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_tax_registry_c_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← CONSTANTE ⚠️" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
pmtamount_36A                                          622650
case_id                                                470325
employername_160M                                      144540
processingdate_168D                                       330
num_group1                                                124


In [32]:
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_tax_registry_c_1.select("case_id", "num_group1").distinct().count()
dups = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️ Duplicatas encontradas — investigar")
    df_tax_registry_c_1.groupBy("case_id", "num_group1").count() \
        .filter(F.col("count") > 1) \
        .orderBy(F.col("count").desc()) \
        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_tax_registry_c_1.groupBy("num_group1").count() \
    .orderBy("num_group1") \
    .show(30, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas : 3343800
  Chaves únicas   : 3343800
  Duplicatas      : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |482265|
|1         |463176|
|2         |447333|
|3         |428710|
|4         |407285|
|5         |356851|
|6         |177060|
|7         |130208|
|8         |105525|
|9         |87921 |
|10        |71280 |
|11        |55641 |
|12        |33219 |
|13        |23467 |
|14        |17205 |
|15        |13099 |
|16        |10076 |
|17        |7719  |
|18        |5379  |
|19        |4060  |
|20        |3179  |
|21        |2526  |
|22        |2012  |
|23        |1600  |
|24        |1218  |
|25        |935   |
|26        |747   |
|27        |602   |
|28        |484   |
|29        |411   |
+----------+------+
only showing top 30 rows



In [33]:
print("=== RANGE DE DATAS — processingdate_168D ===")

df_tax_registry_c_1.select(
    F.min("processingdate_168D").alias("min"),
    F.max("processingdate_168D").alias("max"),
    F.approx_count_distinct("processingdate_168D").alias("distintos")
).show(truncate=False)


=== RANGE DE DATAS — processingdate_168D ===
+----------+----------+---------+
|min       |max       |distintos|
+----------+----------+---------+
|2018-07-11|2019-10-22|330      |
+----------+----------+---------+



In [34]:
print("=== INSPEÇÃO DE FORMATOS — employername_160M ===")

df_tax_registry_c_1.withColumn(
    "formato",
    F.when(F.col("employername_160M").isNull(), "null")
     .when(F.col("employername_160M").rlike(r"^P\d+_\d+_\d+$"), "padrao_P")
     .when(F.col("employername_160M").rlike(r"^[0-9a-f]{8}$"), "hash_hex")
     .otherwise("outro")
).groupBy("formato").count() \
 .orderBy(F.col("count").desc()) \
 .show(truncate=False)

print("\n=== AMOSTRAS — valores fora de padrao_P e hash_hex ===")
df_tax_registry_c_1.filter(
    F.col("employername_160M").isNotNull() &
    ~F.col("employername_160M").rlike(r"^P\d+_\d+_\d+$") &
    ~F.col("employername_160M").rlike(r"^[0-9a-f]{8}$")
).select("employername_160M").distinct().show(10, truncate=False)


=== INSPEÇÃO DE FORMATOS — employername_160M ===
+--------+-------+
|formato |count  |
+--------+-------+
|hash_hex|3286265|
|padrao_P|57513  |
|outro   |22     |
+--------+-------+


=== AMOSTRAS — valores fora de padrao_P e hash_hex ===
+-----------------+
|employername_160M|
+-----------------+
|Q7_28_will       |
|Q6_6_depends     |
+-----------------+



In [35]:
print("=== VALORES NEGATIVOS — pmtamount_36A ===")

neg_count = df_tax_registry_c_1.filter(F.col("pmtamount_36A") < 0).count()
if neg_count > 0:
    print(f"  ⚠️ {neg_count} valores negativos")
    df_tax_registry_c_1.filter(F.col("pmtamount_36A") < 0) \
        .select("pmtamount_36A") \
        .describe() \
        .show(truncate=False)
else:
    print("  ✅ Sem valores negativos")


=== VALORES NEGATIVOS — pmtamount_36A ===
  ✅ Sem valores negativos


In [36]:
print("=== COBERTURA — case_id únicos e sobreposição entre fontes fiscais ===\n")

cases_a = df_tax_registry_a_1.select("case_id").distinct()
cases_b = df_tax_registry_b_1.select("case_id").distinct()
cases_c = df_tax_registry_c_1.select("case_id").distinct()

unique_c = cases_c.count()

overlap_ac = cases_c.join(cases_a, on="case_id", how="inner").count()
overlap_bc = cases_c.join(cases_b, on="case_id", how="inner").count()
overlap_abc = cases_c.join(cases_a, on="case_id", how="inner") \
                     .join(cases_b, on="case_id", how="inner").count()

print(f"  case_id únicos em tax_registry_c_1      : {unique_c}")
print(f"  Sobreposição c_1 ∩ a_1                  : {overlap_ac}")
print(f"  Sobreposição c_1 ∩ b_1                  : {overlap_bc}")
print(f"  Sobreposição c_1 ∩ a_1 ∩ b_1            : {overlap_abc}")
print(f"  Exclusivos de tax_registry_c_1           : {unique_c - overlap_ac - overlap_bc + overlap_abc}")


=== COBERTURA — case_id únicos e sobreposição entre fontes fiscais ===

  case_id únicos em tax_registry_c_1      : 482265
  Sobreposição c_1 ∩ a_1                  : 82920
  Sobreposição c_1 ∩ b_1                  : 0
  Sobreposição c_1 ∩ a_1 ∩ b_1            : 0
  Exclusivos de tax_registry_c_1           : 399345


## 📋 Análise — `tax_registry_c_1`


### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| employername_160M | string | manter + tratar `outro` | Nome de empregador ofuscado — dois formatos válidos + 2 valores residuais |
| num_group1 | long | cast → IntegerType | Range 0–123, long desnecessário |
| pmtamount_36A | double | manter + tratar sentinelas | Valor monetário de dedução fiscal, double correto |
| processingdate_168D | string | cast → DateType | Data de processamento fiscal, armazenada como string |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

---

### Nulos

Nenhuma coluna apresenta nulos. Dataset completo em todas as colunas. Nenhuma ação necessária na Silver.

---

### Sentinelas

Sentinelas numéricos em `pmtamount_36A`:

| Valor sentinela | Ocorrências | Ação |
|---|---|---|
| `99` | 95 | → null |
| `999` | 89 | → null |
| `9999` | 3 | → null |

Nenhum sentinela string detectado. Total afetado: 187 registros (< 0,01%).

---

### Cardinalidade

- **`pmtamount_36A`**: ~622K distintos — maior cardinalidade monetária entre as três fontes fiscais, consistente com o maior volume total de registros desta fonte
- **`case_id`**: ~470K distintos — cobertura similar à `tax_registry_a_1` (~472K) e bem maior que `b_1` (~161K)
- **`employername_160M`**: ~144K distintos — cardinalidade idêntica à `name_4527232M` da fonte `a_1` (também ~144K), sugerindo base de empregadores equivalente
- **`processingdate_168D`**: 330 distintos — janela de ~15 meses
- **`num_group1`**: 124 distintos (range 0–123) — maior profundidade entre as três fontes fiscais

---

### Chave Composta

`(case_id, num_group1)` é chave composta única (0 duplicatas). Nenhuma deduplicação necessária na Silver.

---

### Granularidade — `num_group1`

Mesma estrutura das fontes `a_1` e `b_1`:

- Decaimento gradual de `num_group1 = 0` (482.265) até `= 5` (~356K)
- Queda abrupta no `num_group1 = 6` (~177K) — **padrão consistente nas três fontes fiscais**
- Maior profundidade máxima: até `num_group1 = 123` vs. 102 (`a_1`) e 103 (`b_1`)

> ℹ️ A quebra estrutural em `num_group1 = 6` se repete nas três fontes fiscais (`a_1`, `b_1`, `c_1`), reforçando que esse limiar é uma característica do sistema de origem — possivelmente o limite de registros reportados por ciclo fiscal anual (6 meses × 2 semestres).

---

### Range de Datas

`processingdate_168D` cobre **2018-07-11 a 2019-10-22** (330 datas distintas em ~15 meses).

> ⚠️ Esta é a **única fonte fiscal com janela temporal anterior** às demais — `a_1` e `b_1` começam em set/2019, enquanto `c_1` começa em jul/2018. Isso indica que `c_1` é uma fonte mais antiga ou com maior historicidade no sistema de origem. Cast seguro para `DateType`.

---

### Coluna `*M` — `employername_160M`

| Formato | Ocorrências | Interpretação |
|---|---|---|
| `hash_hex` (`[0-9a-f]{8}`) | 3.286.265 | Formato dominante — ofuscação hexadecimal |
| `padrao_P` (`P\d+_\d+_\d+`) | 57.513 | Formato alternativo válido |
| `outro` | 22 | ⚠️ Valores residuais — tratar → null |

Os 22 valores `outro` seguem o padrão `Q\d+_\d+_palavra` (ex: `Q7_28_will`, `Q6_6_depends`) — mesmo esquema de ofuscação textual já observado em `profession_152M` do `train_applprev_1`. São valores sem decodificação possível e semanticamente inúteis → **converter para null** na Silver.

> ✅ `hash_hex` e `padrao_P` são formatos válidos — não aplicar filtro genérico. Tratar apenas o formato `Q*` → null.

---

### Cobertura Comparada — Três Fontes Fiscais

| Métrica | Valor |
|---|---|
| `case_id` únicos em `tax_registry_a_1` | ~472.480 |
| `case_id` únicos em `tax_registry_b_1` | ~150.732 |
| `case_id` únicos em `tax_registry_c_1` | ~482.265 |
| `c_1` ∩ `a_1` | 82.920 |
| `c_1` ∩ `b_1` | **0** |
| `c_1` ∩ `a_1` ∩ `b_1` | **0** |
| Exclusivos de `c_1` | ~399.345 |

> ⚠️ `tax_registry_c_1` e `tax_registry_b_1` são **completamente disjuntas** (0 clientes em comum). `c_1` compartilha apenas ~17% dos seus clientes com `a_1` (82.920 de 482.265). As três fontes fiscais cobrem populações majoritariamente distintas — confirmando que representam **três autoridades fiscais independentes**. Na modelagem, features de cada fonte devem ser tratadas como sinais complementares sem sobreposição assumida.

---

### Panorama das Três Fontes Fiscais

| Fonte | Total linhas | `case_id` únicos | Janela temporal | Profundidade máx. (`num_group1`) |
|---|---|---|---|---|
| `tax_registry_a_1` | ~3,3M | ~472K | set/2019–out/2020 | 102 |
| `tax_registry_b_1` | ~1,1M | ~150K | set/2019–out/2020 | 103 |
| `tax_registry_c_1` | ~3,4M | ~482K | **jul/2018**–out/2019 | **123** |

> ℹ️ `c_1` é a fonte mais antiga (janela anterior), com maior profundidade histórica e maior cobertura de clientes. `b_1` é a menor e mais recente fonte fiscal. A combinação das três cobre populações majoritariamente distintas, maximizando a cobertura total na modelagem.

---

### Ações Silver

```python
from pyspark.sql import functions as F

# ── CAST — Data (string → DateType) ──────────────────────────────────────────
df = df.withColumn("processingdate_168D",
    F.to_date(F.col("processingdate_168D"), "yyyy-MM-dd"))

# ── CAST — num_group1 (long → IntegerType) ───────────────────────────────────
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))

# ── SENTINELAS → null ─────────────────────────────────────────────────────────
for val in :
    df = df.withColumn("pmtamount_36A",
        F.when(F.col("pmtamount_36A") == val, None)
         .otherwise(F.col("pmtamount_36A")))

# ── employername_160M — formato Q* → null ─────────────────────────────────────
# hash_hex e padrao_P são válidos. Apenas formato Q\d+_\d+_* → null.
df = df.withColumn("employername_160M",
    F.when(F.col("employername_160M").rlike(r"^Q\d+_\d+_\w+$"), None)
     .otherwise(F.col("employername_160M")))
